# CS-781 Phase 2: V1 L-NER Rescue Experiment (previous leaderboard score: 0.323)

Development copy based on the prepared V1 notebook. The first development milestone is to keep the Phase 1 architecture/tokenizer intact, isolate Phase 2 outputs, locate the Phase 2 files, and restore the trained V1 checkpoint before implementing the NER pipeline.

**Important:** attach the trained V1 checkpoint and Phase 2 data as Kaggle inputs before running the corresponding cells. This notebook has not been executed in this environment.

**Alternate experiment version:** 6-epoch character-boundary model with weighted focal loss, warm-up/cosine scheduling, and validation-calibrated inference.

This is a new, isolated V1 experiment copy. It retains character-level supervision, adds a stratified document holdout, corrects one-character decoding and overlapping-window class-count issues, and optionally refits on all labelled documents after validation-based epoch selection. It cannot guarantee a particular Codabench score.


## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from kaggle_secrets import UserSecretsClient
    kaggle_token = UserSecretsClient().get_secret("HF")
    if kaggle_token:
        HF_TOKEN = kaggle_token
except Exception as exc:
    print("Kaggle HF secret was not available in this environment.")

if HF_TOKEN:
    from huggingface_hub import login
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face authentication configured.")
else:
    print("No HF token found. Public dataset access may still work; gated access will require a token and accepted dataset terms.")


## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [ ]:
#VERSION 1: DECODER ONLY MODEL; using GQA+ROPE+RMSNORM

import os
import random
import json
import time
import math
import zipfile
import shutil
import torch
import numpy as np

SEED =42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Basic setup loaded.")
print(f"Random seed: {SEED}")

In [ ]:
#DATASET
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

#for this version i am using just the supreme court training files
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE ="Phase_1_Pretraining/Test/test.json"

#for this version i was not sure of the computation timing so taking just 100M tokens
TRAIN_TOKENS = 100_000_000
VAL_TOKENS = 5_000_000

print("Dataset settings loaded.")
print("Training file:", TRAIN_FILE)
print("Training tokens:", f"{TRAIN_TOKENS:,}")
print("Validation tokens:", f"{VAL_TOKENS:,}")

In [ ]:
#context length basically tells us about how many tokens it can look at in one traing example
MODEL_CONTEXT_LENGTH=1024
TRAIN_BLOCK_SIZE = MODEL_CONTEXT_LENGTH
print("CL:", MODEL_CONTEXT_LENGTH)
print("Training block size:", TRAIN_BLOCK_SIZE)

In [ ]:
#ARCHITECTURE
#Using 12 layers for a deeper model
N_LAYER=12
N_EMBD=768 #every token will be represented using 768 numbers inside the model
N_HEAD=12

#Number of key/value heads
#12 query heads will share 4 key heads and 4 value heads
N_KV_HEAD=4

#using normal feed-forward network
FFN_MULTIPLIER=4
DROPOUT=0.0
USE_ROPE= True
USE_RMSNORM = True

#sanity check
print("Model architecture settings loaded.")
print(f"Layers: {N_LAYER}")
print(f"Hidden size: {N_EMBD}")
print(f"Query heads: {N_HEAD}")
print(f"Key/Value heads: {N_KV_HEAD}")

In [ ]:
#TRAINING SETTINGS
PER_DEVICE_BATCH_SIZE=4
#To avoid any error as GPU might not be able to fit the large batch directly
GRADIENT_ACCUMULATION_STEPS=8
LEARNING_RATE = 3e-4
WEIGHT_DECAY=0.1
WARMUP_STEPS=500          #The learning rate inreases slowly instead of immediately using the entire value from start
MAX_GRAD_NORM=1000
EVAL_EVERY_STEPS=1000        #validation loss check frequency
LOG_EVERY_STEPS=100    #Print frequency
SAVE_EVERY_STEPS=1000
MAX_CHECKPOINTS_TO_KEEP=5
USE_AMP =torch.cuda.is_available()
USE_GRADIENT_CHECKPOINTING = False

#sanity check
print("Training settings loaded.")
print("Batch size:", PER_DEVICE_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:",PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)

In [ ]:
#Generation and output settings

TARGET_GENERATION_WORDS=128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.7            #The starter uses 0.1, but its predictions showed strong repetition, so i am experimenting it
TOP_P =0.95
TOP_K=50 #adding this as an experiment
REPETITION_PENALTY=1.1 #Adding a penalty as the predictions showed heavy repetition

#OUTPUT and CHECKPOINT PATHS

OUTPUT_DIR = "/kaggle/working/cs781_phase1_v1"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
CACHE_DIR = os.path.join(OUTPUT_DIR, "cache")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "final_model")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "config.json")

for directory in[
    OUTPUT_DIR,CHECKPOINT_DIR,CACHE_DIR,LOG_DIR,MODEL_OUTPUT_DIR
]:
    os.makedirs(directory, exist_ok=True)

print("Generation and output settings loaded.")
print("Output directory:", OUTPUT_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Cache directory:", CACHE_DIR)

In [ ]:
# Phase 2 output paths. Keep Phase 2 artifacts separate from Phase 1.
EXPERIMENT_NAME = "V1_Rescue_0p323"
PHASE2_ROOT_DIR = "/kaggle/working/cs781_phase2_lner"
PHASE2_OUTPUT_DIR = os.path.join(PHASE2_ROOT_DIR, EXPERIMENT_NAME)
PHASE2_CHECKPOINT_DIR = os.path.join(PHASE2_OUTPUT_DIR, "checkpoints")
PHASE2_PREDICTION_PATH = os.path.join(PHASE2_OUTPUT_DIR, "prediction.json")
PHASE2_SUBMISSION_ZIP = os.path.join(PHASE2_OUTPUT_DIR, "submission.zip")

os.makedirs(PHASE2_CHECKPOINT_DIR, exist_ok=True)

print("Experiment:", EXPERIMENT_NAME)
print("Phase 2 output directory:", PHASE2_OUTPUT_DIR)
print("Phase 2 checkpoint directory:", PHASE2_CHECKPOINT_DIR)

In [ ]:
#Experiment configuration
#this is to keep a track of my experiments and implemntations so that they donot mix up with the future ones

experiment_config = {
    "seed": SEED,

    # Dataset
    "dataset": HF_DATASET_REPO,
    "train_file": TRAIN_FILE,
    "train_tokens": TRAIN_TOKENS,
    "val_tokens": VAL_TOKENS,

    # Sequence length
    "context_length": MODEL_CONTEXT_LENGTH,
    "block_size": TRAIN_BLOCK_SIZE,

    # Model architecture
    "n_layer": N_LAYER,
    "n_embd": N_EMBD,
    "n_head": N_HEAD,
    "n_kv_head": N_KV_HEAD,
    "ffn_multiplier": FFN_MULTIPLIER,
    "dropout": DROPOUT,
    "use_rope": USE_ROPE,
    "use_rmsnorm": USE_RMSNORM,

    # Training
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "max_grad_norm": MAX_GRAD_NORM,
    "eval_every_steps": EVAL_EVERY_STEPS,
    "log_every_steps": LOG_EVERY_STEPS,
    "save_every_steps": SAVE_EVERY_STEPS,
    "max_checkpoints_to_keep": MAX_CHECKPOINTS_TO_KEEP,
    "use_amp": USE_AMP,

    # Generation
    "target_generation_words": TARGET_GENERATION_WORDS,
    "max_generation_tokens": MAX_GENERATION_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "repetition_penalty": REPETITION_PENALTY,
}

## 4. Load Tokenizer

In [ ]:
#tokenizer from A1
import os
import json

BPE_FILENAME = "bpe_en_25k.json"
bpe_candidates = []

for root, dirs, files in os.walk("/kaggle/input"):
    if BPE_FILENAME in files:
        bpe_candidates.append(os.path.join(root, BPE_FILENAME))

if len(bpe_candidates) == 0:
    raise FileNotFoundError(
        f"{BPE_FILENAME} was not found inside /kaggle/input/. "
        "Make sure the BPE-25k dataset is attached."
    )

BPE_PATH = bpe_candidates[0]

print("Tokenizer found:", BPE_PATH)


# Load the vocabulary and learned BPE merges.
with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = bpe_data["vocab_size"]

# Make sure we have loaded the tokenizer we actually want.
assert bpe_vocab_size == 25_000
assert len(bpe_itos) == 25_000

print("Vocabulary size:", bpe_vocab_size)
print("Number of merge rules:", len(bpe_merges))


# Token -> ID mapping
bpe_stoi = {
    token: idx
    for idx, token in enumerate(bpe_itos)
}
merge_to_id = {}

for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge

    left_id = bpe_stoi[left_token]
    right_id = bpe_stoi[right_token]

    new_token_id = 261 + merge_index
    merge_to_id[(left_id, right_id)] = new_token_id

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]

print("\nSpecial token IDs:")
print("PAD:", PAD_TOKEN_ID)
print("UNK:", UNK_TOKEN_ID)
print("BOS:", BOS_TOKEN_ID)
print("EOS:", EOS_TOKEN_ID)
print("WORD BOUNDARY:", WORD_BOUNDARY_ID)

In [ ]:
#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

In [ ]:
# Resolve Phase 2 files from an attached Kaggle dataset first, then from the official Hugging Face repository.
# The Capstone Document maps phase_2 to the Phase_2_LNER folder.
from huggingface_hub import hf_hub_download

PHASE2_HF_PATHS = {
    "fold1-text.json": "Phase_2_LNER/Train/fold1-text.json",
    "fold1-gold.json": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2-text.json": "Phase_2_LNER/Train/fold2-text.json",
    "fold2-gold.json": "Phase_2_LNER/Train/fold2-gold.json",
    "fold3-text.json": "Phase_2_LNER/Test/fold3-text.json",
    "labels.txt": "Phase_2_LNER/Train/labels.txt",
}

PHASE2_FILES = {}
PHASE2_FILE_SOURCES = {}
PHASE2_RESOLUTION_ERRORS = {}

for filename, hub_filename in PHASE2_HF_PATHS.items():
    local_matches = []
    for search_root in ["/kaggle/input", "/kaggle/working"]:
        if os.path.isdir(search_root):
            for root, dirs, files in os.walk(search_root):
                if filename in files:
                    local_matches.append(os.path.join(root, filename))

    local_matches = sorted(set(local_matches))
    if len(local_matches) == 1:
        PHASE2_FILES[filename] = local_matches[0]
        PHASE2_FILE_SOURCES[filename] = "local"
        print(f"LOCAL   {filename}: {local_matches[0]}")
        continue

    if len(local_matches) > 1:
        print(f"MULTIPLE local copies found for {filename}; trying the official Hugging Face path instead.")

    try:
        downloaded_path = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=hub_filename,
            repo_type="dataset",
            token=HF_TOKEN or None,
        )
        PHASE2_FILES[filename] = downloaded_path
        PHASE2_FILE_SOURCES[filename] = "huggingface"
        print(f"HUGGING FACE {filename}: {downloaded_path}")
    except Exception as exc:
        PHASE2_RESOLUTION_ERRORS[filename] = f"{type(exc).__name__}: {exc}"
        print(f"NOT RESOLVED {filename}: {type(exc).__name__}: {exc}")

PHASE2_REQUIRED_FILENAMES = list(PHASE2_HF_PATHS)
PHASE2_MISSING_FILES = [name for name in PHASE2_REQUIRED_FILENAMES if name not in PHASE2_FILES]
PHASE2_DATA_READY = not PHASE2_MISSING_FILES

print("\nPhase 2 data ready:", PHASE2_DATA_READY)
print("Local files:", sum(v == "local" for v in PHASE2_FILE_SOURCES.values()))
print("Hub files:", sum(v == "huggingface" for v in PHASE2_FILE_SOURCES.values()))

if PHASE2_MISSING_FILES:
    missing_details = "\n".join(
        f"- {name} ({PHASE2_HF_PATHS[name]}): {PHASE2_RESOLUTION_ERRORS.get(name, 'not found')}"
        for name in PHASE2_MISSING_FILES
    )
    raise RuntimeError(
        "Could not resolve all Phase 2 files. Verify that the repository is "
        f"accessible, the HF token has accepted any access conditions, and the file paths match the current dataset.\n{missing_details}"
    )


## 5. Download Training Data From Hugging Face

In [ ]:
# from huggingface_hub import hf_hub_download

# train_file_path = hf_hub_download(
#     repo_id=HF_DATASET_REPO,
#     filename=TRAIN_FILE,
#     repo_type="dataset",
#     token=HF_TOKEN
# )

# print("Training file:")
# print(train_file_path)

## 6. Create Train and Validation Tokens Set

In [ ]:
# # 6. Create Train and Validation Token Cache
# import os
# import json
# import random
# import time
# import numpy as np

# CACHE_DIR = os.path.join(
#     OUTPUT_DIR,
#     "token_cache"
# )

# os.makedirs(CACHE_DIR, exist_ok=True)

# TRAIN_TOKEN_PATH = os.path.join(
#     CACHE_DIR,
#     "train_tokens.bin"
# )

# VAL_TOKEN_PATH = os.path.join(
#     CACHE_DIR,
#     "val_tokens.bin"
# )

# TOKEN_CACHE_META = os.path.join(
#     CACHE_DIR,
#     "cache_metadata.json"
# )

# # Load the official training corpus
# with open(train_file_path, "r", encoding="utf-8") as f:
#     records = json.load(f)

# print("Number of records:", len(records))

# # Resume from an existing cache if possible

# train_count = 0
# val_count = 0
# start_record = 0

# if (
#     os.path.exists(TRAIN_TOKEN_PATH)
#     and os.path.exists(VAL_TOKEN_PATH)
#     and os.path.exists(TOKEN_CACHE_META)
# ):
#     with open(TOKEN_CACHE_META, "r", encoding="utf-8") as f:
#         cache_meta = json.load(f)

#     # Make sure this cache belongs to the same experiment.
#     if (
#         cache_meta.get("train_tokens") == TRAIN_TOKENS
#         and cache_meta.get("val_tokens") == VAL_TOKENS
#         and cache_meta.get("vocab_size") == tokenizer.vocab_size
#         and cache_meta.get("seed") == SEED
#     ):
#         start_record = cache_meta["next_record"]
#         train_count = cache_meta["train_count"]
#         val_count = cache_meta["val_count"]

#         print("Existing token cache found.")
#         print("Resuming from record:", start_record)
#         print("Train tokens already cached:", f"{train_count:,}")
#         print("Validation tokens already cached:", f"{val_count:,}")

#     else:
#         print("Existing cache does not match the current settings.")
#         print("Starting a fresh token cache.")

#         os.remove(TRAIN_TOKEN_PATH)
#         os.remove(VAL_TOKEN_PATH)
#         os.remove(TOKEN_CACHE_META)

# else:
#     print("No existing token cache found.")
#     print("Starting tokenization from the beginning.")

# # Open cache files in append mode

# train_file = open(
#     TRAIN_TOKEN_PATH,
#     "ab"
# )

# val_file = open(
#     VAL_TOKEN_PATH,
#     "ab"
# )

# # Tokenize the corpus
# # We keep the same record-level random split used earlier:roughly 95% for training and 5% for validation.The seed and record index are saved, so the split stays fixed when the process is resumed.

# random.seed(SEED)

# # Recreate the same random decisions for records that were
# # already processed before an interruption.
# for _ in range(start_record):
#     random.random()

# start_time = time.time()

# for record_index in range(start_record, len(records)):

#     # Stop once both requested token budgets are reached.
#     if (
#         train_count >= TRAIN_TOKENS
#         and val_count >= VAL_TOKENS
#     ):
#         break

#     text = records[record_index]["text"]

#     # Use our custom BPE-25k tokenizer.
#     tokens = tokenizer.encode(
#         text,
#         add_bos=False,
#         add_eos=False
#     )

#     # Decide whether this record belongs to validation.
#     is_validation = random.random() < 0.05

#     if is_validation and val_count < VAL_TOKENS:

#         remaining = VAL_TOKENS - val_count
#         tokens = tokens[:remaining]

#         if tokens:
#             token_array = np.asarray(
#                 tokens,
#                 dtype=np.uint32
#             )

#             val_file.write(
#                 token_array.tobytes()
#             )

#             val_count += len(token_array)

#     elif not is_validation and train_count < TRAIN_TOKENS:

#         remaining = TRAIN_TOKENS - train_count
#         tokens = tokens[:remaining]

#         if tokens:
#             token_array = np.asarray(
#                 tokens,
#                 dtype=np.uint32
#             )

#             train_file.write(
#                 token_array.tobytes()
#             )

#             train_count += len(token_array)


#     # Save progress periodically so an interruption does notforce us to start tokenization from zero.
#     if (
#         (record_index + 1) % 100 == 0
#         or train_count >= TRAIN_TOKENS
#         or val_count >= VAL_TOKENS
#     ):

#         train_file.flush()
#         val_file.flush()

#         cache_meta = {
#             "train_tokens": TRAIN_TOKENS,
#             "val_tokens": VAL_TOKENS,
#             "train_count": train_count,
#             "val_count": val_count,
#             "next_record": record_index + 1,
#             "vocab_size": tokenizer.vocab_size,
#             "seed": SEED
#         }

#         with open(
#             TOKEN_CACHE_META,
#             "w",
#             encoding="utf-8"
#         ) as f:
#             json.dump(
#                 cache_meta,
#                 f,
#                 indent=2
#             )

#     # Print progress every 100 records.
#     if (record_index + 1) % 100 == 0:

#         elapsed = time.time() - start_time

#         print(
#             f"Record {record_index + 1:,}/{len(records):,} | "
#             f"Train: {train_count:,}/{TRAIN_TOKENS:,} | "
#             f"Val: {val_count:,}/{VAL_TOKENS:,} | "
#             f"Time: {elapsed / 60:.1f} min"
#         )


# train_file.flush()
# val_file.flush()

# train_file.close()
# val_file.close()

# # Final checks

# assert train_count == TRAIN_TOKENS, (
#     f"Expected {TRAIN_TOKENS:,} train tokens, "
#     f"got {train_count:,}"
# )

# assert val_count == VAL_TOKENS, (
#     f"Expected {VAL_TOKENS:,} validation tokens, "
#     f"got {val_count:,}"
# )

# print("\nTokenization complete.")
# print("Train tokens:", f"{train_count:,}")
# print("Validation tokens:", f"{val_count:,}")
# print("Train cache:", TRAIN_TOKEN_PATH)
# print("Validation cache:", VAL_TOKEN_PATH)

## 7. Create Token Blocks

In [ ]:
# # Create Dataset From Cached Tokens

# import numpy as np
# import torch
# from torch.utils.data import Dataset

# class LegalTokenDataset(Dataset):
#     def __init__(
#         self,
#         token_path,
#         num_tokens,
#         block_size,
#         stride=1
#     ):
#         self.token_path = token_path
#         self.num_tokens = num_tokens
#         self.block_size = block_size
#         self.stride = stride

#         # The file stays on disk and is accessed only when
#         # a particular training example is requested.
#         self.tokens = np.memmap(
#             token_path,
#             dtype=np.uint32,
#             mode="r",
#             shape=(num_tokens,)
#         )

#     def __len__(self):
#         return 1 + (
#             self.num_tokens
#             - self.block_size
#             - 1
#         ) // self.stride

#     def __getitem__(self, index):
#         start = index * self.stride
#         end = start + self.block_size + 1

#         tokens = self.tokens[start:end].astype(
#             np.int64
#         )

#         input_ids = torch.from_numpy(
#             tokens[:-1]
#         )

#         labels = torch.from_numpy(
#             tokens[1:]
#         )

#         return {
#             "input_ids": input_ids,
#             "labels": labels
#         }

# train_dataset = LegalTokenDataset(
#     token_path=TRAIN_TOKEN_PATH,
#     num_tokens=TRAIN_TOKENS,
#     block_size=TRAIN_BLOCK_SIZE,
#     stride=1
# )

# val_dataset = LegalTokenDataset(
#     token_path=VAL_TOKEN_PATH,
#     num_tokens=VAL_TOKENS,
#     block_size=TRAIN_BLOCK_SIZE,
#     stride=TRAIN_BLOCK_SIZE
# )

# print("Train examples:", f"{len(train_dataset):,}")
# print("Validation examples:", f"{len(val_dataset):,}")
# print("Sequence length:", TRAIN_BLOCK_SIZE)

## 8. Create Datasets

In [ ]:
# from torch.utils.data import DataLoader

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=PER_DEVICE_BATCH_SIZE,
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True,
#     drop_last=True
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=PER_DEVICE_BATCH_SIZE,
#     shuffle=False,
#     num_workers=2,
#     pin_memory=True,
#     drop_last=True
# )

# print("Train batches per epoch:", len(train_loader))
# print("Validation batches:", len(val_loader))
# print("Batch size:", PER_DEVICE_BATCH_SIZE)
# print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)

In [ ]:
# #SANITY CHECK
# batch = next(iter(train_loader))

# print("Input shape:", batch["input_ids"].shape)
# print("Label shape:", batch["labels"].shape)

# print(
#     "Minimum token ID:",
#     batch["input_ids"].min().item()
# )

# print(
#     "Maximum token ID:",
#     batch["input_ids"].max().item()
# )

# assert batch["input_ids"].shape == (
#     PER_DEVICE_BATCH_SIZE,
#     TRAIN_BLOCK_SIZE
# )
# assert batch["labels"].shape == (
#     PER_DEVICE_BATCH_SIZE,
#     TRAIN_BLOCK_SIZE
# )
# assert batch["input_ids"].min() >= 0
# assert batch["input_ids"].max() < tokenizer.vocab_size

# print("\nDataLoader sanity check passed.")

## 9. Define The Model

In [ ]:
# 9. V1 Decoder-Only Transformer
# This is our first architecture built from scratch.
# V1 uses:
# - RMSNorm
# - RoPE
# - Grouped Query Attention
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # Apply RoPE to the new Q and K positions.
        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # Add the new K/V to the cached K/V during generation.
        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # GQA: each KV head is shared by a group of query heads.
        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # During training, each token can only attend to
        # itself and tokens before it.
        #
        # IMPORTANT:
        # For scaled_dot_product_attention, True means
        # "this position is allowed to attend".
        if past_key_value is None:
            attn_mask = torch.tril(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                )
            )
        else:
            # During cached decoding, all cached positions
            # are valid keys for the current token.
            attn_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention followed by a residual connection.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN followed by a residual connection.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V1 Language Model
# ---------------------------------------------------------

class V1LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tie the LM head to the token embedding weights.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model = V1LegalLM(
    vocab_size=tokenizer.vocab_size,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    max_seq_len=MODEL_CONTEXT_LENGTH,
    ffn_multiplier=FFN_MULTIPLIER,
    dropout=DROPOUT
).to(DEVICE)

print("V1 model created successfully.")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Layers: {N_LAYER}")
print(f"Embedding dimension: {N_EMBD}")
print(f"Attention heads: {N_HEAD}")
print(f"KV heads: {N_KV_HEAD}")
print(f"Context length: {MODEL_CONTEXT_LENGTH}")

In [ ]:
# Restore the trained Phase 1 V1 weights before any Phase 2 adaptation.
# Set V1_CHECKPOINT_PATH to an exact path if automatic discovery is ambiguous.
V1_CHECKPOINT_PATH = ""

if V1_CHECKPOINT_PATH:
    if not os.path.isfile(V1_CHECKPOINT_PATH):
        raise FileNotFoundError(
            f"The configured V1 checkpoint does not exist: {V1_CHECKPOINT_PATH}"
        )
    checkpoint_path = V1_CHECKPOINT_PATH
else:
    checkpoint_search_roots = ["/kaggle/input", "/kaggle/working"]
    checkpoint_candidates = []
    target_names = ["v1_model.pt", "best.pt", "latest.pt"]

    for search_root in checkpoint_search_roots:
        if os.path.isdir(search_root):
            for root, dirs, files in os.walk(search_root):
                for target_name in target_names:
                    if target_name in files:
                        checkpoint_candidates.append(
                            (target_names.index(target_name),
                             os.path.join(root, target_name))
                        )

    checkpoint_candidates.sort(key=lambda item: (item[0], item[1]))
    if not checkpoint_candidates:
        raise FileNotFoundError(
            "Could not find v1_model.pt, best.pt, or latest.pt under "
            "/kaggle/input or /kaggle/working. Attach your trained Phase 1 V1 "
            "checkpoint as a Kaggle input, or set V1_CHECKPOINT_PATH explicitly."
        )

    best_priority = checkpoint_candidates[0][0]
    best_matches = [
        path for priority, path in checkpoint_candidates
        if priority == best_priority
    ]
    if len(best_matches) > 1:
        raise RuntimeError(
            "Multiple checkpoint files with the same preferred filename were found. "
            "Set V1_CHECKPOINT_PATH to the exact checkpoint you intend to use:\n"
            + "\n".join(best_matches)
        )
    checkpoint_path = best_matches[0]

print("Loading V1 checkpoint:", checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location="cpu")

if not isinstance(checkpoint, dict):
    raise TypeError("Expected the V1 checkpoint to be a dictionary.")

checkpoint_config = checkpoint.get("config")
expected_config = {
    "vocab_size": tokenizer.vocab_size,
    "n_embd": N_EMBD,
    "n_layer": N_LAYER,
    "n_head": N_HEAD,
    "n_kv_head": N_KV_HEAD,
    "ffn_multiplier": FFN_MULTIPLIER,
    "dropout": DROPOUT,
    "context_length": MODEL_CONTEXT_LENGTH,
}

if checkpoint_config is not None:
    mismatches = {}
    for key, expected_value in expected_config.items():
        if key in checkpoint_config and checkpoint_config[key] != expected_value:
            mismatches[key] = {
                "checkpoint": checkpoint_config[key],
                "notebook": expected_value,
            }
    if mismatches:
        raise ValueError(
            "Checkpoint configuration does not match this notebook: "
            + json.dumps(mismatches, indent=2)
        )
    print("Checkpoint configuration checks passed.")
else:
    print("No saved config found in checkpoint; strict weight-shape checks will be used.")

if "model_state_dict" in checkpoint:
    v1_state_dict = checkpoint["model_state_dict"]
elif all(
    isinstance(key, str) and torch.is_tensor(value)
    for key, value in checkpoint.items()
):
    # Support a raw state_dict checkpoint.
    v1_state_dict = checkpoint
else:
    raise KeyError(
        "Checkpoint has no 'model_state_dict' and is not a raw model state_dict. "
        f"Available top-level keys: {list(checkpoint.keys())}"
    )

# strict=True deliberately prevents silent partial or mismatched loading.
model.load_state_dict(v1_state_dict, strict=True)
model.to(DEVICE)
model.eval()

del v1_state_dict
del checkpoint
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Trained Phase 1 V1 weights restored successfully.")
print("Model device:", next(model.parameters()).device)
print("Model in evaluation mode:", not model.training)

## Phase 2: Load and validate the official L-NER annotations

This section loads the two labelled folds and fold-3 test documents, checks document IDs and annotation formats, and creates a reproducible held-out validation split. It does not define or train an NER model.

In [ ]:
from collections import Counter

EXPECTED_PHASE2_LABELS = {
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
}


def _load_json_file(filename):
    with open(PHASE2_FILES[filename], "r", encoding="utf-8") as handle:
        return json.load(handle)


def _normalise_text_map(raw_data, filename):
    # The Phase 2 specification describes text files as document-ID -> full text mappings.
    if not isinstance(raw_data, dict):
        raise TypeError(f"{filename} should contain a JSON object keyed by document ID; got {type(raw_data).__name__}.")
    result = {}
    for document_id, value in raw_data.items():
        if isinstance(value, str):
            text_value = value
        elif isinstance(value, dict) and isinstance(value.get("text"), str):
            # Supports a wrapped record if the published file uses that variant.
            text_value = value["text"]
        else:
            raise TypeError(
                f"Unexpected text record for document {document_id!r} in {filename}: {type(value).__name__}."
            )
        result[str(document_id)] = text_value
    return result


def _first_present(record, keys, field_name, document_id):
    for key in keys:
        if key in record:
            return record[key]
    raise KeyError(
        f"Missing {field_name!r} field for document {document_id!r}. Expected one of {keys}; got keys {list(record)}."
    )


def _normalise_entities_for_document(raw_entities, document_id, filename):
    # Supports the published [start, end, label, optional_text] format and dictionary variants.
    if raw_entities is None:
        return []
    if isinstance(raw_entities, dict):
        if "entities" in raw_entities:
            raw_entities = raw_entities["entities"]
        elif any(k in raw_entities for k in ("start", "start_char", "start_offset")):
            raw_entities = [raw_entities]
        else:
            raise TypeError(
                f"Unexpected annotation object for document {document_id!r} in {filename}: {list(raw_entities)[:12]}"
            )
    if not isinstance(raw_entities, (list, tuple)):
        raise TypeError(
            f"Annotations for document {document_id!r} in {filename} should be a list; got {type(raw_entities).__name__}."
        )

    entities = []
    for entity_index, entity in enumerate(raw_entities):
        if isinstance(entity, dict):
            start = _first_present(entity, ("start", "start_char", "start_offset", "begin"), "start", document_id)
            end = _first_present(entity, ("end", "end_char", "end_offset", "stop"), "end", document_id)
            label = _first_present(entity, ("label", "entity_label", "type", "entity_type"), "label", document_id)
            entity_text = entity.get("text", entity.get("entity_text", entity.get("surface")))
        elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
            start, end, label = entity[:3]
            entity_text = entity[3] if len(entity) >= 4 else None
        else:
            raise TypeError(
                f"Unexpected entity at index {entity_index} for document {document_id!r} in {filename}: {entity!r}"
            )

        try:
            start = int(start)
            end = int(end)
        except (TypeError, ValueError) as exc:
            raise TypeError(
                f"Non-integer span boundaries for document {document_id!r}, entity {entity_index}: {entity!r}"
            ) from exc
        if not isinstance(label, str):
            raise TypeError(f"Entity label must be a string, got {type(label).__name__}: {entity!r}")
        entities.append({"start": start, "end": end, "label": label, "text": entity_text})
    return entities


def _normalise_gold_map(raw_data, filename):
    if not isinstance(raw_data, dict):
        raise TypeError(f"{filename} should contain a JSON object keyed by document ID; got {type(raw_data).__name__}.")
    return {
        str(document_id): _normalise_entities_for_document(entities, str(document_id), filename)
        for document_id, entities in raw_data.items()
    }


with open(PHASE2_FILES["labels.txt"], "r", encoding="utf-8") as handle:
    labels_lines = [line.strip() for line in handle if line.strip() and not line.strip().startswith("#")]

# Accept either one label per line or a label followed by a human-readable description.
LABELS = []
for line in labels_lines:
    first_token = line.split()[0]
    if first_token in EXPECTED_PHASE2_LABELS:
        candidate = first_token
    elif line in EXPECTED_PHASE2_LABELS:
        candidate = line
    else:
        continue
    if candidate not in LABELS:
        LABELS.append(candidate)

if set(LABELS) != EXPECTED_PHASE2_LABELS:
    raise ValueError(
        "labels.txt did not resolve to the expected 12 Phase 2 labels. "
        f"Found={LABELS}; missing={sorted(EXPECTED_PHASE2_LABELS - set(LABELS))}; "
        f"unexpected={sorted(set(LABELS) - EXPECTED_PHASE2_LABELS)}"
    )

FOLD1_TEXTS = _normalise_text_map(_load_json_file("fold1-text.json"), "fold1-text.json")
FOLD1_GOLD = _normalise_gold_map(_load_json_file("fold1-gold.json"), "fold1-gold.json")
FOLD2_TEXTS = _normalise_text_map(_load_json_file("fold2-text.json"), "fold2-text.json")
FOLD2_GOLD = _normalise_gold_map(_load_json_file("fold2-gold.json"), "fold2-gold.json")
TEST_TEXTS = _normalise_text_map(_load_json_file("fold3-text.json"), "fold3-text.json")

for fold_name, texts, gold in (
    ("fold1", FOLD1_TEXTS, FOLD1_GOLD),
    ("fold2", FOLD2_TEXTS, FOLD2_GOLD),
):
    missing_gold = sorted(set(texts) - set(gold))
    orphan_gold = sorted(set(gold) - set(texts))
    if missing_gold or orphan_gold:
        raise ValueError(
            f"{fold_name} text/gold document IDs do not match. "
            f"Missing gold for {len(missing_gold)} IDs; gold without text for {len(orphan_gold)} IDs. "
            f"First missing={missing_gold[:5]}, first orphan={orphan_gold[:5]}"
        )

TEXTS_BY_ID = {}
ANNOTATIONS_BY_ID = {}
for fold_name, texts, gold in (
    ("fold1", FOLD1_TEXTS, FOLD1_GOLD),
    ("fold2", FOLD2_TEXTS, FOLD2_GOLD),
):
    overlap = set(TEXTS_BY_ID) & set(texts)
    if overlap:
        raise ValueError(
            f"Document IDs occur in both training folds ({len(overlap)} overlaps). "
            f"Example IDs: {sorted(overlap)[:10]}"
        )
    TEXTS_BY_ID.update(texts)
    ANNOTATIONS_BY_ID.update(gold)

TEST_OVERLAP = set(TEXTS_BY_ID) & set(TEST_TEXTS)
if TEST_OVERLAP:
    raise ValueError(
        f"Training and test document IDs overlap ({len(TEST_OVERLAP)} IDs). Example: {sorted(TEST_OVERLAP)[:10]}"
    )

# Strictly validate offsets and labels before tokenization; do not silently repair annotations.
ANNOTATION_COUNTS = Counter()
INVALID_SPANS = []
ENTITY_TEXT_MISMATCHES = []
for document_id, entities in ANNOTATIONS_BY_ID.items():
    source_text = TEXTS_BY_ID[document_id]
    for entity_index, entity in enumerate(entities):
        start, end, label = entity["start"], entity["end"], entity["label"]
        ANNOTATION_COUNTS[label] += 1
        if label not in EXPECTED_PHASE2_LABELS:
            INVALID_SPANS.append((document_id, entity_index, "unknown_label", label))
        if not (0 <= start < end <= len(source_text)):
            INVALID_SPANS.append((document_id, entity_index, "invalid_offsets", start, end, len(source_text)))
        elif entity["text"] is not None and source_text[start:end] != entity["text"]:
            ENTITY_TEXT_MISMATCHES.append((document_id, entity_index, entity["text"], source_text[start:end]))

if INVALID_SPANS:
    raise ValueError(
        f"Found {len(INVALID_SPANS)} invalid annotations. Examples: {INVALID_SPANS[:10]}"
    )

print("Official labels:", LABELS)
print(f"Fold 1: {len(FOLD1_TEXTS):,} documents, {sum(map(len, FOLD1_GOLD.values())):,} entities")
print(f"Fold 2: {len(FOLD2_TEXTS):,} documents, {sum(map(len, FOLD2_GOLD.values())):,} entities")
print(f"Combined training pool: {len(TEXTS_BY_ID):,} documents")
print(f"Fold 3 test documents: {len(TEST_TEXTS):,}")
print("Entity counts:")
for label in LABELS:
    print(f"  {label:10s} {ANNOTATION_COUNTS[label]:,}")
print("Optional entity_text mismatches (reported only, offsets preserved):", len(ENTITY_TEXT_MISMATCHES))
if ENTITY_TEXT_MISMATCHES:
    print("Mismatch examples:", ENTITY_TEXT_MISMATCHES[:3])

# Deterministic document-level multilabel-stratified holdout.
# We stratify by whether a label occurs in a document, not by window, so rare labels are
# more likely to be represented in validation without leaking windows from one document.
import random

VALIDATION_FRACTION = 0.10
all_document_ids = list(TEXTS_BY_ID)
if len(all_document_ids) < 2:
    raise ValueError("At least two labelled documents are required to create train/validation splits.")

_split_rng = random.Random(SEED)
_split_rng.shuffle(all_document_ids)
_doc_label_sets = {
    str(doc_id): {str(ent["label"]) for ent in ANNOTATIONS_BY_ID.get(str(doc_id), [])}
    for doc_id in all_document_ids
}
_label_doc_frequency = Counter(
    label for labels_for_doc in _doc_label_sets.values() for label in labels_for_doc
)
_num_validation = max(1, int(round(len(all_document_ids) * VALIDATION_FRACTION)))
_num_validation = min(_num_validation, len(all_document_ids) - 1)
_label_val_targets = {
    label: (min(freq - 1, max(1, int(round(freq * VALIDATION_FRACTION)))) if freq >= 2 else 0)
    for label, freq in _label_doc_frequency.items()
}
_val_selected = []
_val_label_counts = Counter()
_remaining_ids = list(all_document_ids)

while len(_val_selected) < _num_validation and _remaining_ids:
    _deficits = {
        label: max(0, target - _val_label_counts[label])
        for label, target in _label_val_targets.items()
    }
    def _stratification_score(doc_id):
        _labels = _doc_label_sets[str(doc_id)]
        _score = sum((_deficits.get(label, 0) / max(_label_doc_frequency.get(label, 1), 1))
                     for label in _labels)
        # Tiny fixed-seed jitter makes ties deterministic but avoids systematic ID-order bias.
        return _score + (_split_rng.random() * 1e-9)
    _chosen = max(_remaining_ids, key=_stratification_score)
    _remaining_ids.remove(_chosen)
    _val_selected.append(_chosen)
    _val_label_counts.update(_doc_label_sets[str(_chosen)])

VAL_DOC_IDS = _val_selected
TRAIN_DOC_IDS = _remaining_ids

print(f"\nDocument split: train={len(TRAIN_DOC_IDS):,}, validation={len(VAL_DOC_IDS):,}")
print("Train/validation ID overlap:", len(set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS)))
assert not (set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS))
print("Document-level label support (train / validation):")
for _label in LABELS:
    _train_support = sum(_label in _doc_label_sets[str(doc_id)] for doc_id in TRAIN_DOC_IDS)
    _val_support = sum(_label in _doc_label_sets[str(doc_id)] for doc_id in VAL_DOC_IDS)
    _freq = _label_doc_frequency.get(_label, 0)
    print(f"  {_label:12s} {_train_support:5d} / {_val_support:5d} documents (total={_freq})")
    if _freq >= 2 and _val_support == 0:
        print(f"  [WARN] {_label} is absent from validation despite occurring in {_freq} documents.")


## Verify tokenizer-compatible character offsets

The V1 tokenizer splits on whitespace, encodes UTF-8 bytes, and applies the stored merge rules. The helper below reproduces those same ID decisions while tracking each token's source-character span. It checks that its token IDs exactly match the existing tokenizer before any alignment result is trusted.

In [ ]:
import re


def _merge_two_offset_items(left_item, right_item, merged_token_id):
    spans = [
        (left_item[1], left_item[2]),
        (right_item[1], right_item[2]),
    ]
    covered = [(start, end) for start, end in spans if start is not None and end is not None]
    if not covered:
        return (merged_token_id, None, None)
    return (
        merged_token_id,
        min(start for start, _ in covered),
        max(end for _, end in covered),
    )


def _encode_word_with_offsets(word, word_start, tokenizer):
    items = []
    char_position = word_start
    for character in word:
        char_start = char_position
        char_end = char_position + len(character)
        for byte_value in character.encode("utf-8"):
            # This mirrors the existing ByteLevelBPE25k._encode_word implementation.
            items.append((4 + byte_value, char_start, char_end))
        char_position = char_end

    # The boundary token has no independent character span. If merged with word bytes,
    # the resulting token retains only the source characters occupied by those bytes.
    items.append((tokenizer.word_boundary_id, None, None))

    while len(items) > 1:
        best_position = None
        best_token_id = None
        for position in range(len(items) - 1):
            pair = (items[position][0], items[position + 1][0])
            merged_id = tokenizer.merge_to_id.get(pair)
            if merged_id is None:
                continue
            if best_token_id is None or merged_id < best_token_id:
                best_token_id = merged_id
                best_position = position
        if best_position is None:
            break
        position = best_position
        merged_item = _merge_two_offset_items(
            items[position], items[position + 1], best_token_id
        )
        items = items[:position] + [merged_item] + items[position + 2:]

    return items


def encode_with_offsets(text, tokenizer, add_bos=False, add_eos=False, verify_ids=True):
    """Return (token_ids, offsets), with offsets as Python-character [start, end) spans or None."""
    token_ids = []
    offsets = []

    if add_bos:
        token_ids.append(tokenizer.bos_token_id)
        offsets.append(None)

    matches = list(re.finditer(r"\S+", text, flags=re.UNICODE))
    regex_words = [match.group(0) for match in matches]
    if regex_words != text.split():
        raise RuntimeError("Whitespace matching differs from tokenizer.encode(text); refusing unsafe offsets.")

    for match in matches:
        word_items = _encode_word_with_offsets(match.group(0), match.start(), tokenizer)
        token_ids.extend(item[0] for item in word_items)
        offsets.extend(
            (item[1], item[2]) if item[1] is not None and item[2] is not None else None
            for item in word_items
        )

    if add_eos:
        token_ids.append(tokenizer.eos_token_id)
        offsets.append(None)

    if verify_ids:
        reference_ids = tokenizer.encode(text, add_bos=add_bos, add_eos=add_eos)
        if token_ids != reference_ids:
            raise AssertionError(
                "Offset-aware tokenization produced different token IDs from the original tokenizer."
            )
    if len(token_ids) != len(offsets):
        raise AssertionError("Token-ID/offset length mismatch.")
    return token_ids, offsets


OFFSET_TEST_STRINGS = [
    "The Supreme Court of India has jurisdiction over constitutional matters.",
    "  Case No. 123/2024\n\nCounsel for the appellant appeared.",
    "Unicode check: न्यायालय and café.",
    "tabs\tand multiple    spaces must preserve original character positions.",
]
for offset_test_text in OFFSET_TEST_STRINGS:
    test_ids, test_offsets = encode_with_offsets(offset_test_text, tokenizer)
    assert test_ids == tokenizer.encode(offset_test_text)
    assert len(test_ids) == len(test_offsets)
    for span in test_offsets:
        if span is not None:
            start, end = span
            assert 0 <= start < end <= len(offset_test_text)
print(f"Offset-aware tokenization checks passed for {len(OFFSET_TEST_STRINGS)} examples.")


## Audit exact-span alignment before choosing a prediction head

A token-aligned classifier can directly represent an entity boundary only when that boundary is present in the token offsets. This audit samples labelled documents and reports how often gold starts and ends align with token boundaries. It does not change annotations or decide the model architecture.

In [ ]:
def audit_span_alignment(document_ids, max_documents=30):
    rng = random.Random(SEED + 17)
    ids = list(document_ids)
    if len(ids) > max_documents:
        ids = rng.sample(ids, max_documents)

    total_entities = 0
    start_aligned = 0
    end_aligned = 0
    both_aligned = 0
    examples_not_aligned = []
    sampled_token_count = 0
    sampled_char_count = 0

    for document_id in ids:
        text = TEXTS_BY_ID[document_id]
        _, offsets = encode_with_offsets(text, tokenizer)
        valid_offsets = [span for span in offsets if span is not None]
        starts = {start for start, _ in valid_offsets}
        ends = {end for _, end in valid_offsets}
        sampled_token_count += len(offsets)
        sampled_char_count += len(text)

        for entity in ANNOTATIONS_BY_ID[document_id]:
            total_entities += 1
            start_ok = entity["start"] in starts
            end_ok = entity["end"] in ends
            start_aligned += int(start_ok)
            end_aligned += int(end_ok)
            both_aligned += int(start_ok and end_ok)
            if not (start_ok and end_ok) and len(examples_not_aligned) < 10:
                examples_not_aligned.append({
                    "document_id": document_id,
                    "gold_span": (entity["start"], entity["end"], entity["label"]),
                    "entity_text": text[entity["start"]:entity["end"]],
                    "start_aligned": start_ok,
                    "end_aligned": end_ok,
                })

    if total_entities == 0:
        raise ValueError("No entities found in the sampled documents; inspect the annotation structure.")

    report = {
        "sampled_documents": len(ids),
        "sampled_characters": sampled_char_count,
        "sampled_tokens": sampled_token_count,
        "entities_checked": total_entities,
        "start_boundary_alignment_rate": start_aligned / total_entities,
        "end_boundary_alignment_rate": end_aligned / total_entities,
        "both_boundary_alignment_rate": both_aligned / total_entities,
        "examples_not_aligned": examples_not_aligned,
    }
    return report


SPAN_ALIGNMENT_AUDIT = audit_span_alignment(TRAIN_DOC_IDS, max_documents=30)
print("Documents audited:", SPAN_ALIGNMENT_AUDIT["sampled_documents"])
print("Entities checked:", SPAN_ALIGNMENT_AUDIT["entities_checked"])
print("Start-boundary alignment rate:", f"{SPAN_ALIGNMENT_AUDIT['start_boundary_alignment_rate']:.2%}")
print("End-boundary alignment rate:  ", f"{SPAN_ALIGNMENT_AUDIT['end_boundary_alignment_rate']:.2%}")
print("Both-boundaries alignment rate:", f"{SPAN_ALIGNMENT_AUDIT['both_boundary_alignment_rate']:.2%}")
print("Examples requiring closer inspection:")
for example in SPAN_ALIGNMENT_AUDIT["examples_not_aligned"]:
    print(example)


## Phase 2 preprocessing: character spans to token labels

This section prepares labelled windows and reports spans that cannot be represented exactly by the existing tokenizer. It does **not** add an NER model head or a training algorithm. The notebook's existing token IDs and V1 checkpoint are left unchanged. Review the alignment counts before choosing a token-aligned prediction method.

In [ ]:
# Phase 2 label vocabulary and span-to-token alignment utilities.
# Unaligned entities receive approximate BIO targets.
# Conflicting token labels are masked using IGNORE_INDEX.

IGNORE_INDEX = -100

BIO_LABELS = (
    ["O"]
    + [f"B-{label}" for label in LABELS]
    + [f"I-{label}" for label in LABELS]
)

BIO_TO_ID = {name: idx for idx, name in enumerate(BIO_LABELS)}
ID_TO_BIO = {idx: name for name, idx in BIO_TO_ID.items()}
O_LABEL_ID = BIO_TO_ID["O"]


def _overlapping_token_indices(entity, offsets):
    start, end = entity["start"], entity["end"]

    return [
        i
        for i, span in enumerate(offsets)
        if span is not None and span[0] < end and span[1] > start
    ]


def make_bio_targets(document_id, token_offsets, entities):
    """
    Create BIO targets for exact and non-exact entity boundaries.

    Exact spans:
        Use the original exact token boundaries.

    Non-exact spans:
        Approximate using all valid tokens overlapping the gold span.

    Conflicting token labels:
        Mask conflicting positions with IGNORE_INDEX.
    """
    token_label_ids = [
        O_LABEL_ID if span is not None else IGNORE_INDEX
        for span in token_offsets
    ]

    proposed_labels = {}
    ignored_token_indices = set()

    report = {
        "document_id": str(document_id),
        "entity_count": len(entities),
        "exactly_aligned_entities": 0,
        "unaligned_entities": 0,
        "entities_without_overlapping_tokens": 0,
        "overlapping_entity_conflicts": 0,
        "approximated_entities": 0,
        "unaligned_examples": [],
    }

    for entity in entities:
        start = entity["start"]
        end = entity["end"]
        label = entity["label"]

        if label not in LABELS:
            raise ValueError(f"Unknown entity label: {label}")

        if not (0 <= start < end):
            raise ValueError(
                f"Invalid entity span for document {document_id}: "
                f"({start}, {end}, {label})"
            )

        overlapping = _overlapping_token_indices(entity, token_offsets)

        if not overlapping:
            report["entities_without_overlapping_tokens"] += 1
            report["unaligned_entities"] += 1

            if len(report["unaligned_examples"]) < 5:
                report["unaligned_examples"].append(
                    (start, end, label, "no_token_overlap")
                )
            continue

        first_index = overlapping[0]
        last_index = overlapping[-1]

        exact_alignment = (
            token_offsets[first_index][0] == start
            and token_offsets[last_index][1] == end
        )

        if exact_alignment:
            report["exactly_aligned_entities"] += 1
        else:
            report["unaligned_entities"] += 1
            report["approximated_entities"] += 1

            if len(report["unaligned_examples"]) < 5:
                report["unaligned_examples"].append(
                    (start, end, label, "approximated")
                )

        target_ids = (
            [BIO_TO_ID[f"B-{label}"]]
            + [BIO_TO_ID[f"I-{label}"]] * (len(overlapping) - 1)
        )

        proposal = {
            idx: target_id
            for idx, target_id in zip(overlapping, target_ids)
        }

        # Check conflicts before committing this entity's labels.
        conflict_indices = [
            idx
            for idx, target_id in proposal.items()
            if idx in proposed_labels
            and proposed_labels[idx] != target_id
        ]

        if conflict_indices:
            report["overlapping_entity_conflicts"] += 1

            # Mask all tokens participating in the conflicting annotations.
            ignored_token_indices.update(proposal)
            ignored_token_indices.update(
                idx for idx in proposal if idx in proposed_labels
            )
            ignored_token_indices.update(conflict_indices)

            continue

        # Do not allow later entities to overwrite tokens already
        # identified as ambiguous.
        for idx, target_id in proposal.items():
            if idx not in ignored_token_indices:
                proposed_labels[idx] = target_id

    for idx, target_id in proposed_labels.items():
        if idx not in ignored_token_indices:
            token_label_ids[idx] = target_id

    for idx in ignored_token_indices:
        token_label_ids[idx] = IGNORE_INDEX

    return token_label_ids, report


def prepare_lner_windows(
    document_id,
    text,
    entities,
    max_length,
    window_overlap=0,
    tokenizer=None,
):
    """
    Tokenize a document and create overlapping windows.

    Offsets refer to the original document.
    """
    if max_length <= 0:
        raise ValueError("max_length must be positive.")

    if not 0 <= window_overlap < max_length:
        raise ValueError(
            "window_overlap must satisfy 0 <= window_overlap < max_length."
        )

    active_tokenizer = (
        tokenizer if tokenizer is not None else globals()["tokenizer"]
    )

    token_ids, token_offsets = encode_with_offsets(text, active_tokenizer)

    token_targets, alignment_report = make_bio_targets(
        document_id,
        token_offsets,
        entities,
    )

    if not (
        len(token_ids) == len(token_targets) == len(token_offsets)
    ):
        raise AssertionError(
            "Token IDs, labels, and offsets must have identical lengths."
        )

    windows = []

    if not token_ids:
        return windows, alignment_report

    step = max_length - window_overlap
    token_start = 0

    while token_start < len(token_ids):
        token_end = min(token_start + max_length, len(token_ids))

        windows.append({
            "document_id": str(document_id),
            "token_start": token_start,
            "input_ids": token_ids[token_start:token_end],
            "labels": token_targets[token_start:token_end],
            "offsets": token_offsets[token_start:token_end],
        })

        if token_end == len(token_ids):
            break

        token_start += step

    return windows, alignment_report


print(
    f"BIO vocabulary size: {len(BIO_LABELS)} "
    f"(O plus B/I for {len(LABELS)} entity types)"
)
print("Special/unlabelled token positions use IGNORE_INDEX =", IGNORE_INDEX)

In [ ]:
# Verify the updated target builder on the existing training documents.

sample_ids = TRAIN_DOC_IDS[:30]
totals = {
    "entities": 0,
    "exactly_aligned": 0,
    "approximated": 0,
    "without_tokens": 0,
    "conflicts": 0,
}

for document_id in sample_ids:
    text = TEXTS_BY_ID[str(document_id)]
    entities = ANNOTATIONS_BY_ID[str(document_id)]

    _, offsets = encode_with_offsets(text, tokenizer)
    _, report = make_bio_targets(str(document_id), offsets, entities)

    totals["entities"] += report["entity_count"]
    totals["exactly_aligned"] += report["exactly_aligned_entities"]
    totals["approximated"] += report["approximated_entities"]
    totals["without_tokens"] += report["entities_without_overlapping_tokens"]
    totals["conflicts"] += report["overlapping_entity_conflicts"]

print("Updated target-builder audit")
print("=" * 40)
for key, value in totals.items():
    print(f"{key}: {value}")

print(
    "\nEntities receiving exact or approximate supervision:",
    totals["exactly_aligned"] + totals["approximated"],
    "of",
    totals["entities"],
)

## Window-level dataset objects for Phase 2

The dataset class below uses the preprocessing helpers above to build token/label windows on demand when explicitly constructed. It stores token IDs and label IDs as tensors, not the full character-offset objects. The custom collator pads on the right and masks padded targets with `IGNORE_INDEX`. Because this will later feed a token-classification head, the model/training implementation must use the labels only at valid positions.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class LNERWindowDataset(Dataset):
    """Materialize document windows for a selected set of document IDs."""
    def __init__(
        self,
        document_ids,
        text_by_id,
        entities_by_id,
        max_length,
        window_overlap_tokens,
        tokenizer,
    ):
        self.samples = []
        self.alignment_reports = []
        self.document_count = 0
        self.token_count = 0

        for document_id in document_ids:
            doc_id = str(document_id)
            text = text_by_id[doc_id]
            entities = entities_by_id[doc_id]
            windows, report = prepare_lner_windows(
                document_id=doc_id,
                text=text,
                entities=entities,
                max_length=max_length,
                window_overlap=window_overlap_tokens,
                tokenizer=tokenizer,
            )
            self.alignment_reports.append(report)
            self.document_count += 1
            for window in windows:
                input_ids = torch.tensor(window["input_ids"], dtype=torch.long)
                labels = torch.tensor(window["labels"], dtype=torch.long)
                if input_ids.numel() != labels.numel():
                    raise AssertionError(f"Input/label length mismatch for {doc_id}.")
                self.token_count += int(input_ids.numel())
                self.samples.append({
                    "input_ids": input_ids,
                    "labels": labels,
                    "document_id": doc_id,
                    "token_start": int(window["token_start"]),
                })

        self.alignment_summary = {
            "documents": self.document_count,
            "windows": len(self.samples),
            "window_tokens_including_overlap": self.token_count,
            "entities": sum(x["entity_count"] for x in self.alignment_reports),
            "exactly_aligned_entities": sum(x["exactly_aligned_entities"] for x in self.alignment_reports),
            "unaligned_entities": sum(x["unaligned_entities"] for x in self.alignment_reports),
            "overlapping_entity_conflicts": sum(x["overlapping_entity_conflicts"] for x in self.alignment_reports),
        }
        total_entities = self.alignment_summary["entities"]
        self.alignment_summary["exact_alignment_rate"] = (
            self.alignment_summary["exactly_aligned_entities"] / total_entities
            if total_entities else 0.0
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def collate_lner_windows(batch, pad_token_id):
    """Right-pad input IDs and mask pad targets; retain IDs for document-level decoding."""
    if not batch:
        raise ValueError("Cannot collate an empty batch.")
    max_len = max(item["input_ids"].numel() for item in batch)
    input_ids = torch.full(
        (len(batch), max_len), int(pad_token_id), dtype=torch.long
    )
    labels = torch.full(
        (len(batch), max_len), IGNORE_INDEX, dtype=torch.long
    )
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

    for row, item in enumerate(batch):
        length = item["input_ids"].numel()
        input_ids[row, :length] = item["input_ids"]
        labels[row, :length] = item["labels"]
        attention_mask[row, :length] = 1

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
        "document_ids": [item["document_id"] for item in batch],
        "token_starts": [item["token_start"] for item in batch],
    }


def build_phase2_window_datasets(window_overlap_tokens):
    """Explicitly build train/validation windows after the author selects overlap."""
    train_dataset = LNERWindowDataset(
        document_ids=TRAIN_DOC_IDS,
        text_by_id=TEXTS_BY_ID,
        entities_by_id=ANNOTATIONS_BY_ID,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap_tokens=window_overlap_tokens,
        tokenizer=tokenizer,
    )
    validation_dataset = LNERWindowDataset(
        document_ids=VAL_DOC_IDS,
        text_by_id=TEXTS_BY_ID,
        entities_by_id=ANNOTATIONS_BY_ID,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap_tokens=window_overlap_tokens,
        tokenizer=tokenizer,
    )
    return train_dataset, validation_dataset


print("LNERWindowDataset, collate_lner_windows, and build_phase2_window_datasets are defined.")
print("The complete train/validation tensors are NOT built until build_phase2_window_datasets(...) is called.")


## Sanity-check the Phase 2 feature builder

This previews a few labelled documents only. It does not tokenize the complete dataset in advance or start training. The output reports the number of windows and the number of gold entities whose boundaries match the tokenizer's character offsets exactly.

In [ ]:
# Small preprocessing smoke test; no model training occurs in this cell.
PHASE2_PREVIEW_DOCUMENTS = min(3, len(TRAIN_DOC_IDS))
PREPROCESSING_PREVIEW = []

for document_id in TRAIN_DOC_IDS[:PHASE2_PREVIEW_DOCUMENTS]:
    text = TEXTS_BY_ID[document_id]
    entities = ANNOTATIONS_BY_ID[document_id]
    windows, report = prepare_lner_windows(
        document_id=document_id,
        text=text,
        entities=entities,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap=0,
    )
    PREPROCESSING_PREVIEW.append({"windows": windows, "alignment": report})
    print({
        "document_id": str(document_id),
        "characters": len(text),
        "tokens": sum(len(window["input_ids"]) for window in windows),
        "windows": len(windows),
        "entities": report["entity_count"],
        "exactly_aligned_entities": report["exactly_aligned_entities"],
        "unaligned_entities": report["unaligned_entities"],
        "overlapping_entity_conflicts": report["overlapping_entity_conflicts"],
    })

# Keep only summary information in the notebook namespace after printing the sample.
PREPROCESSING_PREVIEW_SUMMARY = [item["alignment"] for item in PREPROCESSING_PREVIEW]
del PREPROCESSING_PREVIEW


## Local strict-span evaluation and submission utilities

These helpers compare exact `(start, end, label)` triples and create the required JSON/ZIP once a model has produced predictions. They do not train or choose a model. For final reporting, compare this local calculation with the official evaluator provided for the capstone.

In [ ]:
from collections import Counter


def _prediction_entity_triples(raw_entities):
    """Normalize prediction entities into (start, end, label) tuples."""
    triples = []
    for entity in raw_entities:
        if isinstance(entity, dict):
            start = entity.get("start", entity.get("start_char", entity.get("start_offset")))
            end = entity.get("end", entity.get("end_char", entity.get("end_offset")))
            label = entity.get("label", entity.get("entity_label", entity.get("type")))
        elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
            start, end, label = entity[:3]
        else:
            raise TypeError(f"Unsupported predicted entity representation: {entity!r}")
        triples.append((int(start), int(end), str(label)))
    return triples


def strict_span_metrics(gold_by_id, predicted_by_id, labels=None, document_ids=None):
    """Compute strict exact-span per-label precision/recall/F1 and macro F1.

    Inputs can use the normalized gold dictionaries already created above and predictions
    in [start, end, label] form. This is a local diagnostic, not a replacement for the
    official capstone evaluator.
    """
    labels = list(labels if labels is not None else LABELS)
    ids = list(document_ids) if document_ids is not None else sorted(
        set(map(str, gold_by_id)) | set(map(str, predicted_by_id))
    )
    gold_counts = {label: Counter() for label in labels}
    pred_counts = {label: Counter() for label in labels}

    for document_id in ids:
        doc_id = str(document_id)
        for entity in gold_by_id.get(doc_id, gold_by_id.get(document_id, [])):
            if isinstance(entity, dict):
                triple = (int(entity["start"]), int(entity["end"]), str(entity["label"]))
            else:
                triple = _prediction_entity_triples([entity])[0]
            if triple[2] in gold_counts:
                gold_counts[triple[2]][(doc_id, triple[0], triple[1], triple[2])] += 1

        for entity in predicted_by_id.get(doc_id, predicted_by_id.get(document_id, [])):
            triple = _prediction_entity_triples([entity])[0]
            if triple[2] in pred_counts:
                pred_counts[triple[2]][(doc_id, triple[0], triple[1], triple[2])] += 1

    per_label = {}
    f1_values = []
    for label in labels:
        tp = sum((gold_counts[label] & pred_counts[label]).values())
        fp = sum(pred_counts[label].values()) - tp
        fn = sum(gold_counts[label].values()) - tp
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        per_label[label] = {
            "tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1,
        }
        f1_values.append(f1)

    return {
        "strict_macro_f1": sum(f1_values) / len(f1_values) if f1_values else 0.0,
        "per_label": per_label,
        "documents_evaluated": len(ids),
    }


def write_lner_submission(predictions_by_id, prediction_path=None, zip_path=None):
    """Validate and write Phase 2 prediction.json and a ZIP with the JSON at its root."""
    prediction_path = prediction_path or PHASE2_PREDICTION_PATH
    zip_path = zip_path or PHASE2_SUBMISSION_ZIP
    normalized = {str(key): _prediction_entity_triples(value) for key, value in predictions_by_id.items()}
    expected_ids = set(map(str, TEST_TEXTS))
    received_ids = set(normalized)
    if expected_ids != received_ids:
        raise ValueError(
            "Prediction document IDs must exactly match fold3-text.json. "
            f"Missing={len(expected_ids - received_ids)}, extra={len(received_ids - expected_ids)}."
        )

    for document_id, triples in normalized.items():
        text = TEST_TEXTS[document_id]
        for start, end, label in triples:
            if label not in LABELS:
                raise ValueError(f"Unknown label {label!r} for document {document_id}.")
            if not (0 <= start < end <= len(text)):
                raise ValueError(
                    f"Invalid span {(start, end, label)} for document {document_id} with text length {len(text)}."
                )

    prediction_dir = os.path.dirname(prediction_path)
    zip_dir = os.path.dirname(zip_path)
    if prediction_dir:
        os.makedirs(prediction_dir, exist_ok=True)
    if zip_dir:
        os.makedirs(zip_dir, exist_ok=True)
    with open(prediction_path, "w", encoding="utf-8") as handle:
        json.dump(normalized, handle, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path, arcname="prediction.json")

    print("Saved predictions:", prediction_path)
    print("Saved submission ZIP:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as archive:
        print("ZIP members:", archive.namelist())


In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Parameters in millions: {total_params / 1e6:.2f}M")

assert 50_000_000 <= total_params <= 300_000_000

print("\nParameter count check passed.")

In [ ]:
# model.eval()

# test_batch = next(iter(train_loader))

# test_input_ids = test_batch["input_ids"][:2].to(DEVICE)
# test_labels = test_batch["labels"][:2].to(DEVICE)

# with torch.no_grad():
#     test_logits, _ = model(
#         test_input_ids,
#         use_cache=False
#     )

# print("Input shape:", test_input_ids.shape)
# print("Labels shape:", test_labels.shape)
# print("Logits shape:", test_logits.shape)

# expected_shape = (
#     test_input_ids.shape[0],
#     test_input_ids.shape[1],
#     tokenizer.vocab_size
# )

# assert test_logits.shape == expected_shape
# assert torch.isfinite(test_logits).all()

# print("\nForward pass check passed.")

In [ ]:
# model.train()

# logits, _ = model(
#     test_input_ids,
#     use_cache=False
# )

# loss = F.cross_entropy(
#     logits.reshape(-1, tokenizer.vocab_size),
#     test_labels.reshape(-1)
# )

# print(f"Initial loss: {loss.item():.6f}")

# assert torch.isfinite(loss), "Loss is not finite."

# # Clear any existing gradients before this test.
# model.zero_grad(set_to_none=True)

# loss.backward()

# # Check that gradients were actually produced.
# parameters_with_grad = 0
# total_grad_norm_sq = 0.0

# for parameter in model.parameters():
#     if parameter.grad is not None:
#         parameters_with_grad += 1
#         total_grad_norm_sq += (
#             parameter.grad.detach()
#             .float()
#             .norm(2)
#             .item() ** 2
#         )

# grad_norm = total_grad_norm_sq ** 0.5

# print("Parameters with gradients:", parameters_with_grad)
# print(f"Gradient norm: {grad_norm:.6f}")

# assert parameters_with_grad > 0
# assert math.isfinite(grad_norm)

# print("\nLoss and backward-pass check passed.")

In [ ]:
#Causal Mask Test
# A decoder-only model must not allow a token to use
# information from future positions.We test this by changing a future token and checking that an earlier position gives the same output.

model.eval()

torch.manual_seed(SEED)

test_ids = torch.randint(
    low=0,
    high=tokenizer.vocab_size,
    size=(1, 16),
    device=DEVICE
)

# Make a copy and change only a future token.
modified_ids = test_ids.clone()
modified_ids[0, 10] = (
    modified_ids[0, 10] + 1
) % tokenizer.vocab_size

with torch.no_grad():
    logits_original, _ = model(
        test_ids,
        use_cache=False
    )

    logits_modified, _ = model(
        modified_ids,
        use_cache=False
    )

# Position 5 cannot see position 10.
earlier_difference = (
    logits_original[0, 5]
    - logits_modified[0, 5]
).abs().max().item()

# Position 10 can see itself, so changing it should affect
# its output in general.
current_difference = (
    logits_original[0, 10]
    - logits_modified[0, 10]
).abs().max().item()

print(
    f"Difference at earlier position: {earlier_difference:.10f}"
)

print(
    f"Difference at modified position: {current_difference:.10f}"
)

assert earlier_difference < 1e-5
assert current_difference > 1e-6

print("\nCausal mask check passed.")

In [ ]:
# KV Cache Equivalence Test
# During generation, KV caching should give the same results
# as processing the complete sequence normally.
# We compare:
# 1. Full-sequence forward pass
# 2. Token-by-token forward pass using the KV cache

model.eval()

torch.manual_seed(SEED)

cache_test_ids = torch.randint(
    low=0,
    high=tokenizer.vocab_size,
    size=(1, 16),
    device=DEVICE
)

with torch.no_grad():

    # Full sequence in one forward pass.
    full_logits, _ = model(
        cache_test_ids,
        use_cache=False
    )

    # Process one token at a time while keeping the
    # previous keys and values in the cache.
    past_key_values = None
    cached_logits = []

    for position in range(
        cache_test_ids.size(1)
    ):

        current_token = cache_test_ids[
            :, position:position + 1
        ]

        logits, past_key_values = model(
            current_token,
            past_key_values=past_key_values,
            use_cache=True
        )

        cached_logits.append(logits)

    cached_logits = torch.cat(
        cached_logits,
        dim=1
    )


# Compare full-sequence and cached outputs.
max_difference = (
    full_logits - cached_logits
).abs().max().item()

mean_difference = (
    full_logits - cached_logits
).abs().mean().item()

print(
    f"Maximum logit difference: {max_difference:.10f}"
)

print(
    f"Mean logit difference:    {mean_difference:.10f}"
)

assert torch.allclose(
    full_logits,
    cached_logits,
    atol=1e-4,
    rtol=1e-4
)

print("\nKV cache equivalence check passed.")

In [ ]:
# # Tiny Overfit Test
# # Sanity check

# model.train()

# tiny_input_ids = test_batch["input_ids"][:2].to(DEVICE)
# tiny_labels = test_batch["labels"][:2].to(DEVICE)

# tiny_optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=1e-3,
#     weight_decay=0.0
# )

# initial_tiny_loss = None

# for step in range(100):

#     tiny_optimizer.zero_grad(
#         set_to_none=True
#     )

#     logits, _ = model(
#         tiny_input_ids,
#         use_cache=False
#     )

#     loss = F.cross_entropy(
#         logits.reshape(-1, tokenizer.vocab_size),
#         tiny_labels.reshape(-1)
#     )

#     if initial_tiny_loss is None:
#         initial_tiny_loss = loss.item()

#     loss.backward()

#     torch.nn.utils.clip_grad_norm_(
#         model.parameters(),
#         max_norm=1.0
#     )

#     tiny_optimizer.step()

#     if step == 0 or (step + 1) % 20 == 0:
#         print(
#             f"Step {step + 1:3d} | "
#             f"Loss: {loss.item():.6f}"
#         )

# final_tiny_loss = loss.item()

# print(
#     f"\nInitial tiny loss: {initial_tiny_loss:.6f}"
# )

# print(
#     f"Final tiny loss:   {final_tiny_loss:.6f}"
# )

# assert final_tiny_loss < initial_tiny_loss

# print("\nTiny overfit check passed.")

In [ ]:
# # Fresh V1 model for the actual training run.

# model = V1LegalLM(
#     vocab_size=tokenizer.vocab_size,
#     n_embd=N_EMBD,
#     n_layer=N_LAYER,
#     n_head=N_HEAD,
#     n_kv_head=N_KV_HEAD,
#     max_seq_len=MODEL_CONTEXT_LENGTH,
#     ffn_multiplier=FFN_MULTIPLIER,
#     dropout=DROPOUT
# ).to(DEVICE)

# print("Fresh V1 model created for training.")
# print(
#     f"Parameters: "
#     f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
# )

## 10. Training Arguments

In [ ]:
# import os
# import json
# import math
# import time

# # Fresh optimizer for the freshly initialized model.
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=LEARNING_RATE,
#     betas=(0.9, 0.95),
#     weight_decay=WEIGHT_DECAY
# )

# # Number of tokens processed in one optimizer step.
# TOKENS_PER_STEP = (
#     PER_DEVICE_BATCH_SIZE
#     * GRADIENT_ACCUMULATION_STEPS
#     * TRAIN_BLOCK_SIZE
# )

# MAX_TRAIN_STEPS = math.ceil(
#     TRAIN_TOKENS / TOKENS_PER_STEP
# )

# print(f"Tokens per optimizer step: {TOKENS_PER_STEP:,}")
# print(f"Total training tokens: {TRAIN_TOKENS:,}")
# print(f"Planned optimizer steps: {MAX_TRAIN_STEPS:,}")

# # Cosine learning-rate schedule with warmup.
# def get_learning_rate(step):
#     if step < WARMUP_STEPS:
#         return LEARNING_RATE * (
#             (step + 1) / WARMUP_STEPS
#         )

#     progress = (
#         step - WARMUP_STEPS
#     ) / max(
#         1,
#         MAX_TRAIN_STEPS - WARMUP_STEPS
#     )

#     progress = min(max(progress, 0.0), 1.0)

#     return LEARNING_RATE * 0.5 * (
#         1.0 + math.cos(math.pi * progress)
#     )

# print("Training setup ready.")

In [ ]:
# # Checkpoint Setup

# CHECKPOINT_DIR = os.path.join(
#     OUTPUT_DIR,
#     "checkpoints"
# )

# os.makedirs(
#     CHECKPOINT_DIR,
#     exist_ok=True
# )

# LATEST_CHECKPOINT = os.path.join(
#     CHECKPOINT_DIR,
#     "latest.pt"
# )

# BEST_CHECKPOINT = os.path.join(
#     CHECKPOINT_DIR,
#     "best.pt"
# )

# history = {
#     "step": [],
#     "train_loss": [],
#     "val_loss": [],
#     "val_perplexity": [],
#     "learning_rate": [],
#     "tokens_seen": []
# }

# best_val_loss = float("inf")

# print("Checkpoint directory:", CHECKPOINT_DIR)

In [ ]:
# # ---------------------------------------------------------
# # 12. Validation
# # ---------------------------------------------------------
# # Evaluate the model without updating its weights.
# # ---------------------------------------------------------

# @torch.no_grad()
# def evaluate_model():
#     model.eval()

#     total_loss = 0.0
#     total_tokens = 0

#     for batch in val_loader:

#         input_ids = batch["input_ids"].to(
#             DEVICE,
#             non_blocking=True
#         )

#         labels = batch["labels"].to(
#             DEVICE,
#             non_blocking=True
#         )

#         logits, _ = model(
#             input_ids,
#             use_cache=False
#         )

#         loss = F.cross_entropy(
#             logits.reshape(
#                 -1,
#                 tokenizer.vocab_size
#             ),
#             labels.reshape(-1)
#         )

#         tokens = labels.numel()

#         total_loss += (
#             loss.item() * tokens
#         )

#         total_tokens += tokens

#     val_loss = (
#         total_loss / total_tokens
#     )

#     val_perplexity = math.exp(
#         min(val_loss, 20)
#     )

#     model.train()

#     return val_loss, val_perplexity

# print("Validation function ready.")

## 11. Training

In [ ]:
# # V1 training loop

# model.train()

# global_step = 0
# tokens_seen = 0
# start_time = time.time()

# optimizer.zero_grad(set_to_none=True)

# train_iterator = iter(train_loader)

# while global_step < MAX_TRAIN_STEPS:

#     accumulated_loss = 0.0

#     for micro_step in range(GRADIENT_ACCUMULATION_STEPS):

#         batch = next(train_iterator)

#         input_ids = batch["input_ids"].to(
#             DEVICE,
#             non_blocking=True
#         )

#         labels = batch["labels"].to(
#             DEVICE,
#             non_blocking=True
#         )

#         logits, _ = model(
#             input_ids,
#             use_cache=False
#         )

#         loss = F.cross_entropy(
#             logits.reshape(-1, tokenizer.vocab_size),
#             labels.reshape(-1)
#         )

#         accumulated_loss += loss.item()

#         loss = loss / GRADIENT_ACCUMULATION_STEPS
#         loss.backward()

#     current_lr = get_learning_rate(global_step)

#     for param_group in optimizer.param_groups:
#         param_group["lr"] = current_lr

#     grad_norm = torch.nn.utils.clip_grad_norm_(
#         model.parameters(),
#         MAX_GRAD_NORM
#     )

#     optimizer.step()
#     optimizer.zero_grad(set_to_none=True)

#     global_step += 1
#     tokens_seen += TOKENS_PER_STEP

#     train_loss = (
#         accumulated_loss / GRADIENT_ACCUMULATION_STEPS
#     )

#     if global_step == 1 or global_step % LOG_EVERY_STEPS == 0:

#         elapsed = time.time() - start_time
#         tokens_per_second = tokens_seen / max(elapsed, 1e-6)

#         print(
#             f"Step {global_step:,}/{MAX_TRAIN_STEPS:,} | "
#             f"Loss: {train_loss:.4f} | "
#             f"LR: {current_lr:.2e} | "
#             f"Grad: {grad_norm.item():.3f} | "
#             f"Tokens: {tokens_seen:,} | "
#             f"Tok/s: {tokens_per_second:,.0f}"
#         )

#         history["step"].append(global_step)
#         history["train_loss"].append(train_loss)
#         history["learning_rate"].append(current_lr)
#         history["tokens_seen"].append(tokens_seen)

#     if (
#         global_step % EVAL_EVERY_STEPS == 0
#         or global_step == MAX_TRAIN_STEPS
#     ):

#         val_loss, val_ppl = evaluate_model()

#         print(
#             f"\nValidation | "
#             f"Step {global_step:,} | "
#             f"Loss: {val_loss:.4f} | "
#             f"Perplexity: {val_ppl:.2f}\n"
#         )

#         history["val_loss"].append(val_loss)
#         history["val_perplexity"].append(val_ppl)

#         if val_loss < best_val_loss:

#             best_val_loss = val_loss

#             torch.save(
#                 {
#                     "step": global_step,
#                     "model_state_dict": model.state_dict(),
#                     "optimizer_state_dict": optimizer.state_dict(),
#                     "best_val_loss": best_val_loss,
#                     "history": history
#                 },
#                 BEST_CHECKPOINT
#             )

#             print("New best checkpoint saved.")

#     if (
#         global_step % SAVE_EVERY_STEPS == 0
#         or global_step == MAX_TRAIN_STEPS
#     ):

#         torch.save(
#             {
#                 "step": global_step,
#                 "model_state_dict": model.state_dict(),
#                 "optimizer_state_dict": optimizer.state_dict(),
#                 "best_val_loss": best_val_loss,
#                 "history": history
#             },
#             LATEST_CHECKPOINT
#         )

#         with open(
#             os.path.join(
#                 OUTPUT_DIR,
#                 "training_history.json"
#             ),
#             "w",
#             encoding="utf-8"
#         ) as f:
#             json.dump(
#                 history,
#                 f,
#                 indent=2
#             )

#         print("Latest checkpoint saved.")

# print("\nV1 training complete.")
# print(f"Best validation loss: {best_val_loss:.6f}")

In [ ]:
# # Load the checkpoint with the best validation loss

# best_checkpoint = torch.load(
#     BEST_CHECKPOINT,
#     map_location=DEVICE
# )

# model.load_state_dict(
#     best_checkpoint["model_state_dict"]
# )

# model.eval()

# best_val_loss = best_checkpoint["best_val_loss"]

# print("Best V1 checkpoint loaded.")
# print(f"Best validation loss: {best_val_loss:.6f}")
# print(f"Best checkpoint step: {best_checkpoint['step']:,}")
# print(f"Best validation perplexity: {math.exp(best_val_loss):.4f}")

## 12. Perplexity

In [ ]:
# # Perplexity from the best validation checkpoint.

# perplexity = math.exp(best_val_loss)

# print(f"Validation loss: {best_val_loss:.6f}")
# print(f"Validation perplexity: {perplexity:.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(10, 5))

# plt.plot(
#     history["step"],
#     history["train_loss"],
#     label="Training Loss"
# )

# val_steps = [
#     min(
#         (i + 1) * EVAL_EVERY_STEPS,
#         MAX_TRAIN_STEPS
#     )
#     for i in range(
#         len(history["val_loss"])
#     )
# ]

# plt.plot(
#     val_steps,
#     history["val_loss"],
#     marker="o",
#     label="Validation Loss"
# )

# plt.xlabel("Training Step")
# plt.ylabel("Loss")
# plt.title("V1 Training and Validation Loss")
# plt.legend()
# plt.grid(True)
# plt.tight_layout()

# LOSS_PLOT_PATH = os.path.join(
#     OUTPUT_DIR,
#     "v1_loss_curve.png"
# )

# plt.savefig(
#     LOSS_PLOT_PATH,
#     dpi=200,
#     bbox_inches="tight"
# )

# plt.show()

# print("Saved:", LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
# import os
# import json

# os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# final_model_path = os.path.join(
#     MODEL_OUTPUT_DIR,
#     "v1_model.pt"
# )

# torch.save(
#     {
#         "model_state_dict": model.state_dict(),
#         "config": {
#             "vocab_size": tokenizer.vocab_size,
#             "n_embd": N_EMBD,
#             "n_layer": N_LAYER,
#             "n_head": N_HEAD,
#             "n_kv_head": N_KV_HEAD,
#             "ffn_multiplier": FFN_MULTIPLIER,
#             "dropout": DROPOUT,
#             "context_length": MODEL_CONTEXT_LENGTH
#         },
#         "best_val_loss": best_val_loss,
#         "best_val_perplexity": math.exp(best_val_loss)
#     },
#     final_model_path
# )

# print("Final V1 model saved to:")
# print(final_model_path)

## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
# test_file_path = hf_hub_download(
#     repo_id=HF_DATASET_REPO,
#     filename=TEST_FILE,
#     repo_type="dataset",
#     token=HF_TOKEN
# )

# print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
# with open(test_file_path, "r", encoding="utf-8") as f:
#     test_data = json.load(f)

# print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
# # Load the best V1 model for test generation.

# generation_model = V1LegalLM(
#     vocab_size=tokenizer.vocab_size,
#     n_embd=N_EMBD,
#     n_layer=N_LAYER,
#     n_head=N_HEAD,
#     n_kv_head=N_KV_HEAD,
#     max_seq_len=MODEL_CONTEXT_LENGTH,
#     ffn_multiplier=FFN_MULTIPLIER,
#     dropout=DROPOUT
# ).to(DEVICE)

# generation_model.load_state_dict(
#     model.state_dict()
# )

# generation_model.eval()

# generation_tokenizer = tokenizer

# print("V1 generation model loaded.")
# print("Device:", DEVICE)
# print(
#     f"Parameters: "
#     f"{sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M"
# )
# print("KV cache is supported for generation.")

## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
# # Generate legal-text continuations using the V1 model and KV cache.

# def apply_repetition_penalty(logits, generated_tokens, penalty):
#     if penalty == 1.0 or len(generated_tokens) == 0:
#         return logits

#     for token_id in set(generated_tokens):
#         if logits[token_id] < 0:
#             logits[token_id] *= penalty
#         else:
#             logits[token_id] /= penalty

#     return logits


# def sample_next_token(logits):
#     logits = logits / max(TEMPERATURE, 1e-6)

#     logits[PAD_TOKEN_ID] = float("-inf")
#     logits[BOS_TOKEN_ID] = float("-inf")

#     if TOP_K > 0:
#         k = min(TOP_K, logits.numel())

#         top_values, _ = torch.topk(
#             logits,
#             k
#         )

#         cutoff = top_values[-1]
#         logits[logits < cutoff] = float("-inf")

#     if TOP_P < 1.0:
#         sorted_logits, sorted_indices = torch.sort(
#             logits,
#             descending=True
#         )

#         sorted_probs = torch.softmax(
#             sorted_logits,
#             dim=-1
#         )

#         cumulative_probs = torch.cumsum(
#             sorted_probs,
#             dim=-1
#         )

#         remove = cumulative_probs > TOP_P

#         remove[1:] = remove[:-1].clone()
#         remove[0] = False

#         sorted_logits[remove] = float("-inf")

#         logits = torch.full_like(
#             logits,
#             float("-inf")
#         )

#         logits.scatter_(
#             0,
#             sorted_indices,
#             sorted_logits
#         )

#     probabilities = torch.softmax(
#         logits,
#         dim=-1
#     )

#     return torch.multinomial(
#         probabilities,
#         num_samples=1
#     ).item()


# def generate_one(
#     prompt,
#     model,
#     tokenizer,
#     device,
#     target_words=128
# ):
#     token_ids = tokenizer.encode(prompt)

#     max_prompt_tokens = (
#         MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
#     )

#     # Keep the most recent part of the prompt if it is too long.
#     if len(token_ids) > max_prompt_tokens:
#         token_ids = token_ids[-max_prompt_tokens:]

#     input_ids = torch.tensor(
#         [token_ids],
#         dtype=torch.long,
#         device=device
#     )

#     generated_tokens = []

#     with torch.inference_mode():

#         # First pass processes the complete prompt and builds the KV cache.
#         logits, past_key_values = model(
#             input_ids,
#             use_cache=True
#         )

#         next_logits = logits[:, -1, :][0]

#         while len(generated_tokens) < MAX_GENERATION_TOKENS:

#             next_logits = apply_repetition_penalty(
#                 next_logits,
#                 generated_tokens,
#                 REPETITION_PENALTY
#             )

#             next_token = sample_next_token(
#                 next_logits.clone()
#             )

#             generated_tokens.append(
#                 next_token
#             )

#             generated_text = tokenizer.decode(
#                 generated_tokens
#             )

#             if len(generated_text.split()) >= target_words:
#                 break

#             # Do not exceed the model context window.
#             current_length = (
#                 len(token_ids)
#                 + len(generated_tokens)
#             )

#             if current_length >= MODEL_CONTEXT_LENGTH:
#                 break

#             next_input = torch.tensor(
#                 [[next_token]],
#                 dtype=torch.long,
#                 device=device
#             )

#             logits, past_key_values = model(
#                 next_input,
#                 past_key_values=past_key_values,
#                 use_cache=True
#             )

#             next_logits = logits[:, -1, :][0]

#     generated_text = tokenizer.decode(
#         generated_tokens
#     )

#     generated_words = generated_text.split()

#     return " ".join(
#         generated_words[:target_words]
#     )


# def generate_batch(
#     prompts,
#     model,
#     tokenizer,
#     device,
#     target_words=128
# ):
#     predictions = []

#     for prompt in prompts:
#         prediction = generate_one(
#             prompt=prompt,
#             model=model,
#             tokenizer=tokenizer,
#             device=device,
#             target_words=target_words
#         )

#         predictions.append(prediction)

#     return predictions


# print("Custom V1 generation functions ready.")
# print("Generation uses the KV cache.")

## 20. Generate Predictions for the Test Set

In [ ]:
# # Generate predictions for the complete test set.

# predictions = {}

# GENERATION_BATCH_SIZE = 32

# items = list(test_data.items())

# total_examples = len(items)

# print(
#     f"Total test examples: {total_examples}"
# )

# for start in range(
#     0,
#     total_examples,
#     GENERATION_BATCH_SIZE
# ):

#     batch = items[
#         start:start + GENERATION_BATCH_SIZE
#     ]

#     batch_ids = [
#         str(example_id)
#         for example_id, _ in batch
#     ]

#     batch_prompts = [
#         record["text"]
#         for _, record in batch
#     ]

#     batch_predictions = generate_batch(
#         prompts=batch_prompts,
#         model=generation_model,
#         tokenizer=generation_tokenizer,
#         device=DEVICE,
#         target_words=128
#     )

#     for example_id, prediction in zip(
#         batch_ids,
#         batch_predictions
#     ):

#         predictions[example_id] = {
#             "text": prediction
#         }

#     completed = start + len(batch)

#     print(
#         f"Completed {completed}/{total_examples} "
#         f"({100 * completed / total_examples:.1f}%)"
#     )

# print("\nTest generation complete.")
# print(
#     f"Generated predictions: {len(predictions)}"
# )

## 21. Save prediction.json

In [ ]:
# with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
#     json.dump(
#         predictions,
#         f,
#         ensure_ascii=False,
#         indent=2
#     )

# print("prediction.json saved:")
# print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
# with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
#     zipf.write(PREDICTION_PATH, arcname="prediction.json")

# print("Codabench submission created:")
# print(SUBMISSION_ZIP)

## Development status

Implemented so far: resilient HF/local Phase 2 file resolution, V1 checkpoint restoration, labelled-data loading and validation, document-level train/validation split, and exact-token-ID character-offset verification. No Phase 2 prediction head or training loop has been added yet. The next step depends on the observed span-boundary alignment audit. Keep the course's individual-implementation requirements in mind when building the model architecture.

## Phase 2: EXP-1 V1 Full Fine-Tuning Baseline Overview

This extension integrates the pretrained Phase 1 V1 Transformer into an end-to-end Legal Named Entity Recognition (L-NER) pipeline for **EXP-1: Full Fine-Tuning**, while strictly preserving all original 85 cells of the baseline notebook.

### Key Pipeline Components:
1. **Data Diagnostics & Boundary Audit (Cell 86)**: Validates annotation dictionaries, sub-token alignment, and sliding-window boundary coverage.
2. **Contextual Hidden-State Extraction (Cell 87)**: Taps the 12-layer V1 Transformer backbone to extract $N_{\text{embd}}=768$ representations.
3. **Academic Compliance & Student Architecture Point (Cells 88–89)**: Identifies the modular task-specific prediction head (`StudentLinearNERHead`) marked with `# TODO [STUDENT IMPLEMENTATION]`.
4. **Offset-Aware Dataset & Collation (Cell 90)**: Window dataset with dynamic batch padding and non-destructive checkpointing.
5. **Training Engine & Validation (Cell 91)**: Epoch training loop, gradient clipping, finite loss validation, and strict span evaluation.
6. **Strict-Span Evaluation Suite (Cell 92)**: Verification of exact vs. off-by-one span matching and per-class performance metrics.
7. **Test Inference & CodaBench Packaging (Cell 93)**: Full inference over Fold-3 test documents and generation of `submission.zip`.
8. **Pre-Flight Smoke Test Suite (Cell 94)**: Automated sanity checks verifying loss finiteness, checkpoint isolation, and submission validity.

In [ ]:
# ==============================================================================
# Phase 2 Data Diagnostics Suite: Alignment, Offsets, and Windowing Integrity
# Dependencies:
#   - Cell 15: tokenizer (ByteLevelBPE25k instance)
#   - Cell 34: TEXTS_BY_ID, ANNOTATIONS_BY_ID, TRAIN_DOC_IDS, VAL_DOC_IDS, LABELS
#   - Cell 36: encode_with_offsets(text, tokenizer, ...)
#   - Cell 40: prepare_lner_windows(document_id, text, entities, max_length, ...)
# ==============================================================================

import re
from collections import Counter


def audit_annotation_boundary_integrity(texts_by_id, annotations_by_id, expected_labels=None):
    """
    Audit gold annotations against raw document strings:
    - Supports both dictionary annotations and tuple representations
    - Identifies offsets outside document bounds or inverted (start >= end)
    - Flags unrecognized entity labels
    - Compares annotated surface text (if present) against raw document slices
    """
    valid_labels = set(expected_labels if expected_labels is not None else LABELS)
    issues = {
        "total_documents_checked": len(annotations_by_id),
        "total_entities_checked": 0,
        "valid_entities": 0,
        "out_of_bounds": [],
        "inverted_or_empty": [],
        "unknown_labels": [],
        "surface_text_mismatches": [],
    }

    for doc_id, entities in annotations_by_id.items():
        doc_text = texts_by_id.get(str(doc_id), "")
        doc_len = len(doc_text)

        for idx, ent in enumerate(entities):
            issues["total_entities_checked"] += 1
            if isinstance(ent, dict):
                start = ent.get("start", ent.get("start_char", ent.get("start_offset", ent.get("begin"))))
                end = ent.get("end", ent.get("end_char", ent.get("end_offset", ent.get("stop"))))
                label = ent.get("label", ent.get("entity_label", ent.get("type", ent.get("entity_type"))))
                surface_gold = ent.get("text", ent.get("entity_text", ent.get("surface")))
            elif isinstance(ent, (list, tuple)) and len(ent) >= 3:
                start, end, label = ent[0], ent[1], ent[2]
                surface_gold = ent[3] if len(ent) > 3 else None
            else:
                continue

            if start < 0 or end > doc_len:
                issues["out_of_bounds"].append((doc_id, idx, start, end, doc_len))
                continue

            if start >= end:
                issues["inverted_or_empty"].append((doc_id, idx, start, end))
                continue

            if label not in valid_labels:
                issues["unknown_labels"].append((doc_id, idx, label))
                continue

            if surface_gold is not None:
                actual_slice = doc_text[start:end]
                if actual_slice != surface_gold:
                    issues["surface_text_mismatches"].append({
                        "doc_id": doc_id,
                        "entity_idx": idx,
                        "start": start,
                        "end": end,
                        "expected": surface_gold,
                        "actual": actual_slice,
                    })

            issues["valid_entities"] += 1

    return issues


def audit_tokenizer_span_alignment(texts_by_id, annotations_by_id, tokenizer_obj, sample_limit=50):
    """
    Evaluates how accurately ByteLevelBPE token boundaries align with gold character spans.
    Provides diagnostics for start-boundary, end-boundary, and unaligned entities.
    """
    total_spans = 0
    start_aligned = 0
    end_aligned = 0
    both_aligned = 0
    unaligned_examples = []
    label_misalignment_counter = Counter()

    doc_ids = list(annotations_by_id.keys())[:sample_limit]

    for doc_id in doc_ids:
        doc_text = texts_by_id.get(str(doc_id), "")
        if not doc_text:
            continue

        token_ids, offsets = encode_with_offsets(doc_text, tokenizer_obj, max_len=None) if 'max_len' in encode_with_offsets.__code__.co_varnames else encode_with_offsets(doc_text, tokenizer_obj)
        valid_offsets = [
            span for span in offsets
            if span is not None
            and isinstance(span, (tuple, list))
            and len(span) >= 2
            and span[0] is not None
            and span[1] is not None
        ]
        token_starts = {s for s, e in valid_offsets}
        token_ends = {e for s, e in valid_offsets}

        for ent in annotations_by_id[doc_id]:
            total_spans += 1
            if isinstance(ent, dict):
                s, e, lbl = ent["start"], ent["end"], ent["label"]
            else:
                s, e, lbl = ent[0], ent[1], ent[2]

            s_ok = s in token_starts
            e_ok = e in token_ends

            if s_ok:
                start_aligned += 1
            if e_ok:
                end_aligned += 1
            if s_ok and e_ok:
                both_aligned += 1
            else:
                label_misalignment_counter[lbl] += 1
                if len(unaligned_examples) < 10:
                    unaligned_examples.append({
                        "doc_id": doc_id,
                        "label": lbl,
                        "span": (s, e),
                        "text": doc_text[s:e],
                        "start_aligned": s_ok,
                        "end_aligned": e_ok,
                    })

    both_rate = (both_aligned / max(1, total_spans)) * 100
    start_rate = (start_aligned / max(1, total_spans)) * 100
    end_rate = (end_aligned / max(1, total_spans)) * 100

    return {
        "total_sampled_spans": total_spans,
        "start_aligned_count": start_aligned,
        "start_aligned_rate": start_rate,
        "end_aligned_count": end_aligned,
        "end_aligned_rate": end_rate,
        "both_aligned_count": both_aligned,
        "both_aligned_rate": both_rate,
        "unaligned_by_label": dict(label_misalignment_counter),
        "sample_unaligned": unaligned_examples,
    }


def audit_windowing_boundary_integrity(texts_by_id, annotations_by_id, max_len=512, window_overlap=256, sample_limit=30):
    """
    Examines long-document chunking across sliding windows using Cell 40 prepare_lner_windows:
    - Verifies document IDs are preserved
    - Evaluates entity coverage across window splits
    """
    total_docs = 0
    total_windows = 0
    total_entities = 0
    captured_entities = 0
    boundary_crossings = 0

    doc_ids = list(annotations_by_id.keys())[:sample_limit]

    for doc_id in doc_ids:
        text = texts_by_id.get(str(doc_id), "")
        entities = annotations_by_id[doc_id]
        windows, report = prepare_lner_windows(
            document_id=doc_id,
            text=text,
            entities=entities,
            max_length=max_len,
            window_overlap=window_overlap,
            tokenizer=tokenizer
        )
        total_docs += 1
        total_windows += len(windows)
        total_entities += len(entities)

        covered_entities = set()
        for win in windows:
            valid_offsets = [
                (span[0], span[1])
                for span in win["offsets"]
                if span is not None
                and isinstance(span, (tuple, list))
                and len(span) >= 2
                and span[0] is not None
                and span[1] is not None
            ]
            if not valid_offsets:
                continue
            win_char_start = min(st for st, en in valid_offsets)
            win_char_end = max(en for st, en in valid_offsets)

            for ent_idx, ent in enumerate(entities):
                s, e, lbl = (ent["start"], ent["end"], ent["label"]) if isinstance(ent, dict) else (ent[0], ent[1], ent[2])
                if s >= win_char_start and e <= win_char_end:
                    covered_entities.add(ent_idx)
                elif (s < win_char_end and e > win_char_start) and not (s >= win_char_start and e <= win_char_end):
                    boundary_crossings += 1

        captured_entities += len(covered_entities)

    return {
        "documents_audited": total_docs,
        "total_windows_generated": total_windows,
        "gold_entities_count": total_entities,
        "fully_covered_entities": captured_entities,
        "boundary_crossings_observed": boundary_crossings,
    }


def run_full_phase2_data_diagnostics():
    print("=" * 70)
    print("PHASE 2 DATA DIAGNOSTICS & ALIGNMENT AUDIT")
    print("=" * 70)

    annot_issues = audit_annotation_boundary_integrity(TEXTS_BY_ID, ANNOTATIONS_BY_ID)
    print(f"[1] Annotation Integrity:")
    print(f"    Total Checked: {annot_issues['total_entities_checked']} across {annot_issues['total_documents_checked']} docs")
    print(f"    Valid Entities: {annot_issues['valid_entities']}")
    print(f"    Out of bounds: {len(annot_issues['out_of_bounds'])}")
    print(f"    Inverted/empty: {len(annot_issues['inverted_or_empty'])}")
    print(f"    Unknown labels: {len(annot_issues['unknown_labels'])}")
    print(f"    Surface text mismatches: {len(annot_issues['surface_text_mismatches'])}")

    align_stats = audit_tokenizer_span_alignment(TEXTS_BY_ID, ANNOTATIONS_BY_ID, tokenizer, sample_limit=50)
    print(f"[2] Tokenizer Span Alignment (50 sample docs):")
    print(f"    Total Spans Audited: {align_stats['total_sampled_spans']}")
    print(f"    Start-Boundary Aligned: {align_stats['start_aligned_count']} ({align_stats['start_aligned_rate']:.2f}%)")
    print(f"    End-Boundary Aligned:   {align_stats['end_aligned_count']} ({align_stats['end_aligned_rate']:.2f}%)")
    print(f"    Both Boundaries Aligned: {align_stats['both_aligned_count']} ({align_stats['both_aligned_rate']:.2f}%)")

    win_stats = audit_windowing_boundary_integrity(TEXTS_BY_ID, ANNOTATIONS_BY_ID, max_len=MODEL_CONTEXT_LENGTH, window_overlap=MODEL_CONTEXT_LENGTH // 2, sample_limit=30)
    print(f"[3] Windowing Integrity (30 sample docs):")
    print(f"    Docs Audited: {win_stats['documents_audited']}")
    print(f"    Windows Generated: {win_stats['total_windows_generated']}")
    print(f"    Fully Captured Entities: {win_stats['fully_covered_entities']} / {win_stats['gold_entities_count']}")
    print(f"    Boundary Crossings (Resolved via window overlap): {win_stats['boundary_crossings_observed']}")
    print("=" * 70)
    return annot_issues, align_stats, win_stats

diagnostics_results = run_full_phase2_data_diagnostics()

In [ ]:
# ==============================================================================
# Contextual Hidden-State Extraction from Trained V1 Model
# Dependencies:
#   - Cell 9: N_EMBD (768)
#   - Cell 29: V1LegalLM, RMSNorm, TransformerBlock
#   - Cell 30: DEVICE
#   - Cell 31: model (V1LegalLM instance restored from v1_model.pt)
# ==============================================================================

import torch
import torch.nn as nn


def extract_v1_hidden_states(v1_model, input_ids, attention_mask=None):
    """
    Extracts contextual representation vectors from the pretrained V1 backbone.
    Preserves token representations across all 12 Transformer blocks up to final RMSNorm.
    
    Args:
        v1_model (V1LegalLM): Pretrained Phase 1 model instance
        input_ids (torch.Tensor): LongTensor of shape (batch_size, seq_len)
        attention_mask (torch.Tensor, optional): Float/BoolTensor of shape (batch_size, seq_len)
        
    Returns:
        torch.Tensor: Contextual hidden states of shape (batch_size, seq_len, N_EMBD)
    """
    B, T = input_ids.shape

    # 1. Token Embeddings
    x = v1_model.token_embedding(input_ids)  # Shape: (B, T, N_EMBD)

    # 2. Sequential Transformer Blocks
    for block in v1_model.blocks:
        x, _ = block(x, past_key_value=None, use_cache=False)

    # 3. Final Pre-LM Normalization Layer
    hidden_states = v1_model.final_norm(x)  # Shape: (B, T, N_EMBD)

    return hidden_states


def verify_v1_hidden_extraction():
    model.eval()
    dummy_input = torch.randint(0, tokenizer.vocab_size, (2, 32), device=DEVICE)
    with torch.no_grad():
        h = extract_v1_hidden_states(model, dummy_input)
    assert h.shape == (2, 32, N_EMBD), f"Unexpected shape {h.shape}, expected (2, 32, {N_EMBD})"
    print(f"[OK] extract_v1_hidden_states verified. Output shape: {h.shape}")

verify_v1_hidden_extraction()

## Phase 2 Architecture Scaffold & Academic Compliance Notice


In [ ]:
# ==============================================================================
# Phase 2 Model Scaffold, Base Classes, and Integration Interfaces
# ==============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------------------------
# 1. Base Prediction Head Interface
# ------------------------------------------------------------------------------
class BasePhase2NERHead(nn.Module):
    def forward(self, hidden_states, attention_mask=None):
        raise NotImplementedError(
            "Implement forward() in your custom prediction head subclass."
        )


# ------------------------------------------------------------------------------
# 2. Student Linear Projection Head
# ------------------------------------------------------------------------------
class StudentLinearNERHead(BasePhase2NERHead):
    """
    Token-level BIO classification head.

    Maps contextual backbone hidden states to BIO label logits.
    """
    def __init__(
        self,
        hidden_dim=N_EMBD,
        num_classes=len(BIO_LABELS),
        dropout_prob=0.1,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, hidden_states, attention_mask=None):
        hidden_states = self.dropout(hidden_states)
        logits = self.classifier(hidden_states)

        return logits


# ------------------------------------------------------------------------------
# 3. Integration Container: Backbone + NER Head
# ------------------------------------------------------------------------------
class V1Phase2NERModel(nn.Module):
    """
    Connects the Phase 1 V1 backbone to a Phase 2 NER prediction head.
    """
    def __init__(self, backbone, ner_head, freeze_backbone=False):
        super().__init__()

        self.backbone = backbone
        self.ner_head = ner_head
        self.freeze_backbone = freeze_backbone

        if freeze_backbone:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

    def forward(self, input_ids, attention_mask=None):
        if self.freeze_backbone:
            with torch.no_grad():
                hidden_states = extract_v1_hidden_states(
                    self.backbone,
                    input_ids,
                    attention_mask,
                )
        else:
            hidden_states = extract_v1_hidden_states(
                self.backbone,
                input_ids,
                attention_mask,
            )

        logits = self.ner_head(hidden_states, attention_mask)

        return logits


# ------------------------------------------------------------------------------
# 4. BIO Decoding with Safe Offset Handling
# ------------------------------------------------------------------------------
def decode_token_bio_predictions(
    logits,
    window_offsets,
    raw_text=None,
    id_to_bio=ID_TO_BIO,
    refine_boundaries=True,
):
    """
    Convert token-level BIO logits into character-level spans.

    Args:
        logits: Tensor [batch_size, sequence_length, num_labels].
        window_offsets: Per-window token offsets into the original document.
        raw_text: Original document text, if available.
        id_to_bio: Mapping from class IDs to BIO labels.
        refine_boundaries: Trim surrounding whitespace and punctuation.

    Returns:
        List of (start, end, label) tuples.

    Note:
        This decoder cannot recover arbitrary character boundaries inside
        tokenizer tokens. Boundary trimming is not a substitute for a
        character-boundary prediction model.
    """
    predictions = (
        torch.argmax(logits, dim=-1)
        .detach()
        .cpu()
        .tolist()
    )

    extracted_spans = []

    for batch_idx, sequence_predictions in enumerate(predictions):
        if batch_idx >= len(window_offsets):
            break

        offsets = window_offsets[batch_idx]

        current_label = None
        current_start = None
        current_end = None

        def flush_current():
            nonlocal current_label, current_start, current_end

            if (
                current_label is not None
                and current_start is not None
                and current_end is not None
                and current_start < current_end
            ):
                extracted_spans.append(
                    (current_start, current_end, current_label)
                )

            current_label = None
            current_start = None
            current_end = None

        for token_idx, label_id in enumerate(sequence_predictions):
            if token_idx >= len(offsets):
                flush_current()
                break

            span = offsets[token_idx]

            if (
                span is None
                or not isinstance(span, (tuple, list))
                or len(span) < 2
                or span[0] is None
                or span[1] is None
            ):
                flush_current()
                continue

            try:
                token_start = int(span[0])
                token_end = int(span[1])
                label_id = int(label_id)
            except (TypeError, ValueError):
                flush_current()
                continue

            if token_start < 0 or token_start >= token_end:
                flush_current()
                continue

            bio_tag = id_to_bio.get(label_id, "O")

            if bio_tag.startswith("B-"):
                flush_current()

                current_label = bio_tag[2:]
                current_start = token_start
                current_end = token_end

            elif bio_tag.startswith("I-"):
                entity_type = bio_tag[2:]

                if current_label == entity_type:
                    # Preserve normal whitespace gaps between token offsets.
                    current_end = max(current_end, token_end)
                else:
                    flush_current()

                    # Treat an I-tag without a matching preceding B-tag
                    # as the beginning of a new entity.
                    current_label = entity_type
                    current_start = token_start
                    current_end = token_end

            else:
                flush_current()

        flush_current()

    # ------------------------------------------------------------------
    # Optional boundary cleanup
    # ------------------------------------------------------------------
    if raw_text is None or not refine_boundaries:
        return extracted_spans

    refined_spans = []
    strip_chars = " \t\n\r.,;:()[]{}'\""

    for start, end, label in extracted_spans:
        start = max(0, int(start))
        end = min(len(raw_text), int(end))

        if start >= end:
            continue

        span_text = raw_text[start:end]
        trimmed_text = span_text.strip(strip_chars)

        if not trimmed_text:
            continue

        leading_whitespace = span_text.find(trimmed_text)
        refined_start = start + leading_whitespace
        refined_end = refined_start + len(trimmed_text)

        if refined_start < refined_end:
            refined_spans.append(
                (refined_start, refined_end, label)
            )

    return refined_spans


print("Phase 2 model scaffold and decoder ready.")
print("Classes: BasePhase2NERHead, StudentLinearNERHead, V1Phase2NERModel.")
print("Decoder: decode_token_bio_predictions.")

In [ ]:
# ==============================================================================
# Phase 2 Training Infrastructure: Optimizer, Schedulers, and Checkpointing
# Dependencies:
#   - Cell 6: SEED
#   - Cell 10: LEARNING_RATE, WEIGHT_DECAY
#   - Cell 12: EXPERIMENT_NAME, PHASE2_ROOT_DIR, PHASE2_CHECKPOINT_DIR
#   - Cell 34: TRAIN_DOC_IDS, VAL_DOC_IDS, TEXTS_BY_ID, ANNOTATIONS_BY_ID
#   - Cell 40: prepare_lner_windows, IGNORE_INDEX
# ==============================================================================

import os
import torch
from torch.utils.data import Dataset, DataLoader


class Phase2NERDataset(Dataset):
    """
    PyTorch Dataset for Phase 2 L-NER windows.
    Stores input_ids, labels, document_id, token_start, and character offsets
    for exact span decoding and evaluation.
    """
    def __init__(self, document_ids, text_by_id, entities_by_id, tokenizer_obj=None,
                 max_length=512, window_overlap=256):
        self.samples = []
        active_tok = tokenizer_obj if tokenizer_obj is not None else globals().get("tokenizer")

        for doc_id in document_ids:
            doc_str = str(doc_id)
            text = text_by_id.get(doc_str, "")
            if not text:
                continue
            entities = entities_by_id.get(doc_str, [])
            windows, report = prepare_lner_windows(
                document_id=doc_str,
                text=text,
                entities=entities,
                max_length=max_length,
                window_overlap=window_overlap,
                tokenizer=active_tok,
            )
            for w in windows:
                self.samples.append({
                    "input_ids": torch.tensor(w["input_ids"], dtype=torch.long),
                    "labels": torch.tensor(w["labels"], dtype=torch.long),
                    "offsets": w["offsets"],
                    "document_id": w["document_id"],
                    "token_start": w["token_start"],
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_phase2_batch(batch, pad_token_id=0, ignore_index=IGNORE_INDEX):
    """
    Pads dynamic sequences within batch and constructs binary attention masks.
    """
    max_len = max(item["input_ids"].size(0) for item in batch)
    B = len(batch)

    padded_input_ids = torch.full((B, max_len), pad_token_id, dtype=torch.long)
    padded_labels = torch.full((B, max_len), ignore_index, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.float32)

    document_ids = []
    offsets = []
    token_starts = []

    for i, item in enumerate(batch):
        seq_len = item["input_ids"].size(0)
        padded_input_ids[i, :seq_len] = item["input_ids"]
        padded_labels[i, :seq_len] = item["labels"]
        attention_mask[i, :seq_len] = 1.0

        document_ids.append(item["document_id"])
        offsets.append(item["offsets"])
        token_starts.append(item["token_start"])

    return {
        "input_ids": padded_input_ids,
        "labels": padded_labels,
        "attention_mask": attention_mask,
        "document_ids": document_ids,
        "offsets": offsets,
        "token_starts": token_starts,
    }


def create_phase2_optimizer(model, lr=3e-5, weight_decay=0.01):
    """
    Creates AdamW optimizer with decoupled weight decay for 1D parameters (biases, norms).
    """
    decay_params = []
    no_decay_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:
            decay_params.append(p)
        else:
            no_decay_params.append(p)

    groups = []
    if decay_params:
        groups.append({"params": decay_params, "weight_decay": weight_decay, "lr": lr})
    if no_decay_params:
        groups.append({"params": no_decay_params, "weight_decay": 0.0, "lr": lr})

    return torch.optim.AdamW(groups)


def save_phase2_checkpoint(model, optimizer, epoch, val_f1, save_path):
    """
    Saves isolated Phase 2 training checkpoint without overwriting Phase 1 weights.
    """
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    state = {
        "epoch": epoch,
        "val_f1": val_f1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer else None,
    }
    torch.save(state, save_path)
    print(f"[Checkpoint] Saved Phase 2 checkpoint to {save_path} (Val F1: {val_f1:.4f})")


def restore_phase2_checkpoint(checkpoint_path, model, optimizer=None):
    """
    Restores Phase 2 checkpoint with strict state verification.
    """
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    state = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state["model_state_dict"], strict=True)
    if optimizer and state.get("optimizer_state_dict"):
        optimizer.load_state_dict(state["optimizer_state_dict"])
    print(f"[Checkpoint] Restored Phase 2 checkpoint from {checkpoint_path} (Epoch: {state.get('epoch')}, Val F1: {state.get('val_f1', 0.0):.4f})")
    return state

print("Phase 2 Training Infrastructure ready.")
print("Utilities: Phase2NERDataset, collate_phase2_batch, create_phase2_optimizer, save_phase2_checkpoint, restore_phase2_checkpoint.")

In [ ]:
# ==============================================================================
# Phase 2 Training Engine and Validation Loop
# Dependencies:
#   - DEVICE
#   - strict_span_metrics(gold_by_id, predicted_by_id, ...)
#   - decode_token_bio_predictions
#   - collate_phase2_batch
#   - create_phase2_optimizer
#   - save_phase2_checkpoint
# ==============================================================================

import time
import torch
import torch.nn.functional as F


def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device=DEVICE,
    clip_norm=1.0,
):
    """Run one training epoch."""

    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids,
            attention_mask=attention_mask,
        )

        B, T, num_classes = logits.shape

        loss = criterion(
            logits.reshape(-1, num_classes),
            labels.reshape(-1),
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss encountered: "
                f"{loss.item()} at batch {num_batches}"
            )

        loss.backward()

        if clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=clip_norm,
            )

        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    if num_batches == 0:
        raise RuntimeError(
            "Training DataLoader produced no batches."
        )

    return total_loss / num_batches


def evaluate_phase2_model(
    model,
    val_dataset,
    texts_by_id,
    gold_by_id,
    device=DEVICE,
    batch_size=8,
    labels=LABELS,
):
    """
    Evaluate the model on held-out documents.

    Returns strict-span metrics and validation cross-entropy.
    """

    model.eval()

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_phase2_batch,
    )

    pred_spans_by_doc = {}

    # Accumulate validation loss over valid target tokens.
    val_loss_sum = 0.0
    val_token_count = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            doc_ids = batch["document_ids"]
            offsets = batch["offsets"]
            labels_batch = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask=attention_mask,
            )

            # Validation loss: ignore padding and ignored targets.
            loss_sum = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                labels_batch.reshape(-1),
                ignore_index=IGNORE_INDEX,
                reduction="sum",
            )

            valid_tokens = int(
                (labels_batch != IGNORE_INDEX).sum().item()
            )

            val_loss_sum += float(loss_sum.item())
            val_token_count += valid_tokens

            # Decode predictions back into character-level spans.
            for i in range(len(doc_ids)):
                doc_id = str(doc_ids[i])
                raw_text = texts_by_id.get(doc_id, "")

                single_logits = logits[i:i + 1]
                single_offsets = [offsets[i]]

                spans = decode_token_bio_predictions(
                    logits=single_logits,
                    window_offsets=single_offsets,
                    raw_text=raw_text,
                    id_to_bio=ID_TO_BIO,
                    refine_boundaries=True,
                )

                if doc_id not in pred_spans_by_doc:
                    pred_spans_by_doc[doc_id] = []

                pred_spans_by_doc[doc_id].extend(spans)

    if val_token_count == 0:
        raise RuntimeError(
            "Validation loss could not be calculated: "
            "no valid target tokens were found."
        )

    # Deduplicate identical predictions from overlapping windows.
    clean_preds_by_doc = {}

    for doc_id, spans in pred_spans_by_doc.items():
        seen = set()
        deduped = []

        for span in spans:
            key = (
                int(span[0]),
                int(span[1]),
                str(span[2]),
            )

            if key not in seen:
                seen.add(key)
                deduped.append(list(key))

        clean_preds_by_doc[doc_id] = deduped

    # Evaluate only documents represented in the prediction set.
    eval_gold_by_id = {
        str(doc_id): gold_by_id.get(str(doc_id), [])
        for doc_id in clean_preds_by_doc
    }

    raw_metrics = strict_span_metrics(
        gold_by_id=eval_gold_by_id,
        predicted_by_id=clean_preds_by_doc,
        labels=labels,
    )

    strict_f1 = raw_metrics.get("strict_macro_f1", 0.0)
    per_label = raw_metrics.get("per_label", {})

    precisions = [
        value.get("precision", 0.0)
        for value in per_label.values()
    ]

    recalls = [
        value.get("recall", 0.0)
        for value in per_label.values()
    ]

    macro_p = (
        sum(precisions) / len(precisions)
        if precisions else 0.0
    )

    macro_r = (
        sum(recalls) / len(recalls)
        if recalls else 0.0
    )

    return {
        "strict_macro_f1": strict_f1,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "val_loss": val_loss_sum / val_token_count,
        "per_label": per_label,
        "documents_evaluated": raw_metrics.get(
            "documents_evaluated",
            len(clean_preds_by_doc),
        ),
        "predictions_by_doc": clean_preds_by_doc,
    }


def run_phase2_training(
    model,
    train_dataset,
    val_dataset,
    texts_by_id,
    gold_by_id,
    epochs=3,
    lr=3e-5,
    batch_size=8,
    checkpoint_path=None,
    device=DEVICE,
    class_weights=None,
):
    """Train, validate, log metrics, and save the best checkpoint.

    Pass class_weights=None for ordinary cross-entropy or a 1D tensor
    aligned with BIO_LABELS for weighted cross-entropy.
    """

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_phase2_batch,
    )

    optimizer = create_phase2_optimizer(
        model,
        lr=lr,
    )

    criterion = torch.nn.CrossEntropyLoss(
        weight=(
            class_weights.to(device=device, dtype=torch.float32)
            if class_weights is not None else None
        ),
        ignore_index=IGNORE_INDEX,
    )

    best_f1 = -1.0
    history = []

    print(
        f"Starting Phase 2 Training "
        f"({epochs} epochs, lr={lr}, device={device})"
    )

    for ep in range(1, epochs + 1):
        t0 = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device=device,
        )

        val_res = evaluate_phase2_model(
            model,
            val_dataset,
            texts_by_id,
            gold_by_id,
            device=device,
        )

        elapsed = time.time() - t0

        val_loss = val_res["val_loss"]
        val_f1 = val_res["strict_macro_f1"]
        val_p = val_res["macro_precision"]
        val_r = val_res["macro_recall"]

        history.append({
            "epoch": ep,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_f1": val_f1,
            "val_precision": val_p,
            "val_recall": val_r,
            "time_sec": elapsed,
        })

        print(
            f"Epoch {ep:02d}/{epochs:02d} "
            f"[{elapsed:.1f}s] - "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Strict Macro-F1: {val_f1:.4f} | "
            f"Precision: {val_p:.4f} | "
            f"Recall: {val_r:.4f}"
        )
        print("  Per-label validation F1:")
        for label_name, label_metrics in val_res.get("per_label", {}).items():
            print(
                f"    {label_name:12s} "
                f"F1={label_metrics.get('f1', 0.0):.3f} "
                f"P={label_metrics.get('precision', 0.0):.3f} "
                f"R={label_metrics.get('recall', 0.0):.3f} "
                f"(TP={label_metrics.get('tp', 0)}, "
                f"FP={label_metrics.get('fp', 0)}, "
                f"FN={label_metrics.get('fn', 0)})"
            )

        if checkpoint_path and val_f1 > best_f1:
            best_f1 = val_f1

            save_phase2_checkpoint(
                model,
                optimizer,
                ep,
                val_f1,
                checkpoint_path,
            )

    return history, best_f1


print("Phase 2 Training Engine and Validation Loop ready.")
print(
    "Functions: train_one_epoch, "
    "evaluate_phase2_model, run_phase2_training."
)

In [ ]:
# ==============================================================================
# Strict Span Evaluation Verification Suite & Comprehensive Metrics
# Dependencies:
#   - Cell 34: LABELS
#   - Cell 46: strict_span_metrics(gold_by_id, predicted_by_id, ...)
# ==============================================================================


def verify_strict_evaluator_properties():
    """
    Verifies properties of strict span evaluation:
    1. Exact match returns strict_macro_f1 = 1.0 on evaluated classes.
    2. Off-by-one character shifts produce strict_macro_f1 = 0.0 under exact matching.
    3. Label mismatches produce strict_macro_f1 = 0.0.
    """
    test_labels = ["COURT", "JUDGE"]

    gold = {
        "doc_0": [{"start": 0, "end": 10, "label": "COURT"}, {"start": 20, "end": 30, "label": "JUDGE"}]
    }

    # Case 1: Exact Match
    exact_pred = {
        "doc_0": [(0, 10, "COURT"), (20, 30, "JUDGE")]
    }
    m_exact = strict_span_metrics(gold, exact_pred, labels=test_labels)
    assert m_exact["strict_macro_f1"] == 1.0, f"Expected 1.0, got {m_exact['strict_macro_f1']}"
    assert m_exact["per_label"]["COURT"]["f1"] == 1.0
    assert m_exact["per_label"]["JUDGE"]["f1"] == 1.0

    # Case 2: Off-by-one character shift (strict match failure)
    shifted_pred = {
        "doc_0": [(0, 11, "COURT"), (20, 29, "JUDGE")]
    }
    m_shift = strict_span_metrics(gold, shifted_pred, labels=test_labels)
    assert m_shift["strict_macro_f1"] == 0.0, f"Expected 0.0 for shifted spans, got {m_shift['strict_macro_f1']}"
    assert m_shift["per_label"]["COURT"]["tp"] == 0

    # Case 3: Label mismatch
    wrong_label_pred = {
        "doc_0": [(0, 10, "JUDGE"), (20, 30, "COURT")]
    }
    m_label = strict_span_metrics(gold, wrong_label_pred, labels=test_labels)
    assert m_label["strict_macro_f1"] == 0.0, f"Expected 0.0 for label mismatches, got {m_label['strict_macro_f1']}"

    print("[OK] Strict Span Evaluator Properties successfully verified on synthetic test cases.")
    return True


def compute_comprehensive_strict_metrics(gold_by_id, predicted_by_id, label_list=None):
    """
    Computes comprehensive per-class and aggregated strict span metrics:
    - strict_macro_f1 (official capstone metric across all classes)
    - strict_macro_precision
    - strict_macro_recall
    - active_macro_f1 (macro average restricted to classes present in gold or pred)
    - aggregate true positives, false positives, false negatives
    """
    target_labels = label_list if label_list is not None else LABELS
    base_metrics = strict_span_metrics(gold_by_id, predicted_by_id, labels=target_labels)

    per_label = base_metrics.get("per_label", {})
    active_f1s = []
    precisions = []
    recalls = []
    total_tp = 0
    total_fp = 0
    total_fn = 0

    for lbl, scores in per_label.items():
        tp = scores.get("tp", 0)
        fp = scores.get("fp", 0)
        fn = scores.get("fn", 0)
        p = scores.get("precision", 0.0)
        r = scores.get("recall", 0.0)
        total_tp += tp
        total_fp += fp
        total_fn += fn
        precisions.append(p)
        recalls.append(r)

        if (tp + fp + fn) > 0:
            active_f1s.append(scores.get("f1", 0.0))

    strict_macro_f1 = base_metrics.get("strict_macro_f1", 0.0)
    macro_p = (sum(precisions) / len(precisions)) if precisions else 0.0
    macro_r = (sum(recalls) / len(recalls)) if recalls else 0.0
    active_macro_f1 = (sum(active_f1s) / len(active_f1s)) if active_f1s else 0.0

    return {
        "strict_macro_f1": strict_macro_f1,
        "strict_macro_precision": macro_p,
        "strict_macro_recall": macro_r,
        "active_macro_f1": active_macro_f1,
        "total_true_positives": total_tp,
        "total_false_positives": total_fp,
        "total_false_negatives": total_fn,
        "per_label": per_label,
    }

verify_strict_evaluator_properties()

In [ ]:
# ==============================================================================
# End-to-End Test Inference & CodaBench Submission Packaging
# Dependencies:
#   - Cell 12: PHASE2_OUTPUT_DIR, PHASE2_PREDICTION_PATH, PHASE2_SUBMISSION_ZIP
#   - Cell 15: tokenizer
#   - Cell 34: TEST_TEXTS, LABELS
#   - Cell 40: prepare_lner_windows
#   - Cell 46: write_lner_submission
#   - Cell 89: decode_token_bio_predictions
# ==============================================================================

import os
import json
import zipfile
import torch


def run_phase2_test_inference(model, test_texts_by_id, device=DEVICE, max_length=512,
                              window_overlap=256, batch_size=8, tokenizer_obj=None):
    """
    Executes inference over test documents (Fold 3):
    - Preserves EVERY required test document ID in test_texts_by_id (even if 0 spans found -> [])
    - Uses exact full-document character offsets
    - Recombines sliding windows and deduplicates span predictions
    - Emits standard JSON structure: {"doc_id": [[start, end, label], ...]}
    """
    model.eval()
    active_tok = tokenizer_obj if tokenizer_obj is not None else globals().get("tokenizer")
    all_predictions = {}
    test_doc_ids = sorted(map(str, test_texts_by_id.keys()))

    print(f"Running inference across {len(test_doc_ids)} test documents...")

    for doc_id in test_doc_ids:
        raw_text = test_texts_by_id[doc_id]
        if not raw_text.strip():
            all_predictions[doc_id] = []
            continue

        windows, _ = prepare_lner_windows(
            document_id=doc_id,
            text=raw_text,
            entities=[],
            max_length=max_length,
            window_overlap=window_overlap,
            tokenizer=active_tok,
        )

        if not windows:
            all_predictions[doc_id] = []
            continue

        doc_spans = []
        for i in range(0, len(windows), batch_size):
            chunk = windows[i:i + batch_size]
            B = len(chunk)
            max_chunk_len = max(len(w["input_ids"]) for w in chunk)

            input_ids = torch.zeros((B, max_chunk_len), dtype=torch.long)
            attn_mask = torch.zeros((B, max_chunk_len), dtype=torch.float32)

            for b_idx, w in enumerate(chunk):
                l = len(w["input_ids"])
                input_ids[b_idx, :l] = torch.tensor(w["input_ids"], dtype=torch.long)
                attn_mask[b_idx, :l] = 1.0

            input_ids = input_ids.to(device)
            attn_mask = attn_mask.to(device)

            with torch.no_grad():
                logits = model(input_ids, attention_mask=attn_mask)

            for b_idx, w in enumerate(chunk):
                single_logits = logits[b_idx:b_idx + 1]
                single_offsets = [w["offsets"]]
                spans = decode_token_bio_predictions(
                    logits=single_logits,
                    window_offsets=single_offsets,
                    raw_text=raw_text,
                    id_to_bio=ID_TO_BIO,
                    refine_boundaries=True
                )
                doc_spans.extend(spans)

        # Deduplicate predictions across windows
        seen = set()
        clean_spans = []
        for s in doc_spans:
            key = (int(s[0]), int(s[1]), str(s[2]))
            if key not in seen:
                seen.add(key)
                clean_spans.append([int(s[0]), int(s[1]), str(s[2])])

        clean_spans.sort(key=lambda x: (x[0], x[1]))
        all_predictions[doc_id] = clean_spans

    return all_predictions


def package_codabench_submission(predictions, output_dir=PHASE2_OUTPUT_DIR, zip_filename="submission.zip"):
    """
    Packages predictions strictly per CodaBench format:
    submission.zip
    └── prediction.json (directly at root of zip, no enclosing folders)
    """
    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, "prediction.json")
    zip_path = os.path.join(output_dir, zip_filename)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="prediction.json")

    print(f"[Submission] Saved prediction.json to {json_path}")
    print(f"[Submission] Created CodaBench package at {zip_path}")
    return json_path, zip_path


def validate_submission_zip(zip_path, expected_doc_ids=None):
    """
    Validates submission ZIP integrity against competition requirements:
    - prediction.json exists directly at root of the archive
    - All expected document IDs are present
    - Span format is [start, end, label] with valid integers and labels
    """
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"Submission zip does not exist: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        namelist = zf.namelist()
        if "prediction.json" not in namelist:
            raise ValueError(f"prediction.json missing from root of {zip_path}. Found: {namelist}")

        content = zf.read("prediction.json").decode("utf-8")
        preds = json.loads(content)

    if not isinstance(preds, dict):
        raise TypeError("prediction.json must contain a top-level JSON dictionary of document IDs.")

    if expected_doc_ids is not None:
        expected_set = {str(d) for d in expected_doc_ids}
        found_set = set(preds.keys())
        missing = expected_set - found_set
        extra = found_set - expected_set
        if missing:
            raise ValueError(f"Submission missing {len(missing)} required document IDs. Examples: {list(missing)[:5]}")
        if extra:
            print(f"[Warning] Submission contains {len(extra)} unexpected document IDs.")

    total_spans = 0
    valid_labels = set(LABELS)

    for doc_id, span_list in preds.items():
        if not isinstance(span_list, list):
            raise TypeError(f"Document {doc_id} predictions must be a list of spans.")
        for item in span_list:
            total_spans += 1
            if len(item) < 3:
                raise ValueError(f"Span {item} in doc {doc_id} must have at least 3 elements [start, end, label].")
            start, end, label = item[0], item[1], item[2]
            if not isinstance(start, int) or not isinstance(end, int):
                raise TypeError(f"Offsets must be integers, got start={start}, end={end} in doc {doc_id}")
            if start >= end:
                raise ValueError(f"Inverted span found: start={start} >= end={end} in doc {doc_id}")
            if label not in valid_labels:
                raise ValueError(f"Unknown entity label '{label}' in doc {doc_id}")
            if doc_id in TEST_TEXTS and not (0 <= start < end <= len(TEST_TEXTS[doc_id])):
                raise ValueError(
                    f"Span {(start, end, label)} is outside document {doc_id} "
                    f"(text length={len(TEST_TEXTS[doc_id])})."
                )

    print(f"[Verification] Submission ZIP validated successfully.")
    print(f"               Documents: {len(preds)}, Total Spans: {total_spans}")
    return True

print("Test Inference and Submission Packaging utilities ready.")
print("Functions: run_phase2_test_inference, package_codabench_submission, validate_submission_zip.")

In [ ]:
# ==============================================================================
# EXP-1 Pre-Flight Verification & Comprehensive Smoke Tests
# Dependencies:
#   - Cells 86 to 93
# ==============================================================================

import os
import shutil
import tempfile
import torch
import torch.nn as nn
import copy


def run_exp1_preflight_smoke_tests():
    print("=" * 70)
    print("RUNNING EXP-1 PRE-FLIGHT VERIFICATION & SMOKE TESTS")
    print("=" * 70)

    # Snapshot the real restored Phase 1 backbone so the smoke tests
    # can explicitly verify it remains untouched.
    original_backbone_state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }

    # 1. Forward and Backward Smoke Test on an independent copy.
    # Never optimize the global Phase 1 model during a preflight test.
    print("[Check 1/6] Testing V1Phase2NERModel forward and backward pass...")
    test_backbone = copy.deepcopy(model)
    test_head = StudentLinearNERHead(hidden_dim=N_EMBD, num_classes=len(BIO_LABELS))
    test_model = V1Phase2NERModel(test_backbone, test_head, freeze_backbone=False).to(DEVICE)
    test_model.train()

    dummy_input = torch.randint(0, tokenizer.vocab_size, (2, 16), device=DEVICE)
    dummy_labels = torch.full((2, 16), IGNORE_INDEX, dtype=torch.long, device=DEVICE)
    dummy_labels[0, :8] = O_LABEL_ID
    dummy_labels[0, 2] = BIO_TO_ID["B-COURT"]
    dummy_labels[0, 3] = BIO_TO_ID["I-COURT"]

    opt = torch.optim.AdamW(test_model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    opt.zero_grad()
    logits = test_model(dummy_input)
    loss = crit(logits.view(-1, len(BIO_LABELS)), dummy_labels.view(-1))
    assert torch.isfinite(loss).item(), f"Loss is non-finite: {loss.item()}"
    loss.backward()
    opt.step()
    print(f"  --> Forward/backward pass passed. Finite loss: {loss.item():.4f}")

    # 2. BIO Decoder Safe None-Offset Handling
    print("[Check 2/6] Testing BIO decoder with None offsets...")
    dummy_offsets = [
        [None, (0, 3), (4, 11), None, (12, 17)] + [None] * 11,
        [(0, 5), None, (6, 12)] + [None] * 13
    ]
    spans = decode_token_bio_predictions(logits, dummy_offsets, raw_text="The Supreme Court of India")
    print(f"  --> Decoded spans safely without exception: {len(spans)} spans extracted.")

    # 3. Strict Evaluator Verification
    print("[Check 3/6] Testing strict span evaluator...")
    verify_strict_evaluator_properties()
    print("  --> Strict evaluator verified.")

    # 4. Checkpoint Non-Destructive Isolation Test
    print("[Check 4/6] Testing non-destructive Phase 2 checkpointing...")
    temp_dir = tempfile.mkdtemp(prefix="phase2_smoke_")
    try:
        temp_ckpt = os.path.join(temp_dir, "test_ckpt.pt")
        save_phase2_checkpoint(test_model, opt, 1, 0.85, temp_ckpt)
        assert os.path.isfile(temp_ckpt), "Checkpoint file was not created!"

        # Verify restoration
        fresh_backbone = copy.deepcopy(model)
        fresh_head = StudentLinearNERHead(hidden_dim=N_EMBD, num_classes=len(BIO_LABELS))
        fresh_model = V1Phase2NERModel(fresh_backbone, fresh_head, freeze_backbone=False).to(DEVICE)
        restore_phase2_checkpoint(temp_ckpt, fresh_model)
        print("  --> Phase 2 checkpoint save/restore verified.")
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

    # 5. Submission Packaging and Zip Structure
    print("[Check 5/6] Testing CodaBench ZIP packaging and validation...")
    temp_sub_dir = tempfile.mkdtemp(prefix="phase2_sub_smoke_")
    try:
        sample_doc_id = list(TEST_TEXTS.keys())[0] if TEST_TEXTS else "sample_test_doc_0"
        sample_preds = {
            sample_doc_id: [[0, 10, "COURT"]]
        }
        json_path, zip_path = package_codabench_submission(sample_preds, output_dir=temp_sub_dir)
        validate_submission_zip(zip_path, expected_doc_ids=[sample_doc_id])
        print("  --> Packaging and ZIP integrity verified.")
    finally:
        shutil.rmtree(temp_sub_dir, ignore_errors=True)

    # 6. Verify that the actual global Phase 1 weights did not change.
    print("[Check 6/6] Verifying Phase 1 model weights are preserved...")
    assert hasattr(model, "token_embedding"), "V1 backbone missing token_embedding"
    assert len(model.blocks) == N_LAYER, f"V1 backbone blocks count mismatch: {len(model.blocks)} != {N_LAYER}"
    for name, tensor in model.state_dict().items():
        if not torch.equal(tensor.detach().cpu(), original_backbone_state[name]):
            raise AssertionError(
                f"Preflight unexpectedly changed the original Phase 1 parameter: {name}"
            )
    print(f"  --> Original Phase 1 V1 weights unchanged ({N_LAYER} layers, N_EMBD={N_EMBD}).")

    print("=" * 70)
    print("ALL PRE-FLIGHT SMOKE TESTS PASSED CLEANLY! NOTEBOOK IS READY FOR EXP-1.")
    print("=" * 70)

run_exp1_preflight_smoke_tests()

## Kaggle execution protocol: V1 character-boundary rescue

Follow this exact sequence to run Experiment 1 and generate your verified CodaBench submission on Kaggle:

### Execution Protocol:
1. **Initialize Baseline Pipeline (Cells 0 through 84)**:
   - Run setup cells, configure paths, and load the pretrained Phase 1 V1 model (`v1_model.pt`).
   - Load and normalize Fold 1, Fold 2, and Fold 3 datasets.

2. **Verify Architecture & Student Implementation (Cells 88–89)**:
   - Inspect `StudentLinearNERHead` in Cell 89.
   - Note the `# TODO [STUDENT IMPLEMENTATION]` marker. You may train with the provided baseline linear projection head or replace it with your custom architecture from course lectures.

3. **Construct Window Datasets (Cell 90)**:
   - Instantiate `Phase2NERDataset` for training (`TRAIN_DOC_IDS`) and validation (`VAL_DOC_IDS`):
     ```python
     train_dataset = Phase2NERDataset(
         TRAIN_DOC_IDS, TEXTS_BY_ID, ANNOTATIONS_BY_ID,
         tokenizer_obj=tokenizer, max_length=MODEL_CONTEXT_LENGTH, window_overlap=MODEL_CONTEXT_LENGTH // 2
     )
     val_dataset = Phase2NERDataset(
         VAL_DOC_IDS, TEXTS_BY_ID, ANNOTATIONS_BY_ID,
         tokenizer_obj=tokenizer, max_length=MODEL_CONTEXT_LENGTH, window_overlap=MODEL_CONTEXT_LENGTH // 2
     )
     ```

4. **Train Phase 2 Model (Cell 91)**:
   - Instantiate the full model and train:
     ```python
     ner_head = StudentLinearNERHead(hidden_dim=N_EMBD, num_classes=len(BIO_LABELS))
     phase2_model = V1Phase2NERModel(model, ner_head, freeze_backbone=False).to(DEVICE)
     best_checkpoint_path = os.path.join(PHASE2_CHECKPOINT_DIR, "best_v1_full_ft.pt")

     history, best_f1 = run_phase2_training(
         phase2_model, train_dataset, val_dataset, TEXTS_BY_ID, ANNOTATIONS_BY_ID,
         epochs=3, lr=3e-5, batch_size=8, checkpoint_path=best_checkpoint_path, device=DEVICE
     )
     ```

5. **Evaluate Strict Metrics (Cell 92)**:
   - Restore the best checkpoint and evaluate comprehensive strict span metrics:
     ```python
     restore_phase2_checkpoint(best_checkpoint_path, phase2_model)
     val_metrics = evaluate_phase2_model(phase2_model, val_dataset, TEXTS_BY_ID, ANNOTATIONS_BY_ID, device=DEVICE)
     print("Validation Strict Macro F1:", val_metrics["strict_macro_f1"])
     ```

6. **Generate Test Predictions & CodaBench Submission (Cell 93)**:
   - Run inference across all Fold-3 test documents and create `submission.zip`:
     ```python
     test_predictions = run_phase2_test_inference(
         phase2_model, TEST_TEXTS, device=DEVICE,
         max_length=MODEL_CONTEXT_LENGTH, window_overlap=MODEL_CONTEXT_LENGTH // 2
     )
     json_path, zip_path = package_codabench_submission(test_predictions, output_dir=PHASE2_OUTPUT_DIR)
     validate_submission_zip(zip_path, expected_doc_ids=list(TEST_TEXTS.keys()))
     ```

7. **Download & Submit**:
   - Download `/kaggle/working/cs781_phase2_lner/V1_FullFT/submission.zip`.
   - Submit directly to CodaBench.

### Added validation diagnostics and weighted-loss experiment

The training engine now accepts optional `class_weights` and prints per-label strict precision, recall, F1, TP, FP, and FN after each epoch. The EXP-1 training cell uses conservative smoothed weights with a floor for `O` and a cap for rare classes. This is a hypothesis to test, not a guaranteed improvement. Compare its validation Strict Macro-F1 and precision/recall against the saved unweighted baseline if one exists. Do not compare results from different validation splits.


**Rescue run default:** the older token-BIO training/inference is disabled by `RUN_LEGACY_TOKEN_BIO_BASELINE = False` to avoid spending several extra epochs on a known weaker experiment. The exact-character V1 training remains active. CUDA FP16 autocast and gradient accumulation are enabled to reduce T4 memory pressure.


In [ ]:
RUN_LEGACY_TOKEN_BIO_BASELINE = False  # Set True only if you explicitly want the older token-BIO baseline too.

if RUN_LEGACY_TOKEN_BIO_BASELINE:
    # EXP-1: Prepare training and validation datasets

    train_dataset = Phase2NERDataset(
        TRAIN_DOC_IDS,
        TEXTS_BY_ID,
        ANNOTATIONS_BY_ID,
        tokenizer_obj=tokenizer,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap=MODEL_CONTEXT_LENGTH // 2,
    )

    val_dataset = Phase2NERDataset(
        VAL_DOC_IDS,
        TEXTS_BY_ID,
        ANNOTATIONS_BY_ID,
        tokenizer_obj=tokenizer,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap=MODEL_CONTEXT_LENGTH // 2,
    )

    print("Training documents:", len(TRAIN_DOC_IDS))
    print("Validation documents:", len(VAL_DOC_IDS))
    print("Training windows:", len(train_dataset))
    print("Validation windows:", len(val_dataset))

    assert len(train_dataset) > 0, "Training dataset is empty."
    assert len(val_dataset) > 0, "Validation dataset is empty."

    print("Dataset preparation completed.")
else:
    train_dataset = None
    val_dataset = None
    print("Skipping legacy token-BIO dataset materialization; the rescue experiment uses exact character offsets.")


In [ ]:
# Optional legacy EXP-1 baseline; default is OFF to prioritize the 0.323 rescue experiment.
if RUN_LEGACY_TOKEN_BIO_BASELINE:
    # EXP-1B: V1 Full Fine-Tuning with conservative smoothed class weighting.
    # This is an experimental configuration, not a guaranteed improvement.
    # Class counts are computed from the actual current training targets only.

    from collections import Counter
    import torch

    _exp1_counts = torch.zeros(len(BIO_LABELS), dtype=torch.float64)
    for _sample in train_dataset.samples:
        _targets = _sample["labels"].long()
        _valid = _targets != IGNORE_INDEX
        if _valid.any():
            _exp1_counts += torch.bincount(
                _targets[_valid], minlength=len(BIO_LABELS)
            ).double()

    _exp1_counts = _exp1_counts.clamp_min(1)
    _exp1_weights = (
        _exp1_counts.sum() / (len(_exp1_counts) * _exp1_counts)
    ).sqrt()
    _exp1_weights = _exp1_weights / _exp1_weights.mean()

    # Cap extreme weights and apply a floor to O so it remains learnable.
    _exp1_weights = _exp1_weights.clamp(min=0.10, max=3.0)
    _exp1_weights = _exp1_weights / _exp1_weights.mean()

    print("Using smoothed class-weighted loss:")
    for _label, _weight in zip(BIO_LABELS, _exp1_weights.tolist()):
        print(f"  {_label:12s} {_weight:.3f}")

    ner_head = StudentLinearNERHead(
        hidden_dim=N_EMBD,
        num_classes=len(BIO_LABELS),
    )

    # Keep the restored Phase-1 model intact. The Phase-2 experiment trains an
    # independent copy, so later character-boundary experiments start from V1 again.
    import copy
    phase2_backbone = copy.deepcopy(model)
    phase2_model = V1Phase2NERModel(
        backbone=phase2_backbone,
        ner_head=ner_head,
        freeze_backbone=False,
    ).to(DEVICE)

    best_checkpoint_path = os.path.join(
        PHASE2_CHECKPOINT_DIR,
        "best_v1_full_ft.pt",
    )

    history, best_f1 = run_phase2_training(
        phase2_model,
        train_dataset,
        val_dataset,
        TEXTS_BY_ID,
        ANNOTATIONS_BY_ID,
        epochs=3,
        lr=3e-5,
        batch_size=8,
        checkpoint_path=best_checkpoint_path,
        device=DEVICE,
        class_weights=_exp1_weights,
    )

    print("\nTraining finished.")
    print("Best validation Strict Macro-F1:", best_f1)
    print("Best checkpoint:", best_checkpoint_path)
else:
    print("Legacy token-BIO training skipped. Proceeding to the exact-character V1 rescue experiment.")


In [ ]:
# # ============================================================
# # EXP-1: FINAL EVALUATION AND SUBMISSION
# # Run only after EXP-1 training completes successfully.
# # ============================================================

# import os
# import json
# import torch

# # 1. Verify that the best checkpoint exists.
# if not os.path.isfile(best_checkpoint_path):
#     raise FileNotFoundError(
#         f"Checkpoint not found: {best_checkpoint_path}. "
#         "Run the EXP-1 training cell first."
#     )

# print("Restoring best EXP-1 checkpoint...")

# # 2. Restore the model that achieved the best validation F1.
# checkpoint_state = restore_phase2_checkpoint(
#     best_checkpoint_path,
#     phase2_model,
# )

# phase2_model.eval()

# print("Checkpoint epoch:", checkpoint_state.get("epoch"))
# print("Checkpoint validation F1:", checkpoint_state.get("val_f1"))

# # 3. Evaluate the restored checkpoint.
# final_val_metrics = evaluate_phase2_model(
#     phase2_model,
#     val_dataset,
#     TEXTS_BY_ID,
#     ANNOTATIONS_BY_ID,
#     device=DEVICE,
#     batch_size=1,
# )

# print("\nFINAL VALIDATION RESULTS")
# print("-" * 45)
# print(f"Validation loss: {final_val_metrics['val_loss']:.4f}")
# print(f"Strict Macro-F1: {final_val_metrics['strict_macro_f1']:.4f}")
# print(f"Macro-Precision: {final_val_metrics['macro_precision']:.4f}")
# print(f"Macro-Recall: {final_val_metrics['macro_recall']:.4f}")

# # 4. Save training history and validation metrics separately.
# experiment_dir = os.path.dirname(
#     os.path.dirname(os.path.abspath(best_checkpoint_path))
# )
# os.makedirs(experiment_dir, exist_ok=True)

# history_path = os.path.join(
#     experiment_dir,
#     "training_history.json",
# )

# with open(history_path, "w", encoding="utf-8") as f:
#     json.dump(history, f, indent=2)

# metrics_to_save = {
#     key: value
#     for key, value in final_val_metrics.items()
#     if key != "predictions_by_doc"
# }

# metrics_path = os.path.join(
#     experiment_dir,
#     "validation_metrics.json",
# )

# with open(metrics_path, "w", encoding="utf-8") as f:
#     json.dump(metrics_to_save, f, indent=2)

# # 5. Generate predictions for the actual Phase 2 test documents.
# print("\nGenerating test predictions...")

# test_predictions = run_phase2_test_inference(
#     phase2_model,
#     TEST_TEXTS,
#     device=DEVICE,
#     max_length=MODEL_CONTEXT_LENGTH,
#     window_overlap=MODEL_CONTEXT_LENGTH // 2,
#     batch_size=1,
#     tokenizer_obj=tokenizer,
# )

# # 6. Package and validate the submission.
# json_path, zip_path = package_codabench_submission(
#     test_predictions,
#     output_dir=experiment_dir,
#     zip_filename="submission.zip",
# )

# validate_submission_zip(
#     zip_path,
#     expected_doc_ids=list(TEST_TEXTS.keys()),
# )

# print("\nEXP-1 FINALIZATION COMPLETE")
# print("Best checkpoint:", best_checkpoint_path)
# print("Training history:", history_path)
# print("Validation metrics:", metrics_path)
# print("Prediction JSON:", json_path)
# print("Submission ZIP:", zip_path)

In [ ]:
# Optional legacy class-weight diagnostic.
if RUN_LEGACY_TOKEN_BIO_BASELINE:
    import torch
    import torch.nn as nn

    # Count the supervised target tokens.
    class_counts = torch.zeros(len(BIO_LABELS), dtype=torch.float64)

    for sample in train_dataset.samples:
        targets = sample["labels"].long()
        valid = targets != IGNORE_INDEX

        if valid.any():
            class_counts += torch.bincount(
                targets[valid],
                minlength=len(BIO_LABELS)
            ).double()

    print("Class-weight diagnostic")
    print("-" * 65)

    for idx, label in enumerate(BIO_LABELS):
        print(f"{label:15s} {int(class_counts[idx]):9d}")
else:
    print("Skipped optional legacy class-weight diagnostic; not required for character-boundary rescue.")


In [ ]:
# Optional legacy weighted loss diagnostic.
if RUN_LEGACY_TOKEN_BIO_BASELINE:
    import torch
    import torch.nn as nn

    counts = class_counts.float().clamp_min(1)

    # Smoothed inverse-frequency weighting
    class_weights = (counts.sum() / (len(counts) * counts)).sqrt()

    # Normalize and limit the maximum weight
    class_weights = class_weights / class_weights.mean()
    class_weights = class_weights.clamp(max=5.0)

    criterion_weighted = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE),
        ignore_index=IGNORE_INDEX
    )

    print("Class weights:")
    for label, weight in zip(BIO_LABELS, class_weights.tolist()):
        print(f"{label:15s} {weight:.3f}")
else:
    print("Skipped optional legacy weighted loss diagnostic; not required for character-boundary rescue.")


In [ ]:
# Optional token-window alignment summary.
if RUN_LEGACY_TOKEN_BIO_BASELINE:
    # EXP-2 PREPARATION: TRAINING-DATA ALIGNMENT AUDIT
    # Diagnostic only. Does not modify the model or datasets.

    from collections import Counter

    def summarize_alignment(dataset, name):
        reports = getattr(dataset, "alignment_reports", None)

        if reports is None:
            print(f"{name}: alignment reports unavailable in this dataset class.")
            print("Use the existing SPAN_ALIGNMENT_AUDIT below instead.")
            return

        totals = Counter()
        for report in reports:
            totals["entities"] += report.get("entity_count", 0)
            totals["aligned"] += report.get("exactly_aligned_entities", 0)
            totals["unaligned"] += report.get("unaligned_entities", 0)
            totals["conflicts"] += report.get("overlapping_entity_conflicts", 0)

        n = totals["entities"]
        print(f"\n{name}")
        print("-" * 45)
        print("Entities:", n)
        print("Exactly aligned:", totals["aligned"])
        print("Unaligned:", totals["unaligned"])
        print("Overlapping conflicts:", totals["conflicts"])
        print("Exact alignment rate:", f"{totals['aligned'] / max(n, 1):.2%}")


    print("Existing sampled alignment audit:")
    print("Entities checked:", SPAN_ALIGNMENT_AUDIT["entities_checked"])
    print(
        "Both-boundary alignment rate:",
        f"{SPAN_ALIGNMENT_AUDIT['both_boundary_alignment_rate']:.2%}"
    )
    print("\nExamples of unaligned gold spans:")
    for item in SPAN_ALIGNMENT_AUDIT["examples_not_aligned"]:
        print(item)
else:
    print("Skipped token-window alignment summary; exact-character alignment audit runs below.")


In [ ]:
# EXP-2: CHARACTER-BOUNDARY REFINEMENT AUDIT
# Measures how often the current tokenizer's offsets can exactly
# represent gold entity spans, and inspects boundary mismatches.

from collections import Counter

boundary_stats = Counter()
examples = []

for document_id in TRAIN_DOC_IDS[:30]:
    text = TEXTS_BY_ID[document_id]
    _, offsets = encode_with_offsets(text, tokenizer)

    valid_offsets = [
        (i, span[0], span[1])
        for i, span in enumerate(offsets)
        if span is not None
    ]

    starts = {start for _, start, _ in valid_offsets}
    ends = {end for _, _, end in valid_offsets}

    for entity in ANNOTATIONS_BY_ID[document_id]:
        start, end, label = (
            entity["start"],
            entity["end"],
            entity["label"],
        )

        boundary_stats["total"] += 1

        if start in starts and end in ends:
            boundary_stats["both_aligned"] += 1
        elif start not in starts and end in ends:
            boundary_stats["start_only_misaligned"] += 1
        elif start in starts and end not in ends:
            boundary_stats["end_only_misaligned"] += 1
        else:
            boundary_stats["both_misaligned"] += 1

        if (
            (start not in starts or end not in ends)
            and len(examples) < 20
        ):
            examples.append({
                "document_id": str(document_id),
                "label": label,
                "gold_text": text[start:end],
                "gold_start": start,
                "gold_end": end,
                "start_aligned": start in starts,
                "end_aligned": end in ends,
            })

print("Boundary audit")
print("=" * 45)
print("Entities:", boundary_stats["total"])
print(
    "Both aligned:",
    boundary_stats["both_aligned"],
    f"({boundary_stats['both_aligned'] / max(boundary_stats['total'], 1):.2%})",
)
print("Start only misaligned:", boundary_stats["start_only_misaligned"])
print("End only misaligned:", boundary_stats["end_only_misaligned"])
print("Both misaligned:", boundary_stats["both_misaligned"])

print("\nMisaligned examples:")
for item in examples:
    print(item)

In [ ]:
# EXP-2: GOLD-SPAN REPRESENTATION AUDIT
# Diagnostic only. Does not modify the model or training data.

from collections import Counter

representation_counts = Counter()
examples = []

for document_id in TRAIN_DOC_IDS[:30]:
    text = TEXTS_BY_ID[str(document_id)]
    entities = ANNOTATIONS_BY_ID[str(document_id)]

    _, offsets = encode_with_offsets(text, tokenizer)
    valid_offsets = [
        (start, end)
        for span in offsets
        if span is not None
        for start, end in [span]
    ]

    token_starts = {start for start, _ in valid_offsets}
    token_ends = {end for _, end in valid_offsets}

    for entity in entities:
        start = entity["start"]
        end = entity["end"]

        start_ok = start in token_starts
        end_ok = end in token_ends

        if start_ok and end_ok:
            representation_counts["both_boundaries_aligned"] += 1
        else:
            representation_counts["not_fully_aligned"] += 1

            if len(examples) < 15:
                examples.append({
                    "document_id": str(document_id),
                    "label": entity["label"],
                    "text": text[start:end],
                    "start_aligned": start_ok,
                    "end_aligned": end_ok,
                    "start": start,
                    "end": end,
                })

print("Representation audit")
print("=" * 45)
print(dict(representation_counts))
print("\nExamples:")
for item in examples:
    print(item)

In [ ]:
# Optional legacy BIO frequency diagnostic.
if RUN_LEGACY_TOKEN_BIO_BASELINE:
    from collections import Counter

    bio_counts = Counter()
    ignored_tokens = 0

    for sample in train_dataset.samples:
        targets = sample["labels"].tolist()

        for target in targets:
            if target == IGNORE_INDEX:
                ignored_tokens += 1
            else:
                bio_counts[ID_TO_BIO[target]] += 1

    total_supervised = sum(bio_counts.values())

    print("Supervised tokens:", total_supervised)
    print("Ignored tokens:", ignored_tokens)
    print(
        f"O-class proportion: "
        f"{100 * bio_counts.get('O', 0) / max(total_supervised, 1):.2f}%"
    )

    print("\nBIO target distribution:")
    for label, count in bio_counts.most_common():
        print(
            f"{label:15s} {count:9d} "
            f"({100 * count / max(total_supervised, 1):6.2f}%)"
        )

    print("\nEntity counts by type:")
    for label in LABELS:
        count = sum(
            bio_counts.get(f"{prefix}-{label}", 0)
            for prefix in ("B", "I")
        )
        print(f"{label:15s} {count:9d}")
else:
    print("Skipped optional legacy BIO frequency diagnostic; not required for character-boundary rescue.")


### Packaging helper fix

The generic ZIP packager and Phase 2 prediction validator above are active Python cells. They must not be commented out: the EXP-1 finalization cell calls `package_prediction_json_zip()` and `validate_phase2_prediction_mapping()`. These definitions have been restored in this version to prevent the previous `NameError` and the subsequent missing-packager error.


In [ ]:
# Generic CodaBench submission packager. This cell must be executable because
# the EXP-1 and EXP-1D finalization cells call package_prediction_json_zip()
# and package_codabench_submission().
import os
import json
import zipfile


def package_prediction_json_zip(
    predictions,
    output_dir,
    expected_doc_ids=None,
    zip_filename="submission.zip",
):
    """Write prediction.json and put it directly at the root of a ZIP archive."""
    if not isinstance(predictions, dict):
        raise TypeError("Predictions must be a dictionary keyed by document ID.")
    if os.path.basename(zip_filename) != zip_filename:
        raise ValueError("zip_filename must be a filename, not a path.")

    normalized_predictions = {}
    for document_id, prediction in predictions.items():
        key = str(document_id)
        if key in normalized_predictions:
            raise ValueError(
                f"Duplicate document ID after string normalization: {key}"
            )
        normalized_predictions[key] = prediction

    if expected_doc_ids is not None:
        expected_ids = {str(doc_id) for doc_id in expected_doc_ids}
        actual_ids = set(normalized_predictions)
        missing_ids = expected_ids - actual_ids
        extra_ids = actual_ids - expected_ids
        if missing_ids or extra_ids:
            raise ValueError(
                f"Document-ID mismatch: {len(missing_ids)} missing, "
                f"{len(extra_ids)} unexpected. "
                f"Missing examples: {list(missing_ids)[:5]}; "
                f"unexpected examples: {list(extra_ids)[:5]}"
            )

    serialized_predictions = json.dumps(
        normalized_predictions,
        ensure_ascii=False,
        allow_nan=False,
        indent=2,
    )
    os.makedirs(output_dir, exist_ok=True)
    prediction_path = os.path.join(output_dir, "prediction.json")
    zip_path = os.path.join(output_dir, zip_filename)

    with open(prediction_path, "w", encoding="utf-8") as f:
        f.write(serialized_predictions)

    with zipfile.ZipFile(
        zip_path, mode="w", compression=zipfile.ZIP_DEFLATED
    ) as archive:
        archive.write(prediction_path, arcname="prediction.json")

    with zipfile.ZipFile(zip_path, "r") as archive:
        if archive.testzip() is not None:
            raise RuntimeError("The submission ZIP failed its integrity test.")
        names = archive.namelist()
        if names != ["prediction.json"]:
            raise ValueError(
                f"Unexpected ZIP contents: {names}. Expected only prediction.json at the archive root."
            )
        loaded_predictions = json.loads(
            archive.read("prediction.json").decode("utf-8")
        )

    if set(loaded_predictions) != set(normalized_predictions):
        raise AssertionError("Document IDs changed during ZIP packaging.")

    print("Submission package created and verified.")
    print("Prediction JSON:", prediction_path)
    print("Submission ZIP:", zip_path)
    print("Documents:", len(loaded_predictions))
    print("Archive contents:", names)
    return prediction_path, zip_path


def package_codabench_submission(
    predictions,
    output_dir,
    zip_filename="submission.zip",
    expected_doc_ids=None,
):
    """Canonical alias for package_prediction_json_zip."""
    return package_prediction_json_zip(
        predictions=predictions,
        output_dir=output_dir,
        expected_doc_ids=expected_doc_ids,
        zip_filename=zip_filename,
    )

In [ ]:
# Phase 2-specific prediction validation and submission archive verification.
# This cell is executable and defines both validate_phase2_prediction_mapping()
# and validate_submission_zip() before any downstream cell calls them.
import os
import json
import zipfile
from numbers import Integral


def validate_phase2_prediction_mapping(
    predictions,
    test_texts_by_id,
    entity_labels,
):
    """Validate exact test-document coverage and character-level entity spans.

    Accepted span formats are [start, end, label] and
    [start, end, label, entity_text].
    """
    if not isinstance(predictions, dict):
        raise TypeError("Phase 2 predictions must be a dictionary.")

    normalized_predictions = {}
    for document_id, spans in predictions.items():
        key = str(document_id)
        if key in normalized_predictions:
            raise ValueError(f"Duplicate document ID after normalization: {key}")
        normalized_predictions[key] = spans

    normalized_texts = {str(document_id): text for document_id, text in test_texts_by_id.items()}
    expected_ids = set(normalized_texts)
    actual_ids = set(normalized_predictions)
    missing_ids = expected_ids - actual_ids
    extra_ids = actual_ids - expected_ids
    if missing_ids or extra_ids:
        raise ValueError(
            f"Test document mismatch: {len(missing_ids)} missing, "
            f"{len(extra_ids)} unexpected. "
            f"Missing examples: {list(missing_ids)[:5]}; "
            f"unexpected examples: {list(extra_ids)[:5]}"
        )

    allowed_labels = set(entity_labels)
    total_spans = 0
    for document_id, spans in normalized_predictions.items():
        text = normalized_texts[document_id]
        if not isinstance(spans, list):
            raise TypeError(f"Predictions for document {document_id} must be a list.")
        seen = set()
        for span in spans:
            if not isinstance(span, (list, tuple)) or len(span) not in (3, 4):
                raise ValueError(
                    f"Invalid entity format in document {document_id}: {span!r}"
                )
            start, end, label = span[:3]
            if (
                not isinstance(start, Integral) or isinstance(start, bool)
                or not isinstance(end, Integral) or isinstance(end, bool)
            ):
                raise TypeError(f"Entity offsets must be integers: {span!r}")
            start, end = int(start), int(end)
            if not (0 <= start < end <= len(text)):
                raise ValueError(
                    f"Out-of-bounds entity in {document_id}: "
                    f"{(start, end, label)!r}, text length={len(text)}"
                )
            if label not in allowed_labels:
                raise ValueError(f"Unknown entity label {label!r} in {document_id}")
            if len(span) == 4 and not isinstance(span[3], str):
                raise TypeError(f"Optional entity_text must be a string: {span!r}")
            key = (start, end, label)
            if key in seen:
                raise ValueError(f"Duplicate entity prediction in {document_id}: {key}")
            seen.add(key)
            total_spans += 1

    print("Phase 2 prediction validation passed.")
    print("Test documents:", len(normalized_predictions))
    print("Predicted entities:", total_spans)
    return True


def validate_submission_zip(zip_path, expected_doc_ids=None):
    """Validate submission ZIP integrity: prediction.json at root and valid structure."""
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"Submission zip does not exist: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as archive:
        namelist = archive.namelist()
        if "prediction.json" not in namelist:
            raise ValueError(f"prediction.json missing from root of {zip_path}. Found: {namelist}")

        content = archive.read("prediction.json").decode("utf-8")
        preds = json.loads(content)

    if not isinstance(preds, dict):
        raise TypeError("prediction.json must contain a top-level dictionary of document IDs.")

    if expected_doc_ids is not None:
        expected_set = {str(d) for d in expected_doc_ids}
        found_set = set(preds.keys())
        missing = expected_set - found_set
        extra = found_set - expected_set
        if missing:
            raise ValueError(f"Submission missing {len(missing)} required document IDs: {list(missing)[:5]}")
        if extra:
            print(f"[Warning] Submission contains {len(extra)} unexpected document IDs.")

    total_spans = 0
    valid_labels = set(LABELS)
    for doc_id, spans in preds.items():
        if not isinstance(spans, list):
            raise TypeError(f"Document {doc_id} predictions must be a list of spans.")
        for item in spans:
            total_spans += 1
            if len(item) < 3:
                raise ValueError(f"Span {item} in doc {doc_id} must have at least 3 elements.")
            start, end, label = item[0], item[1], item[2]
            if not isinstance(start, Integral) or not isinstance(end, Integral):
                raise TypeError(f"Offsets must be integers, got start={start}, end={end} in doc {doc_id}")
            if start >= end:
                raise ValueError(f"Inverted span found: start={start} >= end={end} in doc {doc_id}")
            if label not in valid_labels:
                raise ValueError(f"Unknown entity label {label!r} in doc {doc_id}")

    print(f"Submission ZIP verified: {zip_path}")
    print(f"  Documents: {len(preds)}, Total Spans: {total_spans}")
    return True

In [ ]:
# Optional token-BIO baseline restore/evaluation/submission.
if RUN_LEGACY_TOKEN_BIO_BASELINE:
    # ============================================================
    # EXP-1: RESTORE, EVALUATE, AND PACKAGE SUBMISSION
    # Run after EXP-1 training has saved its best checkpoint.
    # ============================================================

    import os
    import glob
    import json
    import copy
    import torch


    # 1. Locate the best full-fine-tuning checkpoint.
    checkpoint_path = globals().get("best_checkpoint_path")

    if not checkpoint_path or not os.path.isfile(checkpoint_path):

        root_dir = globals().get(
            "PHASE2_ROOT_DIR",
            "/kaggle/working/cs781_phase2_lner",
        )

        candidates = glob.glob(
            os.path.join(
                root_dir,
                "**",
                "best_v1_full_ft.pt",
            ),
            recursive=True,
        )

        candidates = sorted(set(candidates))

        if len(candidates) == 0:
            raise FileNotFoundError(
                "The best V1 Full Fine-Tuning checkpoint was not found. "
                "Complete EXP-1 training first and verify that its checkpoint "
                "is saved under /kaggle/working."
            )

        if len(candidates) > 1:
            raise RuntimeError(
                "Multiple full-fine-tuning checkpoints were found. "
                "Set best_checkpoint_path explicitly before running this cell:\n"
                + "\n".join(candidates)
            )

        checkpoint_path = candidates[0]

    print("Checkpoint:", checkpoint_path)


    # 2. Build an independent inference model.
    # This prevents finalization from changing the original Phase 1 model.

    inference_backbone = copy.deepcopy(model)

    inference_head = StudentLinearNERHead(
        hidden_dim=N_EMBD,
        num_classes=len(BIO_LABELS),
    )

    inference_model = V1Phase2NERModel(
        backbone=inference_backbone,
        ner_head=inference_head,
        freeze_backbone=False,
    ).to(DEVICE)


    # 3. Restore the saved Phase 2 weights.
    checkpoint_state = restore_phase2_checkpoint(
        checkpoint_path,
        inference_model,
    )

    inference_model.eval()

    print("Restored epoch:", checkpoint_state.get("epoch"))
    print("Checkpoint validation F1:", checkpoint_state.get("val_f1"))


    # 4. Rebuild the validation windows consistently.

    final_val_dataset = Phase2NERDataset(
        VAL_DOC_IDS,
        TEXTS_BY_ID,
        ANNOTATIONS_BY_ID,
        tokenizer_obj=tokenizer,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap=MODEL_CONTEXT_LENGTH // 2,
    )

    if len(final_val_dataset) == 0:
        raise RuntimeError("The validation dataset has no windows.")


    # 5. Evaluate the restored best checkpoint.

    final_val_metrics = evaluate_phase2_model(
        inference_model,
        final_val_dataset,
        TEXTS_BY_ID,
        ANNOTATIONS_BY_ID,
        device=DEVICE,
        batch_size=1,
        labels=LABELS,
    )

    print("\nFINAL VALIDATION METRICS")
    print("-" * 45)
    print("Strict Macro-F1:",
          f"{final_val_metrics['strict_macro_f1']:.4f}")
    print("Macro-Precision:",
          f"{final_val_metrics['macro_precision']:.4f}")
    print("Macro-Recall:",
          f"{final_val_metrics['macro_recall']:.4f}")

    # This is optional because validation-loss tracking may not yet be
    # enabled in every version of the evaluation function.
    if "val_loss" in final_val_metrics:
        print("Validation loss:",
              f"{final_val_metrics['val_loss']:.4f}")


    # 6. Generate predictions for the actual Fold-3 test documents.
    # Batch size 1 is conservative for a T4 GPU.

    test_predictions = run_phase2_test_inference(
        inference_model,
        TEST_TEXTS,
        device=DEVICE,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap=MODEL_CONTEXT_LENGTH // 2,
        batch_size=1,
        tokenizer_obj=tokenizer,
    )


    # 7. Run Phase 2-specific validation before packaging.

    validate_phase2_prediction_mapping(
        predictions=test_predictions,
        test_texts_by_id=TEST_TEXTS,
        entity_labels=LABELS,
    )


    # 8. Package the submission.
    # Output lives alongside this experiment's checkpoint.

    experiment_dir = os.path.dirname(
        os.path.dirname(os.path.abspath(checkpoint_path))
    )

    prediction_path, submission_zip_path = package_prediction_json_zip(
        predictions=test_predictions,
        output_dir=experiment_dir,
        expected_doc_ids=TEST_TEXTS.keys(),
        zip_filename="submission.zip",
    )


    # 9. Save a compact experiment record for future comparisons.

    metrics_to_save = {
        key: value
        for key, value in final_val_metrics.items()
        if key != "predictions_by_doc"
    }

    metrics_to_save.update({
        "experiment_id": "EXP-1_V1_FullFT",
        "checkpoint_path": checkpoint_path,
        "checkpoint_epoch": checkpoint_state.get("epoch"),
        "checkpoint_validation_f1": checkpoint_state.get("val_f1"),
        "prediction_json": prediction_path,
        "submission_zip": submission_zip_path,
    })

    metrics_path = os.path.join(
        experiment_dir,
        "validation_metrics.json",
    )

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics_to_save, f, indent=2, ensure_ascii=False)

    manifest = {
        "phase": 2,
        "task": "Legal Named Entity Recognition",
        "experiment_id": "EXP-1_V1_FullFT",
        "checkpoint_path": checkpoint_path,
        "metrics_path": metrics_path,
        "prediction_json": prediction_path,
        "submission_zip": submission_zip_path,
        "validation": "completed",
        "submission_archive_validation": "passed",
    }

    manifest_path = os.path.join(
        experiment_dir,
        "experiment_manifest.json",
    )

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)


    print("\n" + "=" * 55)
    print("EXP-1 FINALIZATION COMPLETE")
    print("Experiment directory:", experiment_dir)
    print("Best checkpoint:", checkpoint_path)
    print("Validation metrics:", metrics_path)
    print("Experiment manifest:", manifest_path)
    print("Prediction JSON:", prediction_path)
    print("Submission ZIP:", submission_zip_path)
    print("=" * 55)
else:
    print("Skipped legacy token-BIO inference; the rescue submission is produced at the end.")


## Validation error analysis (run after Cell 108)\n\nThis compares exact predicted and gold spans on the held-out validation documents and groups errors into exact misses, false positives, same-boundary wrong labels, and near-boundary mismatches. Categories are diagnostic and can overlap conceptually; inspect examples before changing the model.\n

In [ ]:
# Legacy token-BIO error analysis is disabled with the token-BIO baseline.
# The character-boundary rescue prints its own strict Macro-F1 and per-label metrics.
if not RUN_LEGACY_TOKEN_BIO_BASELINE:
    print("Skipping legacy token-BIO error analysis; use the character-boundary per-label report below.")
else:
    print("Set the legacy token-BIO cells to run to enable this optional error analysis.")


## Research-grounded rationale and limits of this experiment

The capstone's primary score requires exact character offsets and the correct entity label; a one-character shift is a false positive plus a false negative under strict matching. This notebook therefore retains the Phase-1 V1 backbone and supervises character-level BIO labels rather than snapping predictions to BPE token boundaries.

Relevant references for the methodological choices and next experiments:

1. Kalamkar et al. (2022), *Named Entity Recognition in Indian court judgments*, Natural Legal Language Processing Workshop. The paper motivates legal-role-specific entity taxonomies and documents class imbalance in Indian court NER. https://aclanthology.org/2022.nllp-1.15/
2. Su et al. (2022), *Global Pointer: Novel Efficient Span-based Approach for Named Entity Recognition*. It motivates directly modelling start/end span decisions and discusses a loss designed for imbalanced labels. This notebook does **not** implement GlobalPointer; it remains a character-BIO experiment. https://arxiv.org/abs/2208.03054
3. Hu et al. (2021), *LoRA: Low-Rank Adaptation of Large Language Models*. The reference for additive low-rank adaptation, distinct from multiplicative LoRMA. https://arxiv.org/abs/2106.09685
4. Bihany, Patel, and Modi (2025), *LoRMA: Low-Rank Multiplicative Adaptation for LLMs*, Findings of ACL 2025. It defines LoRMA as multiplicative rather than additive low-rank adaptation. This is cited for the separate compulsory adaptation experiment; this character-boundary full-fine-tuning run is **not** presented as a LoRMA run. https://aclanthology.org/2025.findings-acl.527/

This is an engineering refinement, not a claim that the score will exceed 0.5. A full Kaggle training run and a new Codabench result are required to establish any gain.

## EXP-1D: Character-boundary model with six-epoch fine-tuning and validation calibration

This experiment retains the exact-character supervision model and adds three testable changes:

1. Six maximum epochs, with the best checkpoint selected by raw validation Strict Macro-F1.
2. A short warm-up plus cosine learning-rate decay, with a lower learning rate for the pretrained V1 backbone than for the new character head.
3. Weighted focal loss, intended to focus more learning on difficult characters rather than letting easy outside (`O`) characters dominate. This is an experiment, not a guaranteed improvement.
4. A small per-entity confidence-bias calibration sweep on the validation set. It aims to reduce false-positive spans and is applied unchanged to the test set. The final calibrated validation score is a tuning score, not an unbiased estimate of generalization.

The preceding three-epoch notebook remains unchanged. This version uses separate output/checkpoint names so the experiment does not overwrite the previous run. Run cells sequentially from the top. Keep the existing uncalibrated submission as a fallback until the official score arrives.


In [ ]:
# Character-level supervision, offset mapping, and dataset construction.
# Gold character offsets are used directly; no tokenizer-boundary approximation is used.

import os
import json
import copy
import bisect
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

CHAR_PAD_ID = 0
CHAR_UNK_ID = 1
MAX_CHAR_POSITION_IN_TOKEN = 32

# Build the character vocabulary from TRAIN documents only. Unseen validation/test
# characters map to CHAR_UNK_ID; validation/test text is not used to create it.
_training_chars = set()
for _doc_id in TRAIN_DOC_IDS:
    _training_chars.update(TEXTS_BY_ID[str(_doc_id)])
CHAR_TO_ID = {ch: i + 2 for i, ch in enumerate(sorted(_training_chars))}
CHAR_VOCAB_SIZE = len(CHAR_TO_ID) + 2


def _normalise_entity(entity):
    if isinstance(entity, dict):
        start = entity.get("start", entity.get("start_char", entity.get("start_offset")))
        end = entity.get("end", entity.get("end_char", entity.get("end_offset")))
        label = entity.get("label", entity.get("entity_label", entity.get("type")))
    elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
        start, end, label = entity[:3]
    else:
        raise TypeError(f"Unsupported entity representation: {entity!r}")
    start, end, label = int(start), int(end), str(label)
    if label not in LABELS:
        raise ValueError(f"Unknown entity label {label!r}")
    return start, end, label


def build_exact_character_targets(text, entities, char_start, char_end):
    """Return character BIO target IDs for [char_start, char_end) in the document.

    Fully contained gold entities receive exact B/I tags at original character
    positions. Entities cut by a window boundary are ignored in that window rather
    than taught as truncated entities. Overlapping/nested annotations are masked
    with IGNORE_INDEX because a single BIO sequence cannot represent nested spans faithfully.
    """
    if not (0 <= char_start <= char_end <= len(text)):
        raise ValueError("Invalid character window bounds.")
    length = char_end - char_start
    targets = [O_LABEL_ID] * length
    if length == 0:
        return targets

    parsed = []
    for entity in entities:
        s, e, label = _normalise_entity(entity)
        if not (0 <= s < e <= len(text)):
            raise ValueError(f"Invalid gold span {(s, e, label)} for text length {len(text)}")
        parsed.append((s, e, label))

    # Identify every annotation participating in an overlap/nested span.
    # Such regions are masked with IGNORE_INDEX rather than arbitrarily corrupted.
    conflict_ids = set()
    ordered = sorted(enumerate(parsed), key=lambda item: (item[1][0], item[1][1]))
    for pos, (idx_a, ent_a) in enumerate(ordered):
        s_a, e_a, _ = ent_a
        for idx_b, ent_b in ordered[pos + 1:]:
            s_b, e_b, _ = ent_b
            if s_b >= e_a:
                break
            if s_a < e_b and s_b < e_a:
                conflict_ids.add(idx_a)
                conflict_ids.add(idx_b)

    ignore_positions = set()
    for entity_idx, (s, e, label) in enumerate(parsed):
        overlap_start = max(s, char_start)
        overlap_end = min(e, char_end)
        if overlap_start >= overlap_end:
            continue

        # Only use complete entity examples for supervision in a window.
        is_partial = s < char_start or e > char_end
        if is_partial or entity_idx in conflict_ids:
            ignore_positions.update(range(overlap_start - char_start, overlap_end - char_start))
            continue

        b_id = BIO_TO_ID[f"B-{label}"]
        i_id = BIO_TO_ID[f"I-{label}"]
        local_start = s - char_start
        local_end = e - char_start
        targets[local_start] = b_id
        for p in range(local_start + 1, local_end):
            targets[p] = i_id

    for p in ignore_positions:
        targets[p] = IGNORE_INDEX
    return targets


def _map_characters_to_window_tokens(text, offsets, char_start, char_end):
    """Map each character to its local token index and relative token position.

    Whitespace and any other offset gaps are mapped to the nearest character-bearing
    token. Character targets remain exact; this mapping only supplies contextual
    token features to the character-level head.
    """
    length = char_end - char_start
    char_to_token = [-1] * length
    char_positions = [0] * length

    for token_idx, span in enumerate(offsets):
        if span is None or len(span) < 2 or span[0] is None or span[1] is None:
            continue
        token_start, token_end = int(span[0]), int(span[1])
        if token_start >= token_end:
            continue
        left = max(token_start, char_start)
        right = min(token_end, char_end)
        for absolute_char in range(left, right):
            local_char = absolute_char - char_start
            if 0 <= local_char < length and char_to_token[local_char] == -1:
                char_to_token[local_char] = token_idx
                char_positions[local_char] = min(absolute_char - token_start,
                                                  MAX_CHAR_POSITION_IN_TOKEN - 1)

    covered = [i for i, token_idx in enumerate(char_to_token) if token_idx >= 0]
    if not covered:
        return None, None

    for local_char in range(length):
        if char_to_token[local_char] >= 0:
            continue
        insertion = bisect.bisect_left(covered, local_char)
        left_char = covered[insertion - 1] if insertion > 0 else None
        right_char = covered[insertion] if insertion < len(covered) else None
        if left_char is None:
            source_char = right_char
        elif right_char is None:
            source_char = left_char
        elif local_char - left_char <= right_char - local_char:
            source_char = left_char
        else:
            source_char = right_char
        char_to_token[local_char] = char_to_token[source_char]
        char_positions[local_char] = char_positions[source_char]

    return char_to_token, char_positions


class CharacterBoundaryWindowDataset(Dataset):
    """Sliding token windows with exact character-level BIO supervision.
    Guarantees 100% character coverage across all documents.
    """
    def __init__(self, document_ids, text_by_id, entities_by_id, tokenizer_obj,
                 max_length=512, window_overlap=256):
        if not (0 <= window_overlap < max_length):
            raise ValueError("window_overlap must satisfy 0 <= overlap < max_length")
        self.samples = []
        self.document_ids = [str(x) for x in document_ids]
        step = max_length - window_overlap

        for doc_id in self.document_ids:
            text = text_by_id.get(doc_id, "")
            if not text:
                continue
            entities = entities_by_id.get(doc_id, [])
            token_ids, token_offsets = encode_with_offsets(text, tokenizer_obj)
            token_start = 0
            while token_start < len(token_ids):
                token_end = min(token_start + max_length, len(token_ids))
                window_ids = token_ids[token_start:token_end]
                window_offsets = token_offsets[token_start:token_end]
                valid_offsets = [
                    (int(span[0]), int(span[1]))
                    for span in window_offsets
                    if span is not None and span[0] is not None and span[1] is not None
                    and int(span[0]) < int(span[1])
                ]
                if valid_offsets:
                    char_start = min(s for s, _ in valid_offsets)
                    char_end = max(e for _, e in valid_offsets)

                    # Invariant: extend window 0 to start at char 0 and final window to len(text)
                    if token_start == 0:
                        char_start = 0
                    if token_end == len(token_ids):
                        char_end = len(text)

                    char_to_token, char_positions = _map_characters_to_window_tokens(
                        text, window_offsets, char_start, char_end
                    )
                    if char_to_token is not None:
                        char_ids = [CHAR_TO_ID.get(ch, CHAR_UNK_ID)
                                    for ch in text[char_start:char_end]]
                        char_labels = build_exact_character_targets(
                            text, entities, char_start, char_end
                        )
                        if not (len(char_ids) == len(char_to_token) == len(char_positions)
                                == len(char_labels)):
                            raise AssertionError("Character feature/target lengths differ")
                        if any(t < 0 or t >= len(window_ids) for t in char_to_token):
                            raise AssertionError("Character-to-token mapping is out of bounds")
                        self.samples.append({
                            "input_ids": torch.tensor(window_ids, dtype=torch.long),
                            "char_ids": torch.tensor(char_ids, dtype=torch.long),
                            "char_to_token": torch.tensor(char_to_token, dtype=torch.long),
                            "char_positions": torch.tensor(char_positions, dtype=torch.long),
                            "char_labels": torch.tensor(char_labels, dtype=torch.long),
                            "document_id": doc_id,
                            "token_start": token_start,
                            "char_start": char_start,
                            "char_end": char_end,
                        })
                if token_end == len(token_ids):
                    break
                token_start += step

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def audit_document_character_coverage(texts_by_id, dataset):
    """Audits that every character in each evaluation document is covered by at least one window."""
    coverage_by_doc = {str(doc_id): np.zeros(len(text), dtype=bool) for doc_id, text in texts_by_id.items()}
    for sample in dataset.samples:
        doc_id = sample["document_id"]
        if doc_id in coverage_by_doc:
            s = int(sample["char_start"])
            e = int(sample["char_end"])
            coverage_by_doc[doc_id][s:e] = True

    total_chars = sum(len(cov) for cov in coverage_by_doc.values())
    uncovered_chars = sum(np.sum(~cov) for cov in coverage_by_doc.values())
    coverage_rate = (total_chars - uncovered_chars) / max(total_chars, 1) * 100
    print(f"Document character coverage audit: {coverage_rate:.2f}% ({total_chars - uncovered_chars}/{total_chars} chars covered)")
    if uncovered_chars > 0:
        print(f"Warning: {uncovered_chars} characters were uncovered across {len(coverage_by_doc)} documents.")
    return coverage_by_doc, uncovered_chars == 0


def collate_character_windows(batch, pad_token_id=0):
    """Pads sequences dynamically within batch and returns aligned tensors."""
    if not batch:
        raise ValueError("Cannot collate an empty character-boundary batch")
    max_tokens = max(item["input_ids"].numel() for item in batch)
    max_chars = max(item["char_ids"].numel() for item in batch)
    batch_size = len(batch)

    input_ids = torch.full((batch_size, max_tokens), int(pad_token_id), dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_tokens), dtype=torch.float32)
    char_ids = torch.full((batch_size, max_chars), CHAR_PAD_ID, dtype=torch.long)
    char_to_token = torch.zeros((batch_size, max_chars), dtype=torch.long)
    char_positions = torch.zeros((batch_size, max_chars), dtype=torch.long)
    char_labels = torch.full((batch_size, max_chars), IGNORE_INDEX, dtype=torch.long)
    char_mask = torch.zeros((batch_size, max_chars), dtype=torch.float32)

    document_ids, char_starts, char_ends, char_lengths = [], [], [], []
    for row, item in enumerate(batch):
        token_len = item["input_ids"].numel()
        char_len = item["char_ids"].numel()
        input_ids[row, :token_len] = item["input_ids"]
        attention_mask[row, :token_len] = 1.0
        char_ids[row, :char_len] = item["char_ids"]
        char_to_token[row, :char_len] = item["char_to_token"]
        char_positions[row, :char_len] = item["char_positions"]
        char_labels[row, :char_len] = item["char_labels"]
        char_mask[row, :char_len] = 1.0
        document_ids.append(item["document_id"])
        char_starts.append(int(item["char_start"]))
        char_ends.append(int(item["char_end"]))
        char_lengths.append(int(char_len))

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "char_ids": char_ids,
        "char_to_token": char_to_token,
        "char_positions": char_positions,
        "char_labels": char_labels,
        "char_mask": char_mask,
        "document_ids": document_ids,
        "char_starts": char_starts,
        "char_ends": char_ends,
        "char_lengths": char_lengths,
    }

print("Character-level supervision, dataset, and collation utilities defined.")
print(f"Character vocabulary size: {CHAR_VOCAB_SIZE} (derived strictly from training split).")

In [ ]:
# Character-level prediction head, balanced focal loss, and strict-span training/evaluation.
import math
import copy
import time
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


class CharacterBoundaryHead(nn.Module):
    """Emit BIO logits for each raw-text character using contextual token features.
    Substantive modelling improvements:
    - Multi-scale dilated convolutions (local sub-word context + full-word dilated context)
    - Channel LayerNorm for training stability
    - Residual connections across convolutional stages
    """
    def __init__(self, hidden_dim, char_vocab_size, num_labels, char_dim=32,
                 token_feature_dim=128, relative_position_dim=16, dropout=0.1):
        super().__init__()
        self.token_projection = nn.Linear(hidden_dim, token_feature_dim)
        self.char_embedding = nn.Embedding(
            char_vocab_size, char_dim, padding_idx=CHAR_PAD_ID
        )
        self.relative_position_embedding = nn.Embedding(
            MAX_CHAR_POSITION_IN_TOKEN, relative_position_dim
        )
        joined_dim = token_feature_dim + char_dim + relative_position_dim
        self.input_projection = nn.Sequential(
            nn.Linear(joined_dim, token_feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.norm0 = nn.LayerNorm(token_feature_dim)

        # Conv1: receptive field 5 (local char context: sub-word/affixes)
        self.conv1 = nn.Conv1d(token_feature_dim, token_feature_dim, kernel_size=5, padding=2)
        self.norm1 = nn.LayerNorm(token_feature_dim)

        # Conv2: dilated conv with dilation 2 (receptive field spans ~13 chars: full word context)
        self.conv2 = nn.Conv1d(token_feature_dim, token_feature_dim, kernel_size=5, padding=4, dilation=2)
        self.norm2 = nn.LayerNorm(token_feature_dim)

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(token_feature_dim, num_labels)

    def forward(self, token_hidden_states, char_ids, char_to_token,
                char_positions, char_mask):
        batch_size, char_length = char_ids.shape
        safe_map = char_to_token.clamp(min=0, max=token_hidden_states.shape[1] - 1)
        gather_index = safe_map.unsqueeze(-1).expand(-1, -1, token_hidden_states.shape[-1])
        char_context = token_hidden_states.gather(dim=1, index=gather_index)

        features = torch.cat([
            self.token_projection(char_context),
            self.char_embedding(char_ids),
            self.relative_position_embedding(
                char_positions.clamp(min=0, max=MAX_CHAR_POSITION_IN_TOKEN - 1)
            ),
        ], dim=-1)
        x = self.input_projection(features)
        mask = char_mask.unsqueeze(-1)
        x = self.norm0(x) * mask

        # Conv1 with residual
        conv1_out = self.conv1(x.transpose(1, 2)).transpose(1, 2)
        h1 = self.norm1(F.gelu(conv1_out) + x) * mask
        h1 = self.dropout(h1)

        # Conv2 with residual
        conv2_out = self.conv2(h1.transpose(1, 2)).transpose(1, 2)
        h2 = self.norm2(F.gelu(conv2_out) + h1) * mask
        h2 = self.dropout(h2)

        return self.classifier(h2)


class V1CharacterBoundaryNERModel(nn.Module):
    """Independent V1 backbone plus an exact-character BIO prediction head."""
    def __init__(self, backbone, char_vocab_size, num_labels, freeze_backbone=False):
        super().__init__()
        self.backbone = backbone
        self.freeze_backbone = freeze_backbone
        self.char_head = CharacterBoundaryHead(
            hidden_dim=N_EMBD,
            char_vocab_size=char_vocab_size,
            num_labels=num_labels,
        )
        if freeze_backbone:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

    def forward(self, input_ids, attention_mask, char_ids, char_to_token,
                char_positions, char_mask):
        if self.freeze_backbone:
            with torch.no_grad():
                token_hidden = extract_v1_hidden_states(
                    self.backbone, input_ids, attention_mask
                )
        else:
            token_hidden = extract_v1_hidden_states(
                self.backbone, input_ids, attention_mask
            )
        return self.char_head(
            token_hidden, char_ids, char_to_token, char_positions, char_mask
        )


def decode_character_bio_logits(char_logits, id_to_bio=ID_TO_BIO, allow_orphan_i=False, min_entity_chars=1):
    """Decode character BIO tags into exact [start, end) spans.
    - Handles BIO transitions cleanly: flushes on label transition, O-tag, or sequence end.
    - Allows 1-character entities by default; validation-based per-label biases handle false-positive control.
    - If allow_orphan_i is False, an I-X tag without an active B-X entity does NOT trigger a rogue entity.
    - Exact gold character spans are never stripped or modified.
    """
    if isinstance(char_logits, np.ndarray):
        tag_ids = np.argmax(char_logits, axis=-1).tolist()
    else:
        tag_ids = torch.as_tensor(char_logits).argmax(dim=-1).detach().cpu().tolist()
    spans = []
    current_label, current_start = None, None

    def flush(end_position):
        nonlocal current_label, current_start
        if current_label is not None and current_start is not None:
            if (end_position - current_start) >= min_entity_chars:
                spans.append((int(current_start), int(end_position), str(current_label)))
        current_label, current_start = None, None

    for position, tag_id in enumerate(tag_ids):
        tag = id_to_bio.get(int(tag_id), "O")
        if tag.startswith("B-"):
            flush(position)
            current_label = tag[2:]
            current_start = position
        elif tag.startswith("I-"):
            label = tag[2:]
            if current_label == label:
                # Continuation of active entity
                pass
            elif current_label is not None:
                # Transition to different entity label
                flush(position)
                if allow_orphan_i:
                    current_label = label
                    current_start = position
                else:
                    current_label = None
                    current_start = None
            else:
                # Orphan I-tag without preceding B-tag
                if allow_orphan_i:
                    current_label = label
                    current_start = position
                else:
                    current_label = None
                    current_start = None
        else:
            flush(position)

    flush(len(tag_ids))
    return spans


def _make_char_class_weights(dataset, device):
    """Compute smoothed class weights from unique supervised document characters.

    Sliding-window training intentionally repeats context, but class-frequency estimates
    should not inflate counts merely because a character appears in overlapping windows.
    If a character is ignored in a partial window but supervised in another, the supervised
    label is retained. Conflicting non-ignored labels for the same absolute character fail fast.
    """
    target_by_character = {}
    for sample in dataset.samples:
        doc_id = str(sample["document_id"])
        char_start = int(sample["char_start"])
        targets = sample["char_labels"].tolist()
        for relative_position, target_id in enumerate(targets):
            target_id = int(target_id)
            if target_id == IGNORE_INDEX:
                continue
            key = (doc_id, char_start + relative_position)
            previous = target_by_character.get(key)
            if previous is None:
                target_by_character[key] = target_id
            elif previous != target_id:
                raise ValueError(
                    f"Conflicting non-ignored character targets for {key}: {previous} vs {target_id}"
                )

    if not target_by_character:
        raise ValueError("No supervised characters available for class-weight estimation.")
    counts = torch.bincount(
        torch.tensor(list(target_by_character.values()), dtype=torch.long),
        minlength=len(BIO_LABELS),
    ).float().clamp_min(1.0)
    o_count = float(counts[O_LABEL_ID].item())
    # Keep O at unit weight; gently upweight rarer BIO classes without extreme ratios.
    weights = (o_count / counts).pow(0.20)
    weights[O_LABEL_ID] = 1.0
    weights = weights.clamp(min=1.0, max=2.5)
    print(
        f"Unique-character class weights: O=1.0 | supervised chars={len(target_by_character):,} | "
        f"min={weights.min().item():.3f} | max={weights.max().item():.3f}"
    )
    return weights.to(device)

class WeightedFocalLoss(nn.Module):
    """Class-weighted focal loss over valid character labels.
    - Excludes IGNORE_INDEX positions from both numerator and denominator.
    - Normalizes by sum of active alpha weights for numerical stability across batches.
    """
    def __init__(self, class_weights=None, gamma=1.5, ignore_index=IGNORE_INDEX):
        super().__init__()
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.detach().float().clone())
        else:
            self.class_weights = None
        self.gamma = float(gamma)
        self.ignore_index = int(ignore_index)

    def forward(self, logits, targets):
        if logits.ndim > 2:
            logits = logits.reshape(-1, logits.shape[-1])
        targets = targets.reshape(-1).long()
        valid = targets != self.ignore_index
        if not torch.any(valid):
            return logits.sum() * 0.0

        valid_logits = logits[valid]
        valid_targets = targets[valid]
        log_probs = F.log_softmax(valid_logits, dim=-1)
        log_pt = log_probs.gather(1, valid_targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()
        focal_factor = (1.0 - pt).clamp(min=0.0, max=1.0) ** self.gamma

        if self.class_weights is not None:
            alpha = self.class_weights.to(logits.device)[valid_targets]
            loss = -alpha * focal_factor * log_pt
            return loss.sum() / alpha.sum().clamp_min(1e-6)
        else:
            loss = -focal_factor * log_pt
            return loss.mean()


def _character_model_inputs(batch, device):
    return {
        "input_ids": batch["input_ids"].to(device),
        "attention_mask": batch["attention_mask"].to(device),
        "char_ids": batch["char_ids"].to(device),
        "char_to_token": batch["char_to_token"].to(device),
        "char_positions": batch["char_positions"].to(device),
        "char_mask": batch["char_mask"].to(device),
    }


def collect_document_character_logits(model, dataset, texts_by_id, device=DEVICE,
                                      batch_size=4, pad_token_id=0, criterion=None):
    """Combine overlapping window logits using a documented trapezoidal weighting rule.
    Weighting rule:
      For each window of length L, character p receives weight:
        w(p) = 1.0 + min(min(p, L - 1 - p), 64) / 64.0
      Characters in overlapping windows are aggregated as:
        logit_doc(c) = sum(w(p) * logit_window(p)) / sum(w(p))
    """
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        collate_fn=lambda batch: collate_character_windows(batch, pad_token_id),
    )
    model.eval()
    accumulated = {}
    accumulated_weight = {}
    loss_total, loss_batches = 0.0, 0

    with torch.no_grad():
        for batch in loader:
            inputs = _character_model_inputs(batch, device)
            logits = model(**inputs)
            if criterion is not None:
                loss = criterion(
                    logits.reshape(-1, logits.shape[-1]),
                    batch["char_labels"].to(device).reshape(-1),
                )
                if torch.isfinite(loss):
                    loss_total += float(loss.item())
                    loss_batches += 1

            logits_cpu = logits.detach().float().cpu().numpy()
            for row, doc_id in enumerate(batch["document_ids"]):
                doc_id = str(doc_id)
                text = texts_by_id[doc_id]
                if doc_id not in accumulated:
                    accumulated[doc_id] = np.zeros(
                        (len(text), len(BIO_LABELS)), dtype=np.float32
                    )
                    accumulated_weight[doc_id] = np.zeros(len(text), dtype=np.float32)
                start = int(batch["char_starts"][row])
                length = int(batch["char_lengths"][row])
                end = start + length
                if not (0 <= start < end <= len(text)):
                    raise ValueError(
                        f"Invalid character window [{start}, {end}) for {doc_id} length {len(text)}"
                    )
                window_logits = logits_cpu[row, :length, :]
                positions = np.arange(length)
                edge_distance = np.minimum(positions, length - 1 - positions)
                blend_weights = 1.0 + np.minimum(edge_distance, 64).astype(np.float32) / 64.0
                accumulated[doc_id][start:end] += window_logits * blend_weights[:, None]
                accumulated_weight[doc_id][start:end] += blend_weights

    document_logits = {}
    for raw_doc_id, text in texts_by_id.items():
        doc_id = str(raw_doc_id)
        if len(text) == 0:
            document_logits[doc_id] = np.zeros((0, len(BIO_LABELS)), dtype=np.float32)
            continue
        sums = accumulated.get(doc_id)
        weight_sums = accumulated_weight.get(doc_id)
        if sums is None:
            result = np.full((len(text), len(BIO_LABELS)), -5.0, dtype=np.float32)
            result[:, O_LABEL_ID] = 0.0
            document_logits[doc_id] = result
            continue
        covered = weight_sums > 0
        result = np.full((len(text), len(BIO_LABELS)), -5.0, dtype=np.float32)
        result[:, O_LABEL_ID] = 0.0
        result[covered] = sums[covered] / weight_sums[covered, None]
        document_logits[doc_id] = result

    mean_loss = loss_total / max(loss_batches, 1) if criterion is not None else None
    return document_logits, mean_loss


def decode_document_logits(document_logits_by_id, allow_orphan_i=False, min_entity_chars=1):
    return {
        str(doc_id): [list(span) for span in decode_character_bio_logits(
            logits, allow_orphan_i=allow_orphan_i, min_entity_chars=min_entity_chars
        )]
        for doc_id, logits in document_logits_by_id.items()
    }


def evaluate_character_boundary_model(model, dataset, texts_by_id, gold_by_id,
                                      device=DEVICE, batch_size=4, criterion=None,
                                      allow_orphan_i=False, min_entity_chars=1):
    """Evaluates character-boundary model using strict-span metrics."""
    logits_by_doc, mean_loss = collect_document_character_logits(
        model, dataset, texts_by_id, device=device, batch_size=batch_size,
        pad_token_id=getattr(tokenizer, "pad_token_id", 0) or 0,
        criterion=criterion,
    )
    predictions = decode_document_logits(
        logits_by_doc, allow_orphan_i=allow_orphan_i, min_entity_chars=min_entity_chars
    )
    all_ids = [str(x) for x in texts_by_id.keys()]
    gold_normalized = {str(k): v for k, v in gold_by_id.items()}
    for doc_id in all_ids:
        predictions.setdefault(doc_id, [])
        gold_normalized.setdefault(doc_id, [])
    metrics = strict_span_metrics(
        gold_by_id=gold_normalized,
        predicted_by_id=predictions,
        labels=LABELS,
        document_ids=all_ids,
    )
    per_label = metrics["per_label"]
    macro_p = sum(v["precision"] for v in per_label.values()) / max(len(per_label), 1)
    macro_r = sum(v["recall"] for v in per_label.values()) / max(len(per_label), 1)
    total_tp = sum(v["tp"] for v in per_label.values())
    total_fp = sum(v["fp"] for v in per_label.values())
    total_fn = sum(v["fn"] for v in per_label.values())

    return {
        "strict_macro_f1": metrics["strict_macro_f1"],
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "total_true_positives": total_tp,
        "total_false_positives": total_fp,
        "total_false_negatives": total_fn,
        "per_label": per_label,
        "predictions_by_doc": predictions,
        "document_logits": logits_by_doc,
        "val_loss": mean_loss,
        "documents_evaluated": metrics["documents_evaluated"],
    }


def train_character_boundary_experiment(train_dataset, val_dataset,
                                        train_texts, val_texts, val_gold,
                                        checkpoint_path, output_dir,
                                        epochs=6, batch_size=2,
                                        grad_accum_steps=2, use_amp=True,
                                        backbone_lr=1e-5, head_lr=3e-4,
                                        loss_type="weighted_focal", focal_gamma=1.5,
                                        device=DEVICE):
    """Full fine-tuning of V1 plus exact character-level BIO head.

    Uses gradient accumulation and CUDA FP16 autocast when available to fit a T4 more
    reliably. Scheduler steps count optimizer updates, not micro-batches.
    """
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)
    pad_token_id = getattr(tokenizer, "pad_token_id", 0) or 0
    grad_accum_steps = max(1, int(grad_accum_steps))
    amp_enabled = bool(use_amp and torch.cuda.is_available() and str(device).startswith("cuda"))
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=lambda batch: collate_character_windows(batch, pad_token_id),
        pin_memory=amp_enabled,
    )

    # Keep old checkpoints as an explicit fallback; start a fresh run from V1 weights.
    best_f1 = -1.0
    if os.path.isfile(checkpoint_path):
        backup_path = checkpoint_path + ".previous_best.pt"
        if not os.path.exists(backup_path):
            try:
                import shutil
                shutil.copy2(checkpoint_path, backup_path)
                print(f"[Checkpoint safety] Previous checkpoint backed up to {backup_path}")
            except Exception as exc:
                print(f"[Checkpoint safety] Backup warning: {exc}")
        print("[Fresh run] Existing checkpoint will be replaced only by this run's best validation epoch.")

    char_model = V1CharacterBoundaryNERModel(
        backbone=copy.deepcopy(model),
        char_vocab_size=CHAR_VOCAB_SIZE,
        num_labels=len(BIO_LABELS),
        freeze_backbone=False,
    ).to(device)

    optimizer = torch.optim.AdamW([
        {"params": char_model.backbone.parameters(), "lr": backbone_lr, "weight_decay": 0.01},
        {"params": char_model.char_head.parameters(), "lr": head_lr, "weight_decay": 0.01},
    ])
    char_weights = _make_char_class_weights(train_dataset, device)
    if loss_type == "weighted_focal":
        criterion = WeightedFocalLoss(char_weights, gamma=focal_gamma, ignore_index=IGNORE_INDEX).to(device)
    elif loss_type == "weighted_cross_entropy":
        criterion = nn.CrossEntropyLoss(weight=char_weights, ignore_index=IGNORE_INDEX)
    else:
        raise ValueError("loss_type must be 'weighted_focal' or 'weighted_cross_entropy'")

    updates_per_epoch = max(1, (len(train_loader) + grad_accum_steps - 1) // grad_accum_steps)
    total_steps = max(1, updates_per_epoch * int(epochs))
    warmup_steps = max(1, int(0.05 * total_steps))
    def _lr_multiplier(step):
        if step < warmup_steps:
            return max(0.10, float(step + 1) / float(warmup_steps))
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return 0.10 + 0.90 * 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=_lr_multiplier)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    history = []
    print("Starting exact-character V1 fine-tuning")
    print(f"Train windows: {len(train_dataset)} | Validation windows: {len(val_dataset)}")
    print(f"Epochs: {epochs} | micro-batch: {batch_size} | accumulation: {grad_accum_steps} "
          f"(effective batch approx. {batch_size * grad_accum_steps}) | AMP: {amp_enabled}")
    print(f"Backbone LR: {backbone_lr} | head LR: {head_lr} | loss={loss_type} | warm-up + cosine decay")

    for epoch in range(1, epochs + 1):
        char_model.train()
        epoch_loss, batch_count = 0.0, 0
        start_time = time.time()
        optimizer.zero_grad(set_to_none=True)
        for batch_idx, batch in enumerate(train_loader):
            inputs = _character_model_inputs(batch, device)
            labels_char = batch["char_labels"].to(device, non_blocking=amp_enabled)
            group_start = (batch_idx // grad_accum_steps) * grad_accum_steps
            group_size = min(grad_accum_steps, len(train_loader) - group_start)
            with torch.autocast(device_type=str(device).split(":")[0], dtype=torch.float16, enabled=amp_enabled):
                logits = char_model(**inputs)
            # Compute the focal/CE reduction in FP32 for numerical stability.
            loss = criterion(logits.float().reshape(-1, logits.shape[-1]), labels_char.reshape(-1))
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite character loss at epoch {epoch}, batch {batch_idx}: {loss.item()}")
            scaler.scale(loss / group_size).backward()
            epoch_loss += float(loss.detach().item())
            batch_count += 1

            is_update = ((batch_idx + 1) % grad_accum_steps == 0) or (batch_idx + 1 == len(train_loader))
            if is_update:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(char_model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        train_loss = epoch_loss / max(batch_count, 1)
        val_metrics = evaluate_character_boundary_model(
            char_model, val_dataset, val_texts, val_gold,
            device=device, batch_size=batch_size, criterion=criterion,
        )
        elapsed = time.time() - start_time
        row = {
            "epoch": epoch, "train_loss": train_loss, "val_loss": val_metrics["val_loss"],
            "strict_macro_f1": val_metrics["strict_macro_f1"],
            "macro_precision": val_metrics["macro_precision"], "macro_recall": val_metrics["macro_recall"],
            "total_true_positives": val_metrics["total_true_positives"],
            "total_false_positives": val_metrics["total_false_positives"],
            "total_false_negatives": val_metrics["total_false_negatives"],
            "per_label": val_metrics["per_label"], "seconds": elapsed,
        }
        history.append(row)
        print(
            f"Epoch {epoch}/{epochs} | train loss={train_loss:.4f} | val strict F1={row['strict_macro_f1']:.4f} "
            f"| P={row['macro_precision']:.4f} | R={row['macro_recall']:.4f} "
            f"| TP={row['total_true_positives']} | FP={row['total_false_positives']} | FN={row['total_false_negatives']} "
            f"| backbone LR={optimizer.param_groups[0]['lr']:.2e} | head LR={optimizer.param_groups[1]['lr']:.2e} "
            f"| AMP={amp_enabled} | {elapsed:.1f}s"
        )
        print("  Per-label validation F1: " + " | ".join(
            f"{label}={val_metrics['per_label'][label]['f1']:.3f}" for label in LABELS
        ))

        if row["strict_macro_f1"] > best_f1:
            best_f1 = row["strict_macro_f1"]
            tmp_checkpoint = checkpoint_path + ".tmp"
            torch.save({
                "epoch": epoch, "val_f1": best_f1, "model_state_dict": char_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(), "loss_type": loss_type,
                "focal_gamma": float(focal_gamma), "char_to_id": CHAR_TO_ID, "bio_labels": BIO_LABELS,
                "architecture": "V1CharacterBoundaryNERModel",
                "run_config": {"epochs": epochs, "batch_size": batch_size, "grad_accum_steps": grad_accum_steps,
                               "use_amp": amp_enabled, "backbone_lr": backbone_lr, "head_lr": head_lr,
                               "loss_type": loss_type, "focal_gamma": float(focal_gamma)},
            }, tmp_checkpoint)
            os.replace(tmp_checkpoint, checkpoint_path)
            print("  [Checkpoint] Saved new best character-boundary checkpoint:", checkpoint_path)

    history_path = os.path.join(output_dir, "character_boundary_training_history.json")
    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)

    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint was not created: {checkpoint_path}")
    state = torch.load(checkpoint_path, map_location="cpu")
    char_model.load_state_dict(state["model_state_dict"], strict=True)
    print(f"Restored best character-boundary checkpoint from epoch {state['epoch']} (F1={state['val_f1']:.4f})")
    return char_model, history, best_f1

print("Character-level model, loss, evaluator, and trainer defined.")

In [ ]:
# ==============================================================================
# Static Unit Tests: Invariant Verification for Exact Character Boundary Pipeline
# Covers 9 important exact-span invariants on CPU without training:
# 1. Entity start/end inside one tokenizer token
# 2. Entity crossing a window boundary
# 3. Overlap between adjacent windows & logit blending
# 4. Adjacent entities of different labels
# 5. Punctuation inside an entity
# 6. An empty-entity document
# 7. Unicode text
# 8. Document shorter than one window
# ==============================================================================

def run_character_boundary_synthetic_tests():
    print("=" * 65)
    print("RUNNING 9 STATIC SYNTHETIC INVARIANT TESTS (CPU ONLY)")
    print("=" * 65)

    # 1. Entity start/end inside one tokenizer token
    text_1 = "Pre-SupremeX-Court jurisdiction"
    ent_1 = [{"start": 4, "end": 11, "label": "COURT"}]
    t_1 = build_exact_character_targets(text_1, ent_1, 0, len(text_1))
    assert t_1[4] == BIO_TO_ID["B-COURT"]
    assert all(t_1[p] == BIO_TO_ID["I-COURT"] for p in range(5, 11))
    assert t_1[11] == O_LABEL_ID
    print("[PASS] Test 1: Entity start/end inside sub-token exactness verified.")

    # 2. Entity crossing a window boundary (masked without corrupting targets)
    text_2 = "High Court of Bombay"
    ent_2 = [{"start": 0, "end": 20, "label": "COURT"}]
    t_2_w1 = build_exact_character_targets(text_2, ent_2, 0, 10)
    t_2_w2 = build_exact_character_targets(text_2, ent_2, 10, 20)
    assert all(x == IGNORE_INDEX for x in t_2_w1)
    assert all(x == IGNORE_INDEX for x in t_2_w2)
    print("[PASS] Test 2: Entity crossing window boundary masked with IGNORE_INDEX.")

    # 3. Overlap between adjacent windows & logit combination
    text_3 = "The Supreme Court of India is established."
    wA_logits = np.zeros((25, len(BIO_LABELS)), dtype=np.float32)
    wB_logits = np.zeros((len(text_3) - 15, len(BIO_LABELS)), dtype=np.float32)
    wA_logits[4, BIO_TO_ID["B-COURT"]] = 5.0
    for p in range(5, 17):
        wA_logits[p, BIO_TO_ID["I-COURT"]] = 5.0
    for p in range(0, 2):  # positions 15 and 16 in doc
        wB_logits[p, BIO_TO_ID["I-COURT"]] = 5.0
    accum = np.zeros((len(text_3), len(BIO_LABELS)), dtype=np.float32)
    accum_w = np.zeros(len(text_3), dtype=np.float32)
    accum[0:25] += wA_logits * 1.0
    accum_w[0:25] += 1.0
    accum[15:len(text_3)] += wB_logits * 1.0
    accum_w[15:len(text_3)] += 1.0
    combined = accum / accum_w[:, None]
    dec_3 = decode_character_bio_logits(combined)
    assert dec_3 == [(4, 17, "COURT")]
    print("[PASS] Test 3: Overlap between adjacent windows correctly blended and decoded.")

    # 4. Adjacent entities of different labels
    text_4 = "Justice Ram Lal vs State of Maharashtra"
    ent_4 = [{"start": 0, "end": 15, "label": "JUDGE"}, {"start": 19, "end": 39, "label": "RESP"}]
    t_4 = build_exact_character_targets(text_4, ent_4, 0, len(text_4))
    one_hot_4 = np.zeros((len(text_4), len(BIO_LABELS)), dtype=np.float32)
    for p, tid in enumerate(t_4):
        one_hot_4[p, tid] = 10.0
    dec_4 = decode_character_bio_logits(one_hot_4)
    assert dec_4 == [(0, 15, "JUDGE"), (19, 39, "RESP")]
    print("[PASS] Test 4: Adjacent entities of different labels cleanly separated.")

    # 5. Punctuation inside an entity
    text_5 = "Section 302, I.P.C. was invoked."
    ent_5 = [{"start": 0, "end": 19, "label": "STAT"}]
    t_5 = build_exact_character_targets(text_5, ent_5, 0, len(text_5))
    one_hot_5 = np.zeros((len(text_5), len(BIO_LABELS)), dtype=np.float32)
    for p, tid in enumerate(t_5):
        one_hot_5[p, tid] = 10.0
    dec_5 = decode_character_bio_logits(one_hot_5)
    assert dec_5 == [(0, 19, "STAT")]
    print("[PASS] Test 5: Punctuation inside entity span preserved exactly.")

    # 6. Empty-entity document
    text_6 = "No legal entities are present in this simple sentence."
    t_6 = build_exact_character_targets(text_6, [], 0, len(text_6))
    assert all(x == O_LABEL_ID for x in t_6)
    one_hot_6 = np.zeros((len(text_6), len(BIO_LABELS)), dtype=np.float32)
    one_hot_6[:, O_LABEL_ID] = 10.0
    dec_6 = decode_character_bio_logits(one_hot_6)
    assert dec_6 == []
    print("[PASS] Test 6: Empty-entity document produces 0 spans.")

    # 7. Unicode text
    text_7 = "न्यायालय New Delhi Bench orders compliance."
    ent_7 = [{"start": 0, "end": 7, "label": "COURT"}]
    t_7 = build_exact_character_targets(text_7, ent_7, 0, len(text_7))
    one_hot_7 = np.zeros((len(text_7), len(BIO_LABELS)), dtype=np.float32)
    for p, tid in enumerate(t_7):
        one_hot_7[p, tid] = 10.0
    dec_7 = decode_character_bio_logits(one_hot_7)
    assert dec_7 == [(0, 7, "COURT")]
    print("[PASS] Test 7: Unicode text offsets and multibyte characters decoded correctly.")

    # 8. Document shorter than one window
    text_8 = "Order."
    t_8 = build_exact_character_targets(text_8, [], 0, len(text_8))
    assert len(t_8) == 6
    print("[PASS] Test 8: Document shorter than one window handled cleanly.")

    # 9. One-character entity must not be silently dropped by the decoder.
    one_char_logits = np.full((1, len(BIO_LABELS)), -10.0, dtype=np.float32)
    one_char_logits[0, BIO_TO_ID["B-CASENO"]] = 10.0
    assert decode_character_bio_logits(one_char_logits) == [(0, 1, "CASENO")]
    print("[PASS] Test 9: one-character gold-compatible span is retained.")

    print("=" * 65)
    print("ALL 9 STATIC SYNTHETIC TESTS PASSED SUCCESSFULLY!")
    print("=" * 65)
    return True

run_character_boundary_synthetic_tests()

In [ ]:
# Run the exact-character boundary experiment.
# This trains a separate model from a fresh copy of the restored V1 checkpoint.

CHAR_BOUNDARY_WINDOW_OVERLAP = MODEL_CONTEXT_LENGTH // 2
CHAR_BOUNDARY_EPOCHS = 6
CHAR_BOUNDARY_BATCH_SIZE = 2
CHAR_BOUNDARY_GRAD_ACCUM_STEPS = 2
CHAR_BOUNDARY_USE_AMP = True
CHAR_BOUNDARY_BACKBONE_LR = 1e-5
CHAR_BOUNDARY_HEAD_LR = 3e-4

_train_texts_char = {str(k): v for k, v in TEXTS_BY_ID.items()}
_train_gold_char = {str(k): v for k, v in ANNOTATIONS_BY_ID.items()}
_val_texts_char = {str(k): TEXTS_BY_ID[str(k)] for k in VAL_DOC_IDS}
_val_gold_char = {str(k): ANNOTATIONS_BY_ID[str(k)] for k in VAL_DOC_IDS}

char_train_dataset = CharacterBoundaryWindowDataset(
    document_ids=TRAIN_DOC_IDS,
    text_by_id=_train_texts_char,
    entities_by_id=_train_gold_char,
    tokenizer_obj=tokenizer,
    max_length=MODEL_CONTEXT_LENGTH,
    window_overlap=CHAR_BOUNDARY_WINDOW_OVERLAP,
)
char_val_dataset = CharacterBoundaryWindowDataset(
    document_ids=VAL_DOC_IDS,
    text_by_id=_val_texts_char,
    entities_by_id=_val_gold_char,
    tokenizer_obj=tokenizer,
    max_length=MODEL_CONTEXT_LENGTH,
    window_overlap=CHAR_BOUNDARY_WINDOW_OVERLAP,
)

assert len(char_train_dataset) > 0, "Character-boundary train dataset is empty."
assert len(char_val_dataset) > 0, "Character-boundary validation dataset is empty."
print("Character-boundary windows:", len(char_train_dataset), "train /", len(char_val_dataset), "validation")

# Audit character coverage across validation documents
audit_document_character_coverage(_val_texts_char, char_val_dataset)

CHAR_BOUNDARY_OUTPUT_DIR = os.path.join(PHASE2_ROOT_DIR, "V1_Rescue_0p323_CharacterBoundary")
CHAR_BOUNDARY_CHECKPOINT_PATH = os.path.join(
    PHASE2_CHECKPOINT_DIR, "best_v1_character_boundary_6ep_calibrated.pt"
)

char_boundary_model, char_boundary_history, char_boundary_best_f1 = train_character_boundary_experiment(
    train_dataset=char_train_dataset,
    val_dataset=char_val_dataset,
    train_texts=_train_texts_char,
    val_texts=_val_texts_char,
    val_gold=_val_gold_char,
    checkpoint_path=CHAR_BOUNDARY_CHECKPOINT_PATH,
    output_dir=CHAR_BOUNDARY_OUTPUT_DIR,
    epochs=CHAR_BOUNDARY_EPOCHS,
    batch_size=CHAR_BOUNDARY_BATCH_SIZE,
    grad_accum_steps=CHAR_BOUNDARY_GRAD_ACCUM_STEPS,
    use_amp=CHAR_BOUNDARY_USE_AMP,
    backbone_lr=CHAR_BOUNDARY_BACKBONE_LR,
    head_lr=CHAR_BOUNDARY_HEAD_LR,
    loss_type="weighted_focal",
    focal_gamma=1.5,
    device=DEVICE,
)

# Final validation metrics from the restored best epoch.
char_boundary_val_metrics = evaluate_character_boundary_model(
    char_boundary_model,
    char_val_dataset,
    _val_texts_char,
    _val_gold_char,
    device=DEVICE,
    batch_size=CHAR_BOUNDARY_BATCH_SIZE,
)
print("\nCHARACTER-BOUNDARY VALIDATION RESULTS (RESTORED BEST CHECKPOINT)")
print("Strict Macro-F1:", f"{char_boundary_val_metrics['strict_macro_f1']:.4f}")
print("Macro-Precision:", f"{char_boundary_val_metrics['macro_precision']:.4f}")
print("Macro-Recall:", f"{char_boundary_val_metrics['macro_recall']:.4f}")
print("True Positives:", char_boundary_val_metrics['total_true_positives'])
print("False Positives:", char_boundary_val_metrics['total_false_positives'])
print("False Negatives:", char_boundary_val_metrics['total_false_negatives'])

print("\nPer-label strict F1:")
for label in LABELS:
    stats = char_boundary_val_metrics["per_label"][label]
    print(f"{label:12s} F1={stats['f1']:.4f} P={stats['precision']:.4f} R={stats['recall']:.4f} TP={stats['tp']} FP={stats['fp']} FN={stats['fn']}")

# Save validation metrics and experiment manifest.
char_metrics_save = {k: v for k, v in char_boundary_val_metrics.items() if k not in ("predictions_by_doc", "document_logits")}
char_metrics_save.update({
    "experiment_id": "EXP-1D_V1_CharacterBoundary",
    "checkpoint_path": CHAR_BOUNDARY_CHECKPOINT_PATH,
    "best_epoch": int(torch.load(CHAR_BOUNDARY_CHECKPOINT_PATH, map_location="cpu")["epoch"]),
    "best_validation_strict_macro_f1": float(char_boundary_best_f1),
    "alignment_strategy": "character-level BIO targets with balanced loss and multi-scale convolutions",
})
with open(os.path.join(CHAR_BOUNDARY_OUTPUT_DIR, "character_boundary_validation_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(char_metrics_save, f, indent=2, ensure_ascii=False)

# Compare against the token BIO baseline if Cell 109 was run.
if "final_val_metrics" in globals():
    base_f1 = float(final_val_metrics["strict_macro_f1"])
    char_f1 = float(char_boundary_val_metrics["strict_macro_f1"])
    print(f"\nBaseline token-BIO validation F1: {base_f1:.4f}")
    print(f"Character-boundary validation F1: {char_f1:.4f}")
    print(f"Absolute change: {char_f1 - base_f1:+.4f}")

In [ ]:
# Validation-only confidence calibration: reduce false positives without retraining.
# Entity thresholds subtract a bias from both B-label and I-label logits relative to O.
# Derived STRICTLY from held-out validation documents; test labels are never used.

import os
import json
import numpy as np
import torch


def decode_document_logits_with_thresholds(document_logits_by_id, thresholds_by_label,
                                           allow_orphan_i=False, min_entity_chars=1):
    """Decode exact character spans after subtracting a per-label logit bias."""
    decoded = {}
    for document_id, numpy_logits in document_logits_by_id.items():
        adjusted = np.asarray(numpy_logits, dtype=np.float32).copy()
        for label in LABELS:
            bias = float(thresholds_by_label.get(label, 0.0))
            if bias != 0.0:
                b_id = BIO_TO_ID[f"B-{label}"]
                i_id = BIO_TO_ID[f"I-{label}"]
                adjusted[:, b_id] -= bias
                adjusted[:, i_id] -= bias
        decoded[str(document_id)] = [list(span) for span in decode_character_bio_logits(
            adjusted, allow_orphan_i=allow_orphan_i, min_entity_chars=min_entity_chars
        )]
    return decoded


def score_thresholds(document_logits_by_id, thresholds_by_label):
    """Evaluates a candidate threshold dictionary on the held-out validation split."""
    predictions = decode_document_logits_with_thresholds(document_logits_by_id, thresholds_by_label)
    doc_ids = [str(x) for x in _val_texts_char.keys()]
    for doc_id in doc_ids:
        predictions.setdefault(doc_id, [])
    gold = {str(k): v for k, v in _val_gold_char.items()}
    for doc_id in doc_ids:
        gold.setdefault(doc_id, [])
    metrics = strict_span_metrics(
        gold_by_id=gold,
        predicted_by_id=predictions,
        labels=LABELS,
        document_ids=doc_ids,
    )
    per_label = metrics["per_label"]
    macro_precision = sum(x["precision"] for x in per_label.values()) / max(1, len(per_label))
    macro_recall = sum(x["recall"] for x in per_label.values()) / max(1, len(per_label))
    total_tp = sum(x["tp"] for x in per_label.values())
    total_fp = sum(x["fp"] for x in per_label.values())
    total_fn = sum(x["fn"] for x in per_label.values())
    return {
        "strict_macro_f1": float(metrics["strict_macro_f1"]),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "total_true_positives": total_tp,
        "total_false_positives": total_fp,
        "total_false_negatives": total_fn,
        "per_label": per_label,
        "predictions_by_doc": predictions,
    }

print("Collecting validation character logits once for calibration...")
_calibration_logits_by_doc, _ = collect_document_character_logits(
    char_boundary_model,
    char_val_dataset,
    _val_texts_char,
    device=DEVICE,
    batch_size=CHAR_BOUNDARY_BATCH_SIZE,
    pad_token_id=getattr(tokenizer, "pad_token_id", 0) or 0,
)

_threshold_candidates = [-0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
CHAR_CONFIDENCE_THRESHOLDS = {label: 0.0 for label in LABELS}
_raw_calibration_metrics = score_thresholds(_calibration_logits_by_doc, CHAR_CONFIDENCE_THRESHOLDS)
_best_metrics = _raw_calibration_metrics
_best_score = _raw_calibration_metrics["strict_macro_f1"]

print(f"Raw validation Strict Macro-F1: {_best_score:.4f} (TP={_best_metrics['total_true_positives']}, FP={_best_metrics['total_false_positives']}, FN={_best_metrics['total_false_negatives']})")
print("Searching per-entity logit biases with false-positive suppression tie-breaking...")

for _label in LABELS:
    _current_best_threshold = CHAR_CONFIDENCE_THRESHOLDS[_label]
    _current_best_metrics = _best_metrics
    _current_best_score = _current_best_metrics["strict_macro_f1"]
    _current_label_fp = _current_best_metrics["per_label"][_label]["fp"]
    _current_label_tp = _current_best_metrics["per_label"][_label]["tp"]

    for _candidate in _threshold_candidates:
        _trial = dict(CHAR_CONFIDENCE_THRESHOLDS)
        _trial[_label] = float(_candidate)
        _trial_metrics = score_thresholds(_calibration_logits_by_doc, _trial)
        trial_f1 = _trial_metrics["strict_macro_f1"]
        trial_tp = _trial_metrics["per_label"][_label]["tp"]
        trial_fp = _trial_metrics["per_label"][_label]["fp"]

        # Rule 1: Strict macro-F1 improvement
        if trial_f1 > _current_best_score + 1e-6:
            _current_best_score = trial_f1
            _current_best_metrics = _trial_metrics
            _current_best_threshold = float(_candidate)
            _current_label_fp = trial_fp
            _current_label_tp = trial_tp
        # Rule 2: Equal macro-F1 tie-breaker:
        # If F1 ties (e.g. on 0-TP classes like WIT, A.COUNSEL), accept candidate if it suppresses FP
        # without reducing TP
        elif abs(trial_f1 - _current_best_score) <= 1e-6:
            if trial_tp >= _current_label_tp and trial_fp < _current_label_fp:
                _current_best_score = trial_f1
                _current_best_metrics = _trial_metrics
                _current_best_threshold = float(_candidate)
                _current_label_fp = trial_fp
                _current_label_tp = trial_tp

    CHAR_CONFIDENCE_THRESHOLDS[_label] = _current_best_threshold
    _best_metrics = _current_best_metrics
    print(f"  {_label:12s} bias={_current_best_threshold:+.1f} -> current macro-F1={_best_metrics['strict_macro_f1']:.4f} (Label TP={_best_metrics['per_label'][_label]['tp']}, FP={_best_metrics['per_label'][_label]['fp']})")

CHAR_CALIBRATED_VAL_METRICS = score_thresholds(
    _calibration_logits_by_doc, CHAR_CONFIDENCE_THRESHOLDS
)

# Safety check: Compare calibrated vs uncalibrated. Fall back if calibration degrades F1.
if CHAR_CALIBRATED_VAL_METRICS["strict_macro_f1"] < _raw_calibration_metrics["strict_macro_f1"]:
    print("\n[Safety Fallback] Calibrated F1 is lower than raw F1. Resetting thresholds to 0.0 (no calibration).")
    CHAR_CONFIDENCE_THRESHOLDS = {label: 0.0 for label in LABELS}
    CHAR_CALIBRATED_VAL_METRICS = _raw_calibration_metrics
else:
    print("\n[Calibration Confirmed] Calibrated validation Strict Macro-F1 >= raw baseline.")

print("\nCALIBRATION COMPARISON (Validation Held-Out Split Only):")
print(f"Raw:        F1={_raw_calibration_metrics['strict_macro_f1']:.4f} "
      f"P={_raw_calibration_metrics['macro_precision']:.4f} "
      f"R={_raw_calibration_metrics['macro_recall']:.4f} "
      f"TP={_raw_calibration_metrics['total_true_positives']} "
      f"FP={_raw_calibration_metrics['total_false_positives']} "
      f"FN={_raw_calibration_metrics['total_false_negatives']}")
print(f"Calibrated: F1={CHAR_CALIBRATED_VAL_METRICS['strict_macro_f1']:.4f} "
      f"P={CHAR_CALIBRATED_VAL_METRICS['macro_precision']:.4f} "
      f"R={CHAR_CALIBRATED_VAL_METRICS['macro_recall']:.4f} "
      f"TP={CHAR_CALIBRATED_VAL_METRICS['total_true_positives']} "
      f"FP={CHAR_CALIBRATED_VAL_METRICS['total_false_positives']} "
      f"FN={CHAR_CALIBRATED_VAL_METRICS['total_false_negatives']}")

print("\nCalibrated per-label results:")
for _label in LABELS:
    _m = CHAR_CALIBRATED_VAL_METRICS["per_label"][_label]
    print(f"  {_label:12s} F1={_m['f1']:.3f} P={_m['precision']:.3f} "
          f"R={_m['recall']:.3f} TP={_m['tp']} FP={_m['fp']} FN={_m['fn']}")

_calibration_dir = CHAR_BOUNDARY_OUTPUT_DIR
os.makedirs(_calibration_dir, exist_ok=True)
with open(os.path.join(_calibration_dir, "character_confidence_thresholds.json"), "w", encoding="utf-8") as _f:
    json.dump(CHAR_CONFIDENCE_THRESHOLDS, _f, indent=2, ensure_ascii=False)
_calibrated_save = {k: v for k, v in CHAR_CALIBRATED_VAL_METRICS.items() if k != "predictions_by_doc"}
_calibrated_save.update({
    "calibration_thresholds": CHAR_CONFIDENCE_THRESHOLDS,
    "raw_strict_macro_f1": float(_raw_calibration_metrics["strict_macro_f1"]),
    "calibrated_strict_macro_f1": float(CHAR_CALIBRATED_VAL_METRICS["strict_macro_f1"]),
    "calibration_safety_note": "Thresholds selected on validation data only; model weights were not modified.",
})
with open(os.path.join(_calibration_dir, "character_boundary_calibrated_validation_metrics.json"), "w", encoding="utf-8") as _f:
    json.dump(_calibrated_save, _f, indent=2, ensure_ascii=False)
print("Saved calibration thresholds and metrics to:", _calibration_dir)

## Final refit using every labelled document

The validation run is used to select an epoch count and calibrate per-label biases. For test inference, the next cell restarts from the same Phase-1 V1 checkpoint and trains that fixed number of epochs on **all** publicly labelled documents (both labelled folds), so the validation holdout is not permanently discarded. The refit does not select new hyperparameters or use test labels. Validation-derived decoding biases remain an estimate and are also compared against a raw, uncalibrated fallback submission.

This cell costs an additional training run. Keep the original best-validation checkpoint and submission ZIP as a fallback; the new run writes to a separate file.

In [ ]:
# Final refit: use all labelled documents for the deployable checkpoint.
# Epoch count is selected from the held-out validation experiment; no test labels are used.
import os, json, time, copy, math
import torch
from torch.utils.data import DataLoader

if not isinstance(char_boundary_history, list) or not char_boundary_history:
    raise RuntimeError("Run the character-boundary training cell first; char_boundary_history is missing.")
_valid_rows = [row for row in char_boundary_history if "strict_macro_f1" in row and "epoch" in row]
if not _valid_rows:
    raise RuntimeError("No valid epoch/strict_macro_f1 records found in char_boundary_history.")
BEST_EPOCHS_FOR_REFIT = int(max(_valid_rows, key=lambda row: float(row["strict_macro_f1"]))["epoch"])
BEST_VALIDATION_F1_FOR_REFIT = float(max(_valid_rows, key=lambda row: float(row["strict_macro_f1"]))["strict_macro_f1"])

_all_labeled_texts = {str(k): v for k, v in TEXTS_BY_ID.items()}
_all_labeled_gold = {str(k): v for k, v in ANNOTATIONS_BY_ID.items()}
_all_labeled_ids = list(_all_labeled_texts.keys())
_all_labeled_dataset = CharacterBoundaryWindowDataset(
    document_ids=_all_labeled_ids,
    text_by_id=_all_labeled_texts,
    entities_by_id=_all_labeled_gold,
    tokenizer_obj=tokenizer,
    max_length=MODEL_CONTEXT_LENGTH,
    window_overlap=CHAR_BOUNDARY_WINDOW_OVERLAP,
)
if len(_all_labeled_dataset) == 0:
    raise RuntimeError("All-labelled-data character window dataset is empty.")

_refit_loader = DataLoader(
    _all_labeled_dataset,
    batch_size=CHAR_BOUNDARY_BATCH_SIZE,
    shuffle=True,
    pin_memory=bool(CHAR_BOUNDARY_USE_AMP and torch.cuda.is_available()),
    collate_fn=lambda batch: collate_character_windows(batch, getattr(tokenizer, "pad_token_id", 0) or 0),
)
final_char_boundary_model = V1CharacterBoundaryNERModel(
    backbone=copy.deepcopy(model),
    char_vocab_size=CHAR_VOCAB_SIZE,
    num_labels=len(BIO_LABELS),
    freeze_backbone=False,
).to(DEVICE)

# Preserve the original two learning-rate groups (pretrained backbone vs task head).
_refit_optimizer = torch.optim.AdamW([
    {"params": final_char_boundary_model.backbone.parameters(), "lr": CHAR_BOUNDARY_BACKBONE_LR, "weight_decay": 0.01},
    {"params": final_char_boundary_model.char_head.parameters(), "lr": CHAR_BOUNDARY_HEAD_LR, "weight_decay": 0.01},
])
_refit_class_weights = _make_char_class_weights(_all_labeled_dataset, DEVICE)
_refit_criterion = WeightedFocalLoss(
    _refit_class_weights, gamma=1.5, ignore_index=IGNORE_INDEX
).to(DEVICE)
_refit_updates_per_epoch = max(1, (len(_refit_loader) + CHAR_BOUNDARY_GRAD_ACCUM_STEPS - 1) // CHAR_BOUNDARY_GRAD_ACCUM_STEPS)
_refit_total_steps = max(1, _refit_updates_per_epoch * BEST_EPOCHS_FOR_REFIT)
_refit_warmup_steps = max(1, int(0.05 * _refit_total_steps))
def _refit_lr_multiplier(step):
    if step < _refit_warmup_steps:
        return max(0.10, float(step + 1) / float(_refit_warmup_steps))
    progress = min(1.0, (step - _refit_warmup_steps) / max(1, _refit_total_steps - _refit_warmup_steps))
    return 0.10 + 0.90 * 0.5 * (1.0 + math.cos(math.pi * progress))
_refit_scheduler = torch.optim.lr_scheduler.LambdaLR(_refit_optimizer, lr_lambda=_refit_lr_multiplier)
_refit_history = []

print(f"Final refit: {len(_all_labeled_ids):,} labelled docs / {len(_all_labeled_dataset):,} windows")
print(f"Training for {BEST_EPOCHS_FOR_REFIT} epochs, selected from validation best (F1={BEST_VALIDATION_F1_FOR_REFIT:.4f}).")
_refit_amp = bool(CHAR_BOUNDARY_USE_AMP and torch.cuda.is_available() and DEVICE.type == "cuda")
try:
    _refit_scaler = torch.amp.GradScaler("cuda", enabled=_refit_amp)
except (AttributeError, TypeError):
    _refit_scaler = torch.cuda.amp.GradScaler(enabled=_refit_amp)
for _epoch in range(1, BEST_EPOCHS_FOR_REFIT + 1):
    final_char_boundary_model.train()
    _loss_sum, _batch_count = 0.0, 0
    _t0 = time.time()
    _refit_optimizer.zero_grad(set_to_none=True)
    for _batch_idx, _batch in enumerate(_refit_loader):
        _inputs = _character_model_inputs(_batch, DEVICE)
        _targets = _batch["char_labels"].to(DEVICE, non_blocking=_refit_amp)
        _group_start = (_batch_idx // CHAR_BOUNDARY_GRAD_ACCUM_STEPS) * CHAR_BOUNDARY_GRAD_ACCUM_STEPS
        _group_size = min(CHAR_BOUNDARY_GRAD_ACCUM_STEPS, len(_refit_loader) - _group_start)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=_refit_amp):
            _logits = final_char_boundary_model(**_inputs)
        _loss = _refit_criterion(_logits.float().reshape(-1, _logits.shape[-1]), _targets.reshape(-1))
        if not torch.isfinite(_loss):
            raise FloatingPointError(f"Non-finite refit loss at epoch {_epoch}, batch {_batch_idx}: {_loss.item()}")
        _refit_scaler.scale(_loss / _group_size).backward()
        _loss_sum += float(_loss.detach().item())
        _batch_count += 1
        _is_update = ((_batch_idx + 1) % CHAR_BOUNDARY_GRAD_ACCUM_STEPS == 0) or (_batch_idx + 1 == len(_refit_loader))
        if _is_update:
            _refit_scaler.unscale_(_refit_optimizer)
            torch.nn.utils.clip_grad_norm_(final_char_boundary_model.parameters(), max_norm=1.0)
            _refit_scaler.step(_refit_optimizer)
            _refit_scaler.update()
            _refit_optimizer.zero_grad(set_to_none=True)
            _refit_scheduler.step()
    _row = {"epoch": _epoch, "train_loss": _loss_sum / max(_batch_count, 1), "seconds": time.time() - _t0}
    _refit_history.append(_row)
    print(f"All-data refit epoch {_epoch}/{BEST_EPOCHS_FOR_REFIT} | loss={_row['train_loss']:.4f} | AMP={_refit_amp} | {_row['seconds']:.1f}s")

FINAL_REFIT_CHECKPOINT_PATH = os.path.join(CHAR_BOUNDARY_OUTPUT_DIR, "v1_character_boundary_all_labelled_docs.pt")
os.makedirs(CHAR_BOUNDARY_OUTPUT_DIR, exist_ok=True)
_tmp_refit_path = FINAL_REFIT_CHECKPOINT_PATH + ".tmp"
torch.save({
    "epoch": BEST_EPOCHS_FOR_REFIT,
    "validation_selected_epoch": BEST_EPOCHS_FOR_REFIT,
    "validation_selected_f1": BEST_VALIDATION_F1_FOR_REFIT,
    "model_state_dict": final_char_boundary_model.state_dict(),
    "optimizer_state_dict": _refit_optimizer.state_dict(),
    "scheduler_state_dict": _refit_scheduler.state_dict(),
    "architecture": "V1CharacterBoundaryNERModel",
    "refit_docs": len(_all_labeled_ids),
    "char_to_id": CHAR_TO_ID,
    "bio_labels": BIO_LABELS,
    "thresholds_source": "validation-only thresholds from character_confidence_thresholds.json",
}, _tmp_refit_path)
os.replace(_tmp_refit_path, FINAL_REFIT_CHECKPOINT_PATH)
with open(os.path.join(CHAR_BOUNDARY_OUTPUT_DIR, "all_data_refit_history.json"), "w", encoding="utf-8") as _f:
    json.dump(_refit_history, _f, indent=2, ensure_ascii=False)
print("Saved all-labelled-data refit checkpoint:", FINAL_REFIT_CHECKPOINT_PATH)
print("Keep the validation-best checkpoint as a fallback; the refit has no held-out score because it trained on all labels.")

In [ ]:
# Generate and validate separate character-boundary submissions (all-data refit + raw fallback).
# Run after the character-boundary training and calibration cells above.

_test_texts_char = {str(k): v for k, v in TEST_TEXTS.items()}
char_test_dataset = CharacterBoundaryWindowDataset(
    document_ids=list(_test_texts_char.keys()),
    text_by_id=_test_texts_char,
    entities_by_id={},
    tokenizer_obj=tokenizer,
    max_length=MODEL_CONTEXT_LENGTH,
    window_overlap=CHAR_BOUNDARY_WINDOW_OVERLAP,
)

print(f"Running inference across {len(_test_texts_char)} test documents ({len(char_test_dataset)} windows)...")
_test_logits_by_doc, _ = collect_document_character_logits(
    final_char_boundary_model if "final_char_boundary_model" in globals() else char_boundary_model,
    char_test_dataset,
    _test_texts_char,
    device=DEVICE,
    batch_size=CHAR_BOUNDARY_BATCH_SIZE,
    pad_token_id=getattr(tokenizer, "pad_token_id", 0) or 0,
)

# 1. Primary Submission: Validation-calibrated thresholds
char_boundary_test_predictions = decode_document_logits_with_thresholds(
    _test_logits_by_doc, CHAR_CONFIDENCE_THRESHOLDS
)
for doc_id in _test_texts_char:
    char_boundary_test_predictions.setdefault(doc_id, [])

# Pre-validation of spans
validate_phase2_prediction_mapping(
    predictions=char_boundary_test_predictions,
    test_texts_by_id=_test_texts_char,
    entity_labels=LABELS,
)

# Package into submission ZIP
char_prediction_path, char_submission_zip_path = package_codabench_submission(
    char_boundary_test_predictions,
    output_dir=CHAR_BOUNDARY_OUTPUT_DIR,
    zip_filename="submission_v1_rescue_all_data_refit_calibrated.zip",
    expected_doc_ids=_test_texts_char.keys(),
)

validate_submission_zip(
    char_submission_zip_path,
    expected_doc_ids=_test_texts_char.keys(),
)

# 2. Raw Fallback Submission (zero thresholds)
_raw_thresholds = {label: 0.0 for label in LABELS}
raw_char_predictions = decode_document_logits_with_thresholds(
    _test_logits_by_doc, _raw_thresholds
)
for doc_id in _test_texts_char:
    raw_char_predictions.setdefault(doc_id, [])

raw_prediction_path, raw_submission_zip_path = package_codabench_submission(
    raw_char_predictions,
    output_dir=CHAR_BOUNDARY_OUTPUT_DIR,
    zip_filename="submission_v1_rescue_all_data_refit_raw.zip",
    expected_doc_ids=_test_texts_char.keys(),
)

validate_submission_zip(
    raw_submission_zip_path,
    expected_doc_ids=_test_texts_char.keys(),
)

print("\n" + "=" * 65)
print("CHARACTER-BOUNDARY SUBMISSIONS READY")
print("1. Primary (Calibrated):", char_submission_zip_path)
print("   JSON:", char_prediction_path)
print("   Validation Strict Macro-F1:", f"{CHAR_CALIBRATED_VAL_METRICS['strict_macro_f1']:.4f}")
print("2. Raw Fallback (No Bias):", raw_submission_zip_path)
print("   JSON:", raw_prediction_path)
print("   Validation Strict Macro-F1:", f"{_raw_calibration_metrics['strict_macro_f1']:.4f}")
print("=" * 65)